# PH-CVT U-Net · BraTS 2021 — Final Study Notebook (T4-ready, resumable)

**One notebook, three uses.** You only change `SESSION` in the first code cell, then choose **Runtime ▸ Run all**.

| Runtime | `SESSION` | What it does | Approx. T4 time |
|---|---|---|---|
| Runtime 1 | `"A"` | Streams the 188 raw test volumes to Drive (once), then trains **UNet s31415, UNet s27182, AttentionUNet s31415** | extraction 20–40 min, then training (measure `epoch_seconds`) |
| Runtime 2 | `"B"` | Trains **ResUNet s31415, ResUNet s27182, AttentionUNet s27182** | training only |
| Afterwards | `"FINAL"` | **No training.** Selected-slice aggregation, full-volume 3D Dice + HD95, deep-ensemble and MC-dropout uncertainty, calibration, statistics, modality-contribution analysis, all paper figures and tables | ~2.5–4 h |
| After FINAL | `"EXTERNAL"` | **No training.** Frozen external validation on OpenNeuro ds007045 (337 patients, 8 hospitals, via Kaggle): download, harmonisation, full-volume Dice/HD95, ensembles, MC dropout, per-hospital results, generalisation gap, uncertainty shift | download 20–40 min + ~2–3 h |

**Minimal plan (default settings).** `SESSION = "FINAL"` and `ANALYSIS_SEEDS = [42, 123, 2026]`: no training at all, every model compared on the same three existing seeds, one unattended run of ~3–4 h on a T4. External validation (`SESSION = "EXTERNAL"`) is optional and can be added later, e.g. during revision.

**Protocol guarantee.** Every data, model, loss, optimiser, scheduler, threshold-calibration and evaluation function below is copied *verbatim* from the notebook that produced seeds 42/123/2026. Only clearly marked session-control, extraction, full-volume, uncertainty and statistics cells are new. New seeds are therefore trained under exactly the same protocol (same split, preprocessing, batch 8 × accumulation 2, AdamW 3e-4, ReduceLROnPlateau, 100-epoch ceiling, patience 15).

**Checkpoints and resume.**
* `last_checkpoint.pt` after every epoch (model, optimiser, scheduler, AMP scaler, RNG, sampler state, history, patience).
* `best_model.pt` whenever validation Dice improves.
* `partial_checkpoint.pt` every 5 min inside an epoch and `interrupted_checkpoint.pt` on interruption.
* Completed runs are detected and skipped in seconds; full-volume evaluation is saved after every patient.

If Colab disconnects: reconnect, keep the same `SESSION`, **Run all** again.

In [ ]:
# =============================================================================
# CELL 0 — SESSION CONTROL  (the ONLY cell you normally edit)
# =============================================================================
# Runtime 1 -> SESSION = "A"      Runtime 2 -> SESSION = "B"      (run both at the same time)
# When both have finished      -> SESSION = "FINAL"               (one runtime, no training)
# External validation         -> SESSION = "EXTERNAL"            (one runtime, no training;
#                                                                  after FINAL for the internal-vs-external comparison)
# Then: Runtime > Run all.  Re-running after a disconnect resumes where it stopped.
# BraTS-Africa five-seed re-run (Part M) -> SESSION = "AFRICA5", then Runtime > Run all
SESSION = "FINAL"      # minimal plan: run FINAL only (no training). Full plan: "A" + "B", then "FINAL".

# ---- Training plan: which NEW model/seed runs each training session performs -------------
# Seeds 42, 123 and 2026 already exist for every baseline; these add seeds 4 and 5.
TRAIN_PLANS = {
    "A": {"UNet": [31415, 27182], "AttentionUNet": [31415]},
    "B": {"ResUNet": [31415, 27182], "AttentionUNet": [27182]},
    "FINAL": {},                       # FINAL never trains
    "EXTERNAL": {},                    # EXTERNAL never trains
    "AFRICA5": {},                     # Part M: BraTS-Africa, all five seeds, inference only
}

# ---- Seeds each model should have when everything is complete (used by FINAL) ------------
FIVE_SEEDS = [42, 123, 2026, 31415, 27182]
DEFAULT_SEEDS = [42, 123, 2026]
TARGET_SEEDS = {
    "PHCVT_proposed": FIVE_SEEDS, "PHCVT_lite": FIVE_SEEDS,
    "UNet": FIVE_SEEDS, "ResUNet": FIVE_SEEDS, "AttentionUNet": FIVE_SEEDS,
}
# ---- Seeds used in ALL analyses ------------------------------------------------------------
# Minimal plan (default): every model is compared on the SAME three seeds that already exist,
# so ensembles, statistics, modality ablation and external results are all like-for-like.
# Full plan: set ANALYSIS_SEEDS = None after sessions A and B have trained seeds 31415/27182.
ANALYSIS_SEEDS = None
if ANALYSIS_SEEDS:
    TARGET_SEEDS = {m: [s for s in seeds if s in ANALYSIS_SEEDS] for m, seeds in TARGET_SEEDS.items()}

ALL_MODELS = ["PHCVT_proposed", "PHCVT_lite", "UNet", "ResUNet", "AttentionUNet",
              "SwinUnet", "TransUNet", "PHCVT_segmentation_only", "PHCVT_no_transformer",
              "PHCVT_no_aspp", "PHCVT_no_attention_fusion", "PHCVT_no_auxiliary"]

# ---- Stage switches (sensible defaults per session; override only if you need to) -------
EXTRACT_RAW_TEST = SESSION in ("A", "FINAL", "EXTERNAL")   # idempotent; EXTERNAL needs one BraTS file for orientation
RUN_FULL_VOLUME  = SESSION == "FINAL"
RUN_EFFICIENCY   = False          # re-profile every model on ONE GPU type
RUN_STATISTICS   = False

# Model families evaluated on full 3D volumes (every completed seed of each is used).
FULLVOLUME_FAMILIES = ["PHCVT_proposed", "UNet", "ResUNet", "AttentionUNet"]
# Optional extras (each adds roughly 10-20 min on a T4). TransUNet/SwinUnet install timm.
# e.g. ["PHCVT_no_transformer", "PHCVT_no_attention_fusion", "TransUNet", "SwinUnet"]
FULLVOLUME_EXTRA_FAMILIES = []

# Modality-contribution analysis (inference only): each MRI sequence is removed in turn at
# test time and the Dice drop per region is measured on full volumes.
RUN_MODALITY_ABLATION = False
MODALITY_ABLATION_FAMILIES = ["PHCVT_proposed", "UNet", "ResUNet", "AttentionUNet"]
MODALITY_ABLATION_MAX_SEEDS = 3    # first 3 seeds per model (~45-70 min on a T4); None = all seeds

RUN_MC_DROPOUT_FULLVOLUME = True   # 12-pass MC dropout for the frozen PH-CVT seed-42 model
MC_PASSES_FULLVOLUME = 12          # identical to CONFIG["mc_dropout_passes"]
FAILURE_DICE = 0.75                # prespecified patient failure definition (mean WT/TC/ET Dice)
EQUIVALENCE_MARGIN = 0.01          # +/- 1 Dice point for the TOST-style equivalence statement

# Optional compute-budget guard for training sessions: after this many hours the current
# run stops cleanly at the next optimiser step (last completed epoch is kept) and the
# runtime is released. None = no limit.
MAX_TRAINING_HOURS = None

# ---- External validation cohort (SESSION = "EXTERNAL") ---------------------------------
# OpenNeuro ds007045 (Filimonova et al., Sci Data 2026): 337 IDH-wildtype glioblastomas,
# 8 hospitals, accessed through its Kaggle mirror. Evaluated FROZEN: same checkpoints, same
# BraTS-2021 validation thresholds, no fine-tuning, same full-volume protocol.
RUN_EXTERNAL = SESSION == "EXTERNAL"
EXTERNAL_NAME = "ds007045"
EXTERNAL_KAGGLE_HANDLE = "keeplershaurya/mri-glioblastoma"
EXTERNAL_IMAGE_FOLDER = "processing_MNI152_skull-stripped_N4"   # skull-stripped, MNI152, N4 (NOT z-scored)
EXTERNAL_MASK_FOLDER = "segmentation"
EXTERNAL_FAMILIES = ["PHCVT_proposed", "UNet", "ResUNet", "AttentionUNet"]
EXTERNAL_MAX_SEEDS = None          # None = all completed seeds (~2-3 h on a T4); 3 = faster
# Only if automatic file matching reports ambiguities: force a file-name substring per sequence,
# e.g. {"t1": "run-01_T1w", "t1ce": "run-02_T1w", "t2": "T2w", "flair": "FLAIR"}
EXTERNAL_FILE_OVERRIDES = {}

RELEASE_RUNTIME_WHEN_DONE = False   # release the GPU (stop spending units) after the last cell

assert SESSION in TRAIN_PLANS, f"SESSION must be one of {list(TRAIN_PLANS)}"
if SESSION in ("A", "B") and ANALYSIS_SEEDS:
    print("WARNING: ANALYSIS_SEEDS is set, so the seeds trained in this session will not be used in the "
          "analyses. For the full plan set ANALYSIS_SEEDS = None.")
print("Seeds used in analyses:", ANALYSIS_SEEDS or "all completed seeds")
print(f"SESSION = {SESSION}")
print("Training plan:", TRAIN_PLANS[SESSION] or "none (no training in this session)")
print(f"Extract raw test volumes: {EXTRACT_RAW_TEST} | Full-volume: {RUN_FULL_VOLUME} | "
      f"Efficiency: {RUN_EFFICIENCY} | Statistics: {RUN_STATISTICS} | Modality ablation: {RUN_MODALITY_ABLATION} | "
      f"External: {RUN_EXTERNAL}")

## Part 1 — Environment, data and fixed patient split (original protocol)
These cells are copied verbatim from the notebook that produced seeds 42/123/2026. They mount Drive, restore the existing HDF5 slice cache and the locked 875/188/188 patient split. Nothing is downloaded or rebuilt when the cache already exists.

In [ ]:
# CELL 1 — Safe dependencies, imports and GPU verification
# Do not install/upgrade torch, torchvision or torchaudio in Colab.
# Pinning pandas==2.2.2 fails to build on Colab's Python 3.13; the preinstalled pandas is used.
%pip install -q kagglehub nibabel scikit-learn tqdm matplotlib h5py scipy

import os, sys, gc, json, math, random, shutil, tarfile, time, warnings
from pathlib import Path
from contextlib import nullcontext
from collections import defaultdict

import numpy as np
import pandas as pd
import nibabel as nib
import h5py
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

seed_everything()

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Environment:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: Enable GPU in Colab: Runtime > Change runtime type > T4 GPU")

In [ ]:
# CELL 2 — Full-cohort configuration and persistent paths
CONFIG = {
    "experiment_name": "PH_CVT_UNet_BraTS2021_Full1251",
    "dataset_handle": "dschettler8845/brats-2021-task1",
    "expected_patient_records": 1251,
    "image_size": 160,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "test_fraction": 0.15,
    "max_positive_slices_per_patient": 16,
    "max_negative_slices_per_patient": 4,
    "min_original_tumor_pixels": 20,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "epochs": 100,
    "early_stopping_patience": 15,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "mc_dropout_passes": 12,
    "resume_training": True,
    "persist_h5_cache_to_drive": True,
    "h5_compression": "lzf",
    "use_channels_last": True,
    "auto_disconnect_after_success": False,
}

LOCAL_ROOT = Path("/content/brats_phcvt_full1251") if IN_COLAB else Path("./brats_phcvt_full1251")
RAW_DIR = LOCAL_ROOT / "raw"
TEMP_EXTRACT_DIR = LOCAL_ROOT / "patient_temp"
LOCAL_CACHE_DIR = LOCAL_ROOT / "h5_cache"
LOCAL_OUTPUT_DIR = LOCAL_ROOT / "outputs"

for p in [RAW_DIR, TEMP_EXTRACT_DIR, LOCAL_CACHE_DIR, LOCAL_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = LOCAL_OUTPUT_DIR
DRIVE_AVAILABLE = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BASE = Path("/content/drive/MyDrive/AI_Brain_Tumor")
        DRIVE_ROOT = DRIVE_BASE / "PH_CVT_BraTS1251"
        SHARED_DATA_ROOT = DRIVE_BASE / "shared_brats1251"
        DRIVE_CACHE_DIR = SHARED_DATA_ROOT / "h5_cache"
        OUTPUT_DIR = DRIVE_ROOT / "outputs"
        for p in [DRIVE_ROOT, SHARED_DATA_ROOT, DRIVE_CACHE_DIR, OUTPUT_DIR]:
            p.mkdir(parents=True, exist_ok=True)
        DRIVE_AVAILABLE = True
        print("Persistent outputs:", OUTPUT_DIR)
        print("Shared full-cohort cache:", DRIVE_CACHE_DIR)
    except Exception as e:
        raise RuntimeError("Drive is required to preserve the existing experiment; reconnect Drive before running.") from e
else:
    DRIVE_ROOT = LOCAL_ROOT
    SHARED_DATA_ROOT = LOCAL_ROOT / "shared"
    DRIVE_CACHE_DIR = SHARED_DATA_ROOT / "h5_cache"
    SHARED_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
SPLIT_PATH = SHARED_DATA_ROOT / "patient_level_split_1251.csv"
MANIFEST_PATH = SHARED_DATA_ROOT / "slice_manifest_1251.csv"
THRESHOLD_PATH = OUTPUT_DIR / "calibrated_thresholds.json"

LOCAL_H5 = {split: LOCAL_CACHE_DIR / f"brats1251_{split}.h5" for split in ["train", "val", "test"]}
DRIVE_H5 = {split: DRIVE_CACHE_DIR / f"brats1251_{split}.h5" for split in ["train", "val", "test"]}

# Preserve old checkpoints/results once before continuing the existing experiment.
if not BEST_MODEL_PATH.exists() or not CHECKPOINT_PATH.exists() or not SPLIT_PATH.exists():
    raise FileNotFoundError('Original best_model.pt, last_checkpoint.pt and patient_level_split_1251.csv are required. Restore the original Drive folder; this notebook never substitutes a fresh proposed model/split.')
PRE_EXTENSION_DIR = OUTPUT_DIR / 'pre_extension_archive'
PRE_EXTENSION_DIR.mkdir(exist_ok=True)
if not (PRE_EXTENSION_DIR/'archive_complete.txt').exists():
    for old in list(OUTPUT_DIR.iterdir()):
        if old.is_file() and old.suffix.lower() in ['.pt','.csv','.json','.png']:
            target=PRE_EXTENSION_DIR/old.name
            if not target.exists():
                tmp=target.with_suffix(target.suffix+'.tmp'); shutil.copy2(old,tmp); os.replace(tmp,target)
    (PRE_EXTENSION_DIR/'archive_complete.txt').write_text('Original outputs preserved before first extension run.')

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

In [ ]:
# Recovery and run policy. No changes to the proposed network or original data paths.
import hashlib, threading, traceback, platform
from contextlib import contextmanager

STUDY_DIR = OUTPUT_DIR / 'extended_studies'
STUDY_DIR.mkdir(parents=True, exist_ok=True)
STUDY = {
    # Per-model seed plan: the two headline models (proposed, lite) get 5 seeds
    # each for a real variance estimate; every other model gets 3 seeds, the
    # accepted floor for reporting mean+/-std in the ML reproducibility literature.
    # PHCVT_proposed seed 42 was already trained separately above (OUTPUT_DIR
    # checkpoint, evaluated into proposed_study_dir); only its 4 NEW seeds are here.
    'default_seeds': [42, 123, 2026],
    'seed_plan': {
        'PHCVT_proposed': [123, 2026, 31415, 27182],
        'PHCVT_lite': [42, 123, 2026, 31415, 27182],
    },
    # Priority models (proposed's remaining seeds, then lite) run first so a
    # Colab disconnect leaves the headline comparison as complete as possible.
    # UNet/ResUNet/AttentionUNet are already fully trained at seed 42 and will be
    # skipped near-instantly by the fast-resume check below.
    'models': ['PHCVT_proposed', 'PHCVT_lite',
               'UNet', 'ResUNet', 'AttentionUNet', 'SwinUnet', 'TransUNet',
               'PHCVT_segmentation_only', 'PHCVT_no_transformer', 'PHCVT_no_aspp',
               'PHCVT_no_attention_fusion', 'PHCVT_no_auxiliary'],
    'run_comparisons': True,
    'run_external': True,
    'partial_snapshot_seconds': 300,
    'idle_minutes': 15,
    'disconnect_on_error': True,
    'external_root': '',  # Optional explicit extracted NIfTI root in mounted Drive.
    'external_manifest': str(STUDY_DIR / 'external_manifest.csv'),
    'external_overlap_review': str(STUDY_DIR / 'external_overlap_review.csv'),
    'bootstrap_samples': 2000,
}
# Canonical baseline source is downloaded once, version-locked and saved in Drive.
# The baseline source/configuration and every model's parameter count are exported.
# ---- SESSION OVERRIDES (new) -------------------------------------------------------------
# The STUDY dictionary above is the original one; only the model/seed lists and the
# unattended-runtime behaviour are adapted to the current SESSION. Nothing that affects
# training numerics is changed.
ALLOW_TRAINING = SESSION in ("A", "B") and bool(TRAIN_PLANS[SESSION])
MISSING_RUNS = []                                   # filled in FINAL if a planned run is absent
if SESSION in ("A", "B"):
    STUDY['models'] = list(TRAIN_PLANS[SESSION])
    STUDY['seed_plan'] = {m: list(s) for m, s in TRAIN_PLANS[SESSION].items()}
else:
    STUDY['models'] = list(ALL_MODELS)
    # PH-CVT seed 42 lives in OUTPUT_DIR and is always added separately, so it is excluded here.
    STUDY['seed_plan'] = {m: [s for s in seeds if not (m == 'PHCVT_proposed' and s == 42)]
                          for m, seeds in TARGET_SEEDS.items()}
STUDY['run_external'] = False
STUDY['disconnect_on_error'] = SESSION in ("A", "B")   # unattended training: stop spending units on error
STUDY['idle_minutes'] = 15 if SESSION in ("A", "B") else 45
# One config file per session, so two parallel runtimes never overwrite each other's record.
(STUDY_DIR / f'study_config_session{SESSION}.json').write_text(json.dumps(STUDY, indent=2))

RUN_STATUS = {}
_WRITE_LOCK = threading.RLock()
IDLE = {'busy': True, 'since': time.monotonic(), 'closed': False}

def atomic_json(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with _WRITE_LOCK:
        tmp.write_text(json.dumps(value, indent=2, default=str)); os.replace(tmp, path)

def release_runtime(reason):
    if IDLE['closed'] or not IN_COLAB or not DRIVE_AVAILABLE:
        return
    with _WRITE_LOCK:
        atomic_json({'reason': reason, 'time': time.time(), 'status': RUN_STATUS}, STUDY_DIR / f'runtime_exit_session{SESSION}.json')
        IDLE['closed'] = True
    print('Saved status to Drive. Releasing Colab runtime:', reason, flush=True)
    from google.colab import runtime
    runtime.unassign()

def idle_watchdog():
    while not IDLE['closed']:
        if not IDLE['busy'] and time.monotonic() - IDLE['since'] > 60 * STUDY['idle_minutes']:
            release_runtime('No cell executing for configured idle period'); return
        time.sleep(10)

def before_cell(*args, **kwargs):
    IDLE.update(busy=True, since=time.monotonic())

def after_cell(result=None):
    IDLE.update(busy=False, since=time.monotonic())
    err = getattr(result, 'error_in_exec', None) or getattr(result, 'error_before_exec', None)
    if err is not None and STUDY['disconnect_on_error']:
        release_runtime('Cell interrupted or failed; resume from last completed epoch')

# Re-running this cell does not accumulate event handlers or watchdogs.
if '_old_runtime_handlers' in globals():
    for event, handler in _old_runtime_handlers:
        try: get_ipython().events.unregister(event, handler)
        except ValueError: pass
try:
    shell = get_ipython()
    shell.events.register('pre_run_cell', before_cell)
    shell.events.register('post_run_cell', after_cell)
    _old_runtime_handlers = [('pre_run_cell', before_cell), ('post_run_cell', after_cell)]
except NameError:
    pass
if '_watchdog_thread' not in globals() or not _watchdog_thread.is_alive():
    _watchdog_thread = threading.Thread(target=idle_watchdog, daemon=True)
    _watchdog_thread.start()

print('Resume enabled; max epochs=100, patience=15. Idle means no executing cell, not low GPU utilization.')
print('STUDY models:', STUDY['models'])

### Kaggle authentication (only needed for the one-time raw test-volume extraction)
In Colab **Secrets**, keep the secret named **`kaggle_API`** with notebook access enabled (already used by your earlier notebooks).

In [ ]:
# CELL 3 — Authenticate and locate the BraTS 2021 dataset robustly
import re
import kagglehub

PATIENT_RE = re.compile(r"(BraTS2021_\d{5})")

def configure_kaggle_auth():
    token_found = False
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("kaggle_API")
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
                token_found = True
                print("Using kaggle_API from Colab Secrets.")
        except Exception as e:
            print("Could not read the Colab secret:", repr(e))

    legacy_file = Path.home() / ".kaggle" / "kaggle.json"
    access_file = Path.home() / ".kaggle" / "access_token"
    if legacy_file.exists() or access_file.exists():
        token_found = True
        print("Using an existing Kaggle credential file.")

    if not token_found:
        raise RuntimeError(
            "Kaggle authentication was not found. Add a Colab secret named "
            "kaggle_API and enable notebook access."
        )

def find_training_archive(root: Path):
    candidates = []
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file():
                n = p.name.lower()
                if (n.endswith(".tar") or n.endswith(".tar.gz") or n.endswith(".tgz")) and "brats" in n and "training" in n:
                    candidates.append(p)
    return max(candidates, key=lambda p: p.stat().st_size) if candidates else None

def find_preextracted_root(root: Path):
    if not root.exists():
        return None
    matches = list(root.rglob("BraTS2021_*_flair.nii.gz"))
    return matches[0].parent.parent if matches else None

# When a persistent HDF5 cache already exists, the MRI download is not required.
cache_ready = all(p.exists() and p.stat().st_size > 0 for p in DRIVE_H5.values()) if DRIVE_AVAILABLE else False
training_tar = None
preextracted_root = None

if cache_ready:
    print("Persistent full-cohort HDF5 cache already exists. Dataset download can be skipped.")
else:
    configure_kaggle_auth()
    dataset_path = Path(kagglehub.dataset_download(CONFIG["dataset_handle"]))
    print("KaggleHub dataset path:", dataset_path)
    print("Path exists:", dataset_path.exists())

    search_roots = [dataset_path, dataset_path.parent, RAW_DIR, Path.home() / ".cache" / "kagglehub"]
    for root in search_roots:
        training_tar = training_tar or find_training_archive(root)
        preextracted_root = preextracted_root or find_preextracted_root(root)

    if training_tar:
        print("Training archive:", training_tar)
        print("Archive size (GB):", round(training_tar.stat().st_size / 1e9, 2))
    elif preextracted_root:
        print("Pre-extracted BraTS root:", preextracted_root)
    else:
        print("First files under returned dataset path:")
        for p in list(dataset_path.rglob("*"))[:100]:
            print(" ", p)
        raise FileNotFoundError("No BraTS training archive or extracted NIfTI patient folders were found.")

In [ ]:
# CELL 4 — Discover all patient records and create one fixed patient-level split

def patient_ids_in_tar(tar_path):
    ids = set()
    with tarfile.open(tar_path, "r:*") as tf:
        for member in tf:
            m = PATIENT_RE.search(member.name)
            if m:
                ids.add(m.group(1))
    return sorted(ids)

def patient_dirs_from_root(root):
    result = {}
    for flair_path in Path(root).rglob("BraTS2021_*_flair.nii.gz"):
        m = PATIENT_RE.search(flair_path.name)
        if m:
            result[m.group(1)] = flair_path.parent
    return result

if SPLIT_PATH.exists():
    split_table = pd.read_csv(SPLIT_PATH)
    all_patient_ids = sorted(split_table["patient_id"].astype(str).tolist())
    print("Using saved full-cohort patient split:", SPLIT_PATH)
else:
    if cache_ready and MANIFEST_PATH.exists():
        manifest_for_ids = pd.read_csv(MANIFEST_PATH, usecols=["patient_id"])
        all_patient_ids = sorted(manifest_for_ids["patient_id"].astype(str).unique().tolist())
    elif training_tar is not None:
        all_patient_ids = patient_ids_in_tar(training_tar)
    else:
        preextracted_patient_dirs = patient_dirs_from_root(preextracted_root)
        all_patient_ids = sorted(preextracted_patient_dirs)

    print("Total patient records discovered:", len(all_patient_ids))
    if len(all_patient_ids) != CONFIG["expected_patient_records"]:
        raise ValueError(
            f"Expected {CONFIG['expected_patient_records']} records but found {len(all_patient_ids)}. "
            "Do not continue until the dataset version is confirmed."
        )

    train_ids, temp_ids = train_test_split(
        all_patient_ids,
        test_size=1.0 - CONFIG["train_fraction"],
        random_state=SEED,
        shuffle=True,
    )
    relative_test_fraction = CONFIG["test_fraction"] / (CONFIG["val_fraction"] + CONFIG["test_fraction"])
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=relative_test_fraction,
        random_state=SEED,
        shuffle=True,
    )

    lookup = {pid: "train" for pid in train_ids}
    lookup.update({pid: "val" for pid in val_ids})
    lookup.update({pid: "test" for pid in test_ids})
    split_table = pd.DataFrame({
        "patient_id": all_patient_ids,
        "split": [lookup[pid] for pid in all_patient_ids],
    })
    split_table.to_csv(SPLIT_PATH, index=False)

train_ids = split_table.loc[split_table["split"] == "train", "patient_id"].tolist()
val_ids = split_table.loc[split_table["split"] == "val", "patient_id"].tolist()
test_ids = split_table.loc[split_table["split"] == "test", "patient_id"].tolist()

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)
assert len(train_ids) + len(val_ids) + len(test_ids) == CONFIG["expected_patient_records"]

print(split_table["split"].value_counts())
print("Patient leakage check: PASSED")

### NEW — Raw NIfTI volumes of the 188 test patients
Required once for the full-volume 3D evaluation. Runs in session **A** (and is retried in **FINAL** if incomplete). Already-complete patients are skipped.

In [ ]:
# =============================================================================
# NEW — Raw NIfTI for the 188 held-out TEST patients (needed for full-volume evaluation)
# =============================================================================
# * Streams the BraTS 2021 archive ONCE and keeps only test patients (single sequential pass).
# * Writes to Drive:  PH_CVT_BraTS1251/raw/<patient_id>/<patient_id>_{t1,t1ce,t2,flair,seg}.nii.gz
# * Idempotent and resumable: patients already complete on Drive are never re-copied.
# * Never raises: if anything fails it prints the reason and the rest of the notebook continues
#   (FINAL retries automatically). Training data, cache and checkpoints are not touched.
RAW_TEST_DRIVE = DRIVE_ROOT / 'raw'
RAW_TEST_DRIVE.mkdir(parents=True, exist_ok=True)
MODALITIES = ['t1', 't1ce', 't2', 'flair']
RAW_FILES = MODALITIES + ['seg']

def raw_patient_complete(pid, root=RAW_TEST_DRIVE):
    return all((root / pid / f'{pid}_{s}.nii.gz').exists() and
               (root / pid / f'{pid}_{s}.nii.gz').stat().st_size > 0 for s in RAW_FILES)

def extract_raw_test_volumes():
    need = sorted(pid for pid in test_ids if not raw_patient_complete(pid))
    if not need:
        print(f'Raw test volumes already complete on Drive: {len(test_ids)}/{len(test_ids)}')
        return
    print(f'{len(need)} test patients still need raw volumes; downloading the archive...')
    configure_kaggle_auth()
    ds_path = Path(kagglehub.dataset_download(CONFIG['dataset_handle']))
    staging = LOCAL_ROOT / 'raw_test_staging'
    staging.mkdir(parents=True, exist_ok=True)
    need_set = set(need)
    tar_path = find_training_archive(ds_path) or find_training_archive(ds_path.parent)
    if tar_path is not None:
        print('Streaming archive:', tar_path)
        with tarfile.open(tar_path, 'r:*') as tf:
            for member in tf:                                   # one sequential pass, no index build
                m = PATIENT_RE.search(member.name)
                if not member.isfile() or m is None or m.group(1) not in need_set:
                    continue
                fname = Path(member.name).name
                if not (fname.startswith(m.group(1)) and fname.endswith('.nii.gz')):
                    continue                                    # skip hidden/metadata files
                target = staging / m.group(1) / fname
                target.parent.mkdir(parents=True, exist_ok=True)
                tmp = target.with_suffix('.part')
                with tf.extractfile(member) as src, open(tmp, 'wb') as dst:
                    shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
                os.replace(tmp, target)
    else:
        root = find_preextracted_root(ds_path) or find_preextracted_root(ds_path.parent)
        if root is None:
            raise FileNotFoundError('Neither a BraTS training archive nor extracted NIfTI folders were found.')
        dirs = patient_dirs_from_root(root)
        for pid in need:
            if pid in dirs:
                (staging / pid).mkdir(parents=True, exist_ok=True)
                for s in RAW_FILES:
                    src = dirs[pid] / f'{pid}_{s}.nii.gz'
                    if src.exists():
                        shutil.copy2(src, staging / pid / src.name)
    copied = 0
    for pid in tqdm(need, desc='Copying test volumes to Drive'):
        if not raw_patient_complete(pid, staging):
            print('  incomplete in archive, skipped:', pid); continue
        dst = RAW_TEST_DRIVE / pid
        dst.mkdir(parents=True, exist_ok=True)
        for s in RAW_FILES:                                     # file-level atomic copy
            src = staging / pid / f'{pid}_{s}.nii.gz'
            tmp = dst / (src.name + '.part')
            shutil.copy2(src, tmp); os.replace(tmp, dst / src.name)
        copied += 1
    shutil.rmtree(staging, ignore_errors=True)
    done = sum(raw_patient_complete(pid) for pid in test_ids)
    print(f'Copied {copied} patients. Raw test volumes on Drive: {done}/{len(test_ids)}')

if EXTRACT_RAW_TEST:
    try:
        extract_raw_test_volumes()
    except Exception as exc:
        print('Raw test extraction did not complete (the notebook continues):', repr(exc))
else:
    print('Raw test extraction not requested in this session.')

In [ ]:
# CELL 5 — Restore or build a fast full-cohort HDF5 slice cache
# The first run preprocesses all 1,251 patients. Later runs copy the persistent cache from Drive to local Colab storage.

def modality_path(patient_dir: Path, patient_id: str, suffix: str):
    matches = list(patient_dir.rglob(f"{patient_id}_{suffix}.nii.gz"))
    if not matches:
        raise FileNotFoundError(f"Missing {suffix} for {patient_id} under {patient_dir}")
    return matches[0]

def robust_normalize(volume):
    volume = volume.astype(np.float32)
    brain = volume != 0
    if brain.sum() == 0:
        return np.zeros_like(volume, dtype=np.float32)
    values = volume[brain]
    lo, hi = np.percentile(values, [0.5, 99.5])
    volume = np.clip(volume, lo, hi)
    values = volume[brain]
    mean, std = float(values.mean()), float(values.std())
    if std < 1e-6:
        std = 1.0
    volume[brain] = (volume[brain] - mean) / std
    volume[~brain] = 0
    return volume

def evenly_select(indices, max_items):
    indices = np.asarray(sorted(set(indices)), dtype=int)
    if len(indices) <= max_items:
        return indices.tolist()
    positions = np.linspace(0, len(indices) - 1, max_items).round().astype(int)
    return indices[positions].tolist()

def choose_slices(seg, flair):
    tumor_counts = (seg > 0).sum(axis=(0, 1))
    positive = np.where(tumor_counts >= CONFIG["min_original_tumor_pixels"])[0]
    positive = evenly_select(positive, CONFIG["max_positive_slices_per_patient"])

    brain_counts = (flair != 0).sum(axis=(0, 1))
    valid_brain = np.where(brain_counts > 500)[0]
    if len(positive):
        low = max(int(min(positive)) - 12, int(valid_brain.min()) if len(valid_brain) else 0)
        high = min(int(max(positive)) + 12, int(valid_brain.max()) if len(valid_brain) else seg.shape[2] - 1)
        negative_candidates = [s for s in valid_brain if low <= s <= high and tumor_counts[s] == 0]
    else:
        negative_candidates = [s for s in valid_brain if tumor_counts[s] == 0]
    negative = evenly_select(negative_candidates, CONFIG["max_negative_slices_per_patient"])
    return [(int(s), 1) for s in positive] + [(int(s), 0) for s in negative]

def resize_batch(images, masks, image_size):
    x = torch.from_numpy(images).float()
    y = torch.from_numpy(masks[:, None]).float()
    x = F.interpolate(x, size=(image_size, image_size), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(image_size, image_size), mode="nearest")
    return x.numpy().astype(np.float16), y[:, 0].numpy().astype(np.uint8)

def create_h5(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    f = h5py.File(path, "w")
    size = CONFIG["image_size"]
    compression = CONFIG["h5_compression"]
    f.create_dataset("images", shape=(0, 4, size, size), maxshape=(None, 4, size, size), dtype="float16", chunks=(1, 4, size, size), compression=compression)
    f.create_dataset("masks", shape=(0, size, size), maxshape=(None, size, size), dtype="uint8", chunks=(1, size, size), compression=compression)
    f.create_dataset("has_tumor", shape=(0,), maxshape=(None,), dtype="uint8", chunks=True)
    f.create_dataset("slice_idx", shape=(0,), maxshape=(None,), dtype="int16", chunks=True)
    f.create_dataset("patient_id", shape=(0,), maxshape=(None,), dtype=h5py.string_dtype("utf-8"), chunks=True)
    return f

def append_h5(f, images, masks, labels, slice_indices, patient_ids):
    n0 = f["images"].shape[0]
    n1 = n0 + len(images)
    for key in ["images", "masks", "has_tumor", "slice_idx", "patient_id"]:
        ds = f[key]
        ds.resize((n1,) + ds.shape[1:])
    f["images"][n0:n1] = images
    f["masks"][n0:n1] = masks
    f["has_tumor"][n0:n1] = labels
    f["slice_idx"][n0:n1] = slice_indices
    f["patient_id"][n0:n1] = patient_ids

def process_patient(patient_id, patient_dir, split, h5_files, manifest_rows):
    t1 = robust_normalize(nib.load(modality_path(patient_dir, patient_id, "t1")).get_fdata())
    t1ce = robust_normalize(nib.load(modality_path(patient_dir, patient_id, "t1ce")).get_fdata())
    t2 = robust_normalize(nib.load(modality_path(patient_dir, patient_id, "t2")).get_fdata())
    flair = robust_normalize(nib.load(modality_path(patient_dir, patient_id, "flair")).get_fdata())
    seg = nib.load(modality_path(patient_dir, patient_id, "seg")).get_fdata().astype(np.uint8)
    seg[seg == 4] = 3

    chosen = choose_slices(seg, flair)
    if not chosen:
        return
    slice_indices = [s for s, _ in chosen]
    labels = np.asarray([lab for _, lab in chosen], dtype=np.uint8)

    images = np.stack([
        np.stack([t1[:, :, s], t1ce[:, :, s], t2[:, :, s], flair[:, :, s]], axis=0)
        for s in slice_indices
    ], axis=0).astype(np.float32)
    masks = np.stack([seg[:, :, s] for s in slice_indices], axis=0).astype(np.uint8)
    images, masks = resize_batch(images, masks, CONFIG["image_size"])

    append_h5(
        h5_files[split], images, masks, labels,
        np.asarray(slice_indices, dtype=np.int16),
        np.asarray([patient_id] * len(chosen), dtype=object),
    )
    for local_idx, (slice_idx, has_tumor) in enumerate(chosen):
        manifest_rows.append({
            "patient_id": patient_id,
            "split": split,
            "slice_idx": int(slice_idx),
            "has_tumor": int(has_tumor),
            "h5_index": int(h5_files[split]["images"].shape[0] - len(chosen) + local_idx),
        })
    del t1, t1ce, t2, flair, seg, images, masks
    gc.collect()


def verify_original_cache(paths):
    if not MANIFEST_PATH.exists(): raise FileNotFoundError('Original slice manifest is required for cache integrity checks.')
    manifest=pd.read_csv(MANIFEST_PATH)
    if manifest.duplicated(['patient_id','slice_idx']).any(): raise ValueError('Duplicate slices in original manifest')
    split_lookup=dict(zip(split_table.patient_id.astype(str),split_table.split))
    for split,path in paths.items():
        with h5py.File(path,'r') as f:
            required=['images','masks','patient_id','slice_idx','has_tumor']
            if any(k not in f for k in required): raise ValueError(f'Incomplete cache: {path}')
            count=len(f['images'])
            if not count or any(len(f[k])!=count for k in required): raise ValueError(f'Unequal/empty cache datasets: {path}')
            if f['images'].shape[1:]!=(4,160,160) or f['masks'].shape[1:]!=(160,160): raise ValueError('Cache shape differs from original protocol')
            ids=[v.decode() if isinstance(v,bytes) else str(v) for v in f['patient_id'][:]]
            if any(split_lookup.get(pid)!=split for pid in ids): raise ValueError('Cache/split patient mismatch')
            rows=manifest[manifest.split==split].sort_values('h5_index')
            if len(rows)!=count or ids!=rows.patient_id.astype(str).tolist() or not np.array_equal(f['slice_idx'][:],rows.slice_idx.to_numpy()):
                raise ValueError(f'Incomplete or reordered cache {path}; restore the complete original cache.')
            if set(ids)!=set(split_table.loc[split_table.split==split,'patient_id'].astype(str)):
                raise ValueError('Some split patients are absent from the cache')
    print('Original cache, manifest and split alignment: PASS')

# Restore persistent cache to fast local storage.
if all(p.exists() and p.stat().st_size > 0 for p in LOCAL_H5.values()):
    verify_original_cache(LOCAL_H5)
    print("Using existing local HDF5 cache.")
elif DRIVE_AVAILABLE and all(p.exists() and p.stat().st_size > 0 for p in DRIVE_H5.values()):
    verify_original_cache(DRIVE_H5)
    print("Restoring full-cohort HDF5 cache from Google Drive...")
    for split in ["train", "val", "test"]:
        shutil.copy2(DRIVE_H5[split], LOCAL_H5[split])
        print(split, "restored:", round(LOCAL_H5[split].stat().st_size / 1e9, 2), "GB")
else:
    print("Building HDF5 cache for all patient records. Do not interrupt this first preprocessing run.")
    for p in LOCAL_H5.values():
        if p.exists():
            p.unlink()
    h5_files = {split: create_h5(LOCAL_H5[split]) for split in ["train", "val", "test"]}
    manifest_rows = []
    split_lookup = dict(zip(split_table["patient_id"], split_table["split"]))

    try:
        if preextracted_root is not None:
            patient_dirs = patient_dirs_from_root(preextracted_root)
            for patient_id in tqdm(all_patient_ids, desc="Preprocessing 1,251 patients"):
                process_patient(patient_id, patient_dirs[patient_id], split_lookup[patient_id], h5_files, manifest_rows)
        else:
            # Build a member map once, extract one patient, preprocess it, then delete it.
            with tarfile.open(training_tar, "r:*") as tf:
                member_map = defaultdict(list)
                for member in tf.getmembers():
                    m = PATIENT_RE.search(member.name)
                    if m:
                        member_map[m.group(1)].append(member)

                for patient_id in tqdm(all_patient_ids, desc="Streaming extraction + preprocessing"):
                    patient_temp = TEMP_EXTRACT_DIR / patient_id
                    if patient_temp.exists():
                        shutil.rmtree(patient_temp)
                    patient_temp.mkdir(parents=True, exist_ok=True)
                    root_resolved = patient_temp.resolve()
                    for member in member_map[patient_id]:
                        destination = (patient_temp / member.name).resolve()
                        if not str(destination).startswith(str(root_resolved) + os.sep):
                            raise RuntimeError(f"Unsafe archive member: {member.name}")
                        tf.extract(member, path=patient_temp, filter="data")
                    flair_matches = list(patient_temp.rglob(f"{patient_id}_flair.nii.gz"))
                    if not flair_matches:
                        raise FileNotFoundError(f"No extracted FLAIR file for {patient_id}")
                    process_patient(patient_id, flair_matches[0].parent, split_lookup[patient_id], h5_files, manifest_rows)
                    shutil.rmtree(patient_temp, ignore_errors=True)
    finally:
        for f in h5_files.values():
            f.flush()
            f.close()

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(MANIFEST_PATH, index=False)

    if DRIVE_AVAILABLE and CONFIG["persist_h5_cache_to_drive"]:
        print("Copying completed HDF5 cache to Google Drive for future runs and comparative studies...")
        for split in ["train", "val", "test"]:
            shutil.copy2(LOCAL_H5[split], DRIVE_H5[split])
            print(split, "saved:", round(DRIVE_H5[split].stat().st_size / 1e9, 2), "GB")

for split in ["train", "val", "test"]:
    with h5py.File(LOCAL_H5[split], "r") as f:
        labels = f["has_tumor"][:]
        print(f"{split}: {len(labels):,} slices | positive={(labels==1).sum():,} | negative={(labels==0).sum():,}")
verify_original_cache(LOCAL_H5)

In [ ]:
# CELL 6 — Inspect cached multimodal images and BraTS targets

def label_to_regions(mask):
    wt = mask > 0
    tc = np.logical_or(mask == 1, mask == 3)
    et = mask == 3
    return np.stack([wt, tc, et], axis=0).astype(np.uint8)

with h5py.File(LOCAL_H5["train"], "r") as f:
    positive_indices = np.where(f["has_tumor"][:] == 1)[0]
    idx = int(positive_indices[len(positive_indices) // 2])
    sample_img = f["images"][idx].astype(np.float32)
    sample_mask = f["masks"][idx].astype(np.uint8)
    sample_patient = f["patient_id"][idx]
    if isinstance(sample_patient, bytes):
        sample_patient = sample_patient.decode()
    sample_slice = int(f["slice_idx"][idx])

sample_regions = label_to_regions(sample_mask)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, name in enumerate(["T1", "T1ce", "T2", "FLAIR"]):
    axes[0, i].imshow(sample_img[i], cmap="gray")
    axes[0, i].set_title(name)
    axes[0, i].axis("off")
axes[1, 0].imshow(sample_mask, cmap="nipy_spectral", vmin=0, vmax=3)
axes[1, 0].set_title("Original label mask")
for i, name in enumerate(["WT", "TC", "ET"], start=1):
    axes[1, i].imshow(sample_regions[i-1], cmap="gray")
    axes[1, i].set_title(name)
for ax in axes[1]: ax.axis("off")
plt.tight_layout(); plt.show()
print("Image shape:", sample_img.shape, "Patient:", sample_patient, "Slice:", sample_slice)

In [ ]:
# CELL 7 — HDF5 dataset, augmentation and data loaders
class H5BraTSSliceDataset(Dataset):
    def __init__(self, h5_path, augment=False):
        self.h5_path = str(h5_path)
        self.augment = augment
        self._h5 = None
        with h5py.File(self.h5_path, "r") as f:
            self.length = len(f["images"])

    def __len__(self):
        return self.length

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_h5"] = None
        return state

    def _file(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def _augment(self, image, regions):
        if random.random() < 0.5:
            image = np.flip(image, axis=2).copy(); regions = np.flip(regions, axis=2).copy()
        if random.random() < 0.5:
            image = np.flip(image, axis=1).copy(); regions = np.flip(regions, axis=1).copy()
        if random.random() < 0.5:
            k = random.choice([1, 2, 3])
            image = np.rot90(image, k=k, axes=(1, 2)).copy(); regions = np.rot90(regions, k=k, axes=(1, 2)).copy()
        if random.random() < 0.35:
            scale = np.random.uniform(0.90, 1.10, size=(4, 1, 1)).astype(np.float32)
            shift = np.random.uniform(-0.10, 0.10, size=(4, 1, 1)).astype(np.float32)
            image = image * scale + shift
        if random.random() < 0.25:
            image = image + np.random.normal(0, 0.03, image.shape).astype(np.float32)
        return image, regions

    def __getitem__(self, index):
        f = self._file()
        image = f["images"][index].astype(np.float32)
        mask = f["masks"][index].astype(np.uint8)
        regions = label_to_regions(mask).astype(np.float32)
        if self.augment:
            image, regions = self._augment(image, regions)
        patient_id = f["patient_id"][index]
        if isinstance(patient_id, bytes): patient_id = patient_id.decode()
        return {
            "image": torch.from_numpy(image),
            "regions": torch.from_numpy(regions),
            "has_tumor": torch.tensor(float(f["has_tumor"][index]), dtype=torch.float32),
            "patient_id": patient_id,
            "slice_idx": int(f["slice_idx"][index]),
        }

train_dataset = H5BraTSSliceDataset(LOCAL_H5["train"], augment=True)
val_dataset = H5BraTSSliceDataset(LOCAL_H5["val"], augment=False)
test_dataset = H5BraTSSliceDataset(LOCAL_H5["test"], augment=False)

with h5py.File(LOCAL_H5["train"], "r") as f:
    train_labels = f["has_tumor"][:].astype(int)
counts = np.bincount(train_labels, minlength=2)
class_weight_map = {label: len(train_labels) / max(count, 1) for label, count in enumerate(counts)}
sample_weights = np.asarray([class_weight_map[int(y)] for y in train_labels], dtype=np.float64)
train_sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=CONFIG["num_workers"] > 0,
)
if CONFIG["num_workers"] > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(train_dataset, sampler=train_sampler, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("Train slices:", len(train_dataset))
print("Validation slices:", len(val_dataset))
print("Test slices:", len(test_dataset))
print("Training batches per epoch:", len(train_loader))
print("Effective batch size:", CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"])

## Part 2 — Models, losses, trainer and evaluation utilities (original protocol)
Architecture, loss, optimiser, scheduler, checkpointing and evaluation code are unchanged, so new seeds are directly comparable with the existing ones and every saved checkpoint loads with `strict=True`.

In [ ]:
# CELL 8 — PH-CVT U-Net architecture
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1, bias=False)
    def forward(self, x):
        residual = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)

class ASPP(nn.Module):
    def __init__(self, channels, dropout=0.15):
        super().__init__()
        branch_channels = channels // 4
        def branch(kernel_size, dilation):
            padding = dilation if kernel_size == 3 else 0
            return nn.Sequential(nn.Conv2d(channels, branch_channels, kernel_size, padding=padding, dilation=dilation, bias=False), nn.BatchNorm2d(branch_channels), nn.GELU())
        self.branches = nn.ModuleList([branch(1,1), branch(3,1), branch(3,2), branch(3,4)])
        self.project = nn.Sequential(nn.Conv2d(branch_channels*4, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.GELU(), nn.Dropout2d(dropout))
    def forward(self, x): return self.project(torch.cat([b(x) for b in self.branches], dim=1))

class TransformerBottleneck(nn.Module):
    def __init__(self, channels=384, spatial_size=10, depth=4, heads=8, dropout=0.15):
        super().__init__()
        self.position = nn.Parameter(torch.zeros(1, spatial_size*spatial_size, channels))
        nn.init.trunc_normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=channels, nhead=heads, dim_feedforward=channels*4, dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(channels)
    def forward(self, x):
        b,c,h,w = x.shape
        tokens = x.flatten(2).transpose(1,2)
        if tokens.shape[1] != self.position.shape[1]:
            raise ValueError(f"Expected {self.position.shape[1]} tokens, got {tokens.shape[1]}")
        tokens = self.norm(self.encoder(tokens + self.position))
        return tokens.transpose(1,2).reshape(b,c,h,w)

class AttentionFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(channels*2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.Sigmoid())
        self.refine = ResidualConvBlock(channels, channels, dropout=0.15)
    def forward(self, cnn_features, transformer_features):
        alpha = self.gate(torch.cat([cnn_features, transformer_features], dim=1))
        return self.refine(alpha*transformer_features + (1-alpha)*cnn_features)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualConvBlock(out_channels+skip_channels, out_channels, dropout=dropout)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]: x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))

class ProbabilisticHybridCNNViTUNet(nn.Module):
    MODEL_NAME = "PH-CVT U-Net"
    def __init__(self, in_channels=4, region_channels=3, image_size=160):
        super().__init__()
        if image_size % 16 != 0: raise ValueError("image_size must be divisible by 16")
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ResidualConvBlock(in_channels,32)
        self.enc2 = ResidualConvBlock(32,64)
        self.enc3 = ResidualConvBlock(64,128)
        self.enc4 = ResidualConvBlock(128,256,dropout=0.05)
        self.cnn_bottleneck = ResidualConvBlock(256,384,dropout=0.10)
        self.aspp = ASPP(384,dropout=0.15)
        self.transformer = TransformerBottleneck(384,image_size//16,4,8,0.15)
        self.fusion = AttentionFusion(384)
        self.dec4 = DecoderBlock(384,256,256,0.10)
        self.dec3 = DecoderBlock(256,128,128,0.08)
        self.dec2 = DecoderBlock(128,64,64,0.05)
        self.dec1 = DecoderBlock(64,32,32,0.03)
        self.segmentation_head = nn.Conv2d(32,region_channels,1)
        self.auxiliary_head = nn.Conv2d(128,region_channels,1)
        self.detection_head = nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(384,128),nn.GELU(),nn.Dropout(0.35),nn.Linear(128,1))
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        cnn=self.aspp(self.cnn_bottleneck(self.pool(e4)))
        fused=self.fusion(cnn,self.transformer(cnn))
        d4=self.dec4(fused,e4); d3=self.dec3(d4,e3); d2=self.dec2(d3,e2); d1=self.dec1(d2,e1)
        seg=self.segmentation_head(d1)
        aux=F.interpolate(self.auxiliary_head(d3),size=seg.shape[-2:],mode="bilinear",align_corners=False)
        det=self.detection_head(fused).squeeze(1)
        return {"seg_logits":seg,"aux_logits":aux,"det_logit":det}

model = ProbabilisticHybridCNNViTUNet(4,3,CONFIG["image_size"]).to(DEVICE)
if CONFIG["use_channels_last"] and DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
with torch.no_grad():
    dummy = torch.randn(2,4,CONFIG["image_size"],CONFIG["image_size"],device=DEVICE)
    if CONFIG["use_channels_last"] and DEVICE.type == "cuda": dummy=dummy.contiguous(memory_format=torch.channels_last)
    outputs=model(dummy)
parameter_count=sum(p.numel() for p in model.parameters())
print("Model:",model.MODEL_NAME)
print("Segmentation output:",outputs["seg_logits"].shape)
print("Detection output:",outputs["det_logit"].shape)
print(f"Parameters: {parameter_count/1e6:.2f} M")
del dummy,outputs
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# CELL 8b — PH-CVT-Lite U-Net: a ~3.6M-parameter variant of the proposed model
# Same architecture (Residual CNN encoder -> ASPP -> Transformer bottleneck ->
# attention fusion -> Residual decoder -> seg/aux/detection heads), only the
# widths, bottleneck channels and transformer depth are reduced.
# Nothing in the original ProbabilisticHybridCNNViTUNet class above is modified.

class ProbabilisticHybridCNNViTUNetLite(nn.Module):
    MODEL_NAME = "PH-CVT-Lite U-Net"
    def __init__(self, in_channels=4, region_channels=3, image_size=160,
                 widths=(16, 32, 64, 128), bottleneck=192, transformer_depth=2, transformer_heads=4):
        super().__init__()
        if image_size % 16 != 0: raise ValueError("image_size must be divisible by 16")
        self.loss_weights = (1., .2, .25)  # matches full PHCVT default: seg + aux + detection
        w1, w2, w3, w4 = widths
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ResidualConvBlock(in_channels, w1)
        self.enc2 = ResidualConvBlock(w1, w2)
        self.enc3 = ResidualConvBlock(w2, w3)
        self.enc4 = ResidualConvBlock(w3, w4, dropout=0.05)
        self.cnn_bottleneck = ResidualConvBlock(w4, bottleneck, dropout=0.10)
        self.aspp = ASPP(bottleneck, dropout=0.15)
        self.transformer = TransformerBottleneck(bottleneck, image_size // 16, transformer_depth, transformer_heads, 0.15)
        self.fusion = AttentionFusion(bottleneck)
        self.dec4 = DecoderBlock(bottleneck, w4, w4, 0.10)
        self.dec3 = DecoderBlock(w4, w3, w3, 0.08)
        self.dec2 = DecoderBlock(w3, w2, w2, 0.05)
        self.dec1 = DecoderBlock(w2, w1, w1, 0.03)
        self.segmentation_head = nn.Conv2d(w1, region_channels, 1)
        self.auxiliary_head = nn.Conv2d(w3, region_channels, 1)
        self.detection_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(bottleneck, 64), nn.GELU(), nn.Dropout(0.35), nn.Linear(64, 1)
        )

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        cnn = self.aspp(self.cnn_bottleneck(self.pool(e4)))
        fused = self.fusion(cnn, self.transformer(cnn))
        d4 = self.dec4(fused, e4); d3 = self.dec3(d4, e3); d2 = self.dec2(d3, e2); d1 = self.dec1(d2, e1)
        seg = self.segmentation_head(d1)
        aux = F.interpolate(self.auxiliary_head(d3), size=seg.shape[-2:], mode="bilinear", align_corners=False)
        det = self.detection_head(fused).squeeze(1)
        return {"seg_logits": seg, "aux_logits": aux, "det_logit": det}

# --- sanity check: forward pass + parameter count ---
lite_model = ProbabilisticHybridCNNViTUNetLite(4, 3, CONFIG["image_size"]).to(DEVICE)
if CONFIG["use_channels_last"] and DEVICE.type == "cuda":
    lite_model = lite_model.to(memory_format=torch.channels_last)
with torch.no_grad():
    dummy = torch.randn(2, 4, CONFIG["image_size"], CONFIG["image_size"], device=DEVICE)
    if CONFIG["use_channels_last"] and DEVICE.type == "cuda": dummy = dummy.contiguous(memory_format=torch.channels_last)
    outputs = lite_model(dummy)
lite_param_count = sum(p.numel() for p in lite_model.parameters())
print("Model:", lite_model.MODEL_NAME)
print("Segmentation output:", outputs["seg_logits"].shape)
print("Detection output:", outputs["det_logit"].shape)
print(f"Parameters: {lite_param_count/1e6:.2f} M  (full PH-CVT was {parameter_count/1e6:.2f} M)")
del dummy, outputs
if torch.cuda.is_available(): torch.cuda.empty_cache()

# To actually train this instead of the full model, either:
#   1) point the existing training loop at `lite_model` instead of `model`, or
#   2) add "PHCVT_lite" to STUDY['models'] and register it in make_model() (Cell 24)
#      by returning ProbabilisticHybridCNNViTUNetLite(4,3,CONFIG["image_size"]).
# Widths/bottleneck/transformer_depth are constructor args — tune them to trade
# parameters against accuracy, e.g. widths=(24,48,96,160), bottleneck=256 for a
# ~6-7M mid-size version, or drop transformer_depth to 1 / remove AttentionFusion
# (replace self.fusion(cnn, self.transformer(cnn)) with cnn + self.transformer(cnn))
# to push below 2M.

In [ ]:
# CELL 9 — Losses and metric utilities
REGION_NAMES = ["WT", "TC", "ET"]

def estimate_region_positive_weights(dataset, max_samples=2000):
    indices = np.linspace(0, len(dataset)-1, min(max_samples,len(dataset))).round().astype(int)
    positives=np.zeros(3,dtype=np.float64); total=0
    for idx in tqdm(indices,desc="Estimating region weights"):
        regions=dataset[idx]["regions"].numpy(); positives += regions.sum(axis=(1,2)); total += regions.shape[1]*regions.shape[2]
    return np.clip((total-positives)/np.maximum(positives,1),1.0,12.0).astype(np.float32)

region_pos_weights=torch.tensor(estimate_region_positive_weights(train_dataset),device=DEVICE).view(1,3,1,1)
detection_pos_weight=torch.tensor([1.0],dtype=torch.float32,device=DEVICE)
print("Region positive weights:",region_pos_weights.flatten().tolist())

class DiceBCELoss(nn.Module):
    def __init__(self,pos_weights,smooth=1.0):
        super().__init__(); self.register_buffer("pos_weights",pos_weights); self.smooth=smooth
    def forward(self,logits,targets):
        base=F.binary_cross_entropy_with_logits(logits,targets,reduction="none")
        bce=(base*(1.0+targets*(self.pos_weights-1.0))).mean()
        probs=torch.sigmoid(logits); dims=(0,2,3)
        inter=(probs*targets).sum(dims); denom=probs.sum(dims)+targets.sum(dims)
        dice=(2*inter+self.smooth)/(denom+self.smooth)
        return 0.55*(1-dice.mean())+0.45*bce

segmentation_loss_fn=DiceBCELoss(region_pos_weights)
detection_loss_fn=nn.BCEWithLogitsLoss(pos_weight=detection_pos_weight)

def empty_region_stats(device=DEVICE):
    return {k:torch.zeros(3,dtype=torch.double,device=device) for k in ["intersection","pred_sum","target_sum","union","tp","fp","fn","tn"]}

def batch_region_statistics(logits,targets,thresholds=(0.5,0.5,0.5)):
    probs=torch.sigmoid(logits); th=torch.tensor(thresholds,device=probs.device).view(1,3,1,1)
    pred=probs>=th; targ=targets>=0.5
    return {"intersection":(pred&targ).sum((0,2,3)).double(),"pred_sum":pred.sum((0,2,3)).double(),"target_sum":targ.sum((0,2,3)).double(),"union":(pred|targ).sum((0,2,3)).double(),"tp":(pred&targ).sum((0,2,3)).double(),"fp":(pred&~targ).sum((0,2,3)).double(),"fn":((~pred)&targ).sum((0,2,3)).double(),"tn":((~pred)&(~targ)).sum((0,2,3)).double()}

def add_stats(total,batch):
    for k in total: total[k]+=batch[k]
    return total

def metrics_from_stats(stats):
    e=1e-8
    return {"dice":((2*stats["intersection"]+e)/(stats["pred_sum"]+stats["target_sum"]+e)).cpu().numpy(),"iou":((stats["intersection"]+e)/(stats["union"]+e)).cpu().numpy(),"precision":((stats["tp"]+e)/(stats["tp"]+stats["fp"]+e)).cpu().numpy(),"recall":((stats["tp"]+e)/(stats["tp"]+stats["fn"]+e)).cpu().numpy(),"specificity":((stats["tn"]+e)/(stats["tn"]+stats["fp"]+e)).cpu().numpy()}

In [ ]:
# CELL 10 — Training/validation functions and atomic checkpoint utility

def autocast_context():
    return torch.cuda.amp.autocast(enabled=AMP_ENABLED) if DEVICE.type=="cuda" else nullcontext()

def prepare_images(images):
    images=images.to(DEVICE,non_blocking=True)
    if CONFIG["use_channels_last"] and DEVICE.type=="cuda": images=images.contiguous(memory_format=torch.channels_last)
    return images

def run_epoch(model,loader,optimizer=None,scaler=None):
    training=optimizer is not None; model.train(training)
    total_loss=0.0; total_items=0; stats=empty_region_stats(); labels_all=[]; probs_all=[]
    if training: optimizer.zero_grad(set_to_none=True)
    progress=tqdm(loader,desc="Train" if training else "Validate",leave=False)
    for step,batch in enumerate(progress):
        images=prepare_images(batch["image"]); regions=batch["regions"].to(DEVICE,non_blocking=True); det_targets=batch["has_tumor"].to(DEVICE,non_blocking=True)
        with autocast_context():
            out=model(images)
            weights=getattr(model,"loss_weights",(1.0,0.20,0.25))
            loss=weights[0]*segmentation_loss_fn(out["seg_logits"],regions)
            if weights[1]: loss=loss+weights[1]*segmentation_loss_fn(out["aux_logits"],regions)
            if weights[2]: loss=loss+weights[2]*detection_loss_fn(out["det_logit"],det_targets)
            if not torch.isfinite(loss): raise FloatingPointError("Non-finite loss")
        if training:
            scaler.scale(loss/CONFIG["gradient_accumulation_steps"]).backward()
            should_step=(step+1)%CONFIG["gradient_accumulation_steps"]==0 or step+1==len(loader)
            if should_step:
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                callback=globals().get('_OPTIMIZER_BOUNDARY_CALLBACK')
                if callback is not None: callback()
        bs=images.shape[0]; total_loss+=float(loss.detach())*bs; total_items+=bs
        add_stats(stats,batch_region_statistics(out["seg_logits"].detach(),regions))
        if "det_logit" in out:
            probs_all.extend(torch.sigmoid(out["det_logit"]).detach().cpu().numpy().tolist()); labels_all.extend(det_targets.detach().cpu().numpy().astype(int).tolist())
        progress.set_postfix(loss=f"{float(loss.detach()):.4f}")
    region=metrics_from_stats(stats); pred=(np.asarray(probs_all)>=0.5).astype(int)
    det={"accuracy":accuracy_score(labels_all,pred),"precision":precision_score(labels_all,pred,zero_division=0),"recall":recall_score(labels_all,pred,zero_division=0),"f1":f1_score(labels_all,pred,zero_division=0)} if labels_all else {k:float("nan") for k in ["accuracy","precision","recall","f1"]}
    return {"loss":total_loss/max(total_items,1),"dice":region["dice"],"mean_dice":float(np.mean(region["dice"])),"detection":det}

def atomic_torch_save(payload,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp")
    torch.save(payload,tmp); os.replace(tmp,path)

In [ ]:
# Unified trainer: old checkpoints are accepted strictly; new checkpoints retain provenance.
def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    if not state: return
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if torch.cuda.is_available() and state['cuda']:
        torch.cuda.set_rng_state_all([v.cpu() for v in state['cuda']])

def atomic_torch_save(payload, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with _WRITE_LOCK:
        torch.save(payload, tmp)
        os.replace(tmp, path)

def fit_experiment(net, folder, name, seed=42, require_existing=False):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    last, best = folder / 'last_checkpoint.pt', folder / 'best_model.pt'
    hp = folder / 'training_history.csv'
    # Per-epoch full-weight snapshots were removed: they were never read by any
    # resume/eval path (only last_checkpoint.pt and best_model.pt are), and at up
    # to 100 epochs per model/seed they were the single biggest driver of Drive usage.
    signature = {'name': name, 'seed': seed, 'split_sha256': file_hash(SPLIT_PATH),
                 'image_size': CONFIG['image_size'], 'batch_size': CONFIG['batch_size'],
                 'accumulation': CONFIG['gradient_accumulation_steps'],
                 'loss_weights': list(getattr(net, 'loss_weights', (1., .20, .25))),
                 'state_shapes': {k: list(v.shape) for k, v in net.state_dict().items()}}
    if require_existing and (not last.exists() or not best.exists()):
        raise FileNotFoundError(f'Original continuation requires BOTH {last} and {best}; no silent fresh training.')
    try:
        opt = torch.optim.AdamW(net.parameters(), lr=CONFIG['learning_rate'],
                              weight_decay=CONFIG['weight_decay'], fused=DEVICE.type == 'cuda')
    except TypeError:
        opt = torch.optim.AdamW(net.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=.5, patience=2, min_lr=1e-6)
    amp = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    hist=[]; completed_epoch=-1; best_score=-1.; bad=0
    if last.exists():
        ck = torch.load(last, map_location='cpu', weights_only=False)
        if ck.get('signature') is not None and ck['signature'] != signature:
            raise ValueError(f'Checkpoint protocol mismatch for {name}; use a separate experiment folder.')
        net.load_state_dict(ck['model_state'], strict=True)
        opt.load_state_dict(ck['optimizer_state']); sched.load_state_dict(ck['scheduler_state'])
        if ck.get('scaler_state'): amp.load_state_dict(ck['scaler_state'])
        hist=ck.get('history', []); completed_epoch=int(ck['epoch'])
        best_score=float(ck.get('best_val_dice', -1)); bad=int(ck.get('epochs_without_improvement', 0))
        restore_rng(ck.get('rng_state'))
        for attr,key in [('sampler','sampler_rng'),(None,'loader_rng')]:
            obj=getattr(train_loader,attr,None) if attr else train_loader
            generator=getattr(obj,'generator',None)
            if generator is not None and ck.get(key) is not None: generator.set_state(ck[key].cpu())
        print(f'{name}: resume epoch {completed_epoch+2}, best={best_score:.6f}, patience used={bad}/15')
        del ck
    if best.exists():
        bc = torch.load(best, map_location='cpu', weights_only=False)
        best_score=max(best_score, float(bc.get('best_val_dice', -1)))
        del bc
    if completed_epoch >= 0 and not best.exists():
        raise FileNotFoundError(f'Missing best checkpoint for resumed run: {best}')
    def payload(epoch, complete=True):
        return {'model_state': net.state_dict(), 'optimizer_state': opt.state_dict(),
                'scheduler_state': sched.state_dict(), 'scaler_state': amp.state_dict(),
                'epoch': epoch, 'epoch_complete': complete, 'history': hist,
                'best_val_dice': best_score, 'epochs_without_improvement': bad,
                'config': CONFIG.copy(), 'signature': signature, 'rng_state': rng_state(),
                'sampler_rng':getattr(getattr(train_loader,'sampler',None),'generator',None).get_state() if getattr(getattr(train_loader,'sampler',None),'generator',None) is not None else None,
                'loader_rng':train_loader.generator.get_state() if getattr(train_loader,'generator',None) is not None else None}
    if not last.exists(): atomic_torch_save(payload(-1), last)
    last_snapshot = time.monotonic()
    def snapshot_boundary():
        nonlocal last_snapshot
        if (folder / 'STOP_REQUESTED').exists():
            raise KeyboardInterrupt('STOP_REQUESTED file found')
        if time.monotonic()-last_snapshot >= STUDY['partial_snapshot_seconds']:
            atomic_torch_save(payload(completed_epoch, False), folder / 'partial_checkpoint.pt')
            last_snapshot=time.monotonic()
    global _OPTIMIZER_BOUNDARY_CALLBACK
    _OPTIMIZER_BOUNDARY_CALLBACK=snapshot_boundary
    RUN_STATUS[name]='training'
    try:
        for epoch in range(completed_epoch+1, CONFIG['epochs']):
            if bad >= CONFIG['early_stopping_patience']: break
            t=time.time()
            tr=run_epoch(net, train_loader, opt, amp)
            with torch.no_grad(): va=run_epoch(net, val_loader)
            if not np.isfinite(va['mean_dice']): raise FloatingPointError('Non-finite validation Dice')
            sched.step(va['mean_dice'])
            row={'epoch': epoch+1, 'learning_rate':opt.param_groups[0]['lr'],
                 'train_loss':tr['loss'], 'val_loss':va['loss'],
                 'train_mean_dice':tr['mean_dice'], 'val_mean_dice':va['mean_dice'],
                 'val_wt_dice':va['dice'][0], 'val_tc_dice':va['dice'][1], 'val_et_dice':va['dice'][2],
                 'val_detection_f1':va['detection']['f1'], 'epoch_seconds':time.time()-t}
            hist.append(row)
            improved=va['mean_dice'] > best_score + 1e-4
            if improved: best_score=float(va['mean_dice']); bad=0
            else: bad+=1
            weights={'model_state':net.state_dict(), 'epoch':epoch, 'config':CONFIG.copy(),
                     'model_name':name, 'best_val_dice':best_score, 'signature':signature}
            if improved: atomic_torch_save(weights, best)
            completed_epoch=epoch
            atomic_torch_save(payload(epoch), last)
            tmp=hp.with_suffix('.csv.tmp'); pd.DataFrame(hist).to_csv(tmp,index=False); os.replace(tmp,hp)
            print(f'{name} epoch {epoch+1}/100 | val={va["mean_dice"]:.5f} | best={best_score:.5f} | patience={bad}/15')
        RUN_STATUS[name]='complete'
    except BaseException as exc:
        # Keep the last completed epoch intact. Partial states are separate and never silently resumed.
        try:
            atomic_torch_save(payload(completed_epoch, False), folder / 'interrupted_checkpoint.pt')
            atomic_json({'error_type':type(exc).__name__, 'last_completed_epoch':completed_epoch+1,
                         'resume_policy':'restore last_checkpoint.pt and replay unfinished epoch'}, folder/'interruption.json')
        finally:
            RUN_STATUS[name]='interrupted'; opt.zero_grad(set_to_none=True)
        raise
    finally:
        _OPTIMIZER_BOUNDARY_CALLBACK=None
        atomic_json(RUN_STATUS, STUDY_DIR/f'run_status_session{SESSION}.json')   # per-session file (parallel runtimes)
    if not best.exists(): raise RuntimeError(f'No validated checkpoint produced for {name}')
    net.load_state_dict(torch.load(best, map_location='cpu', weights_only=False)['model_state'], strict=True)
    net.eval()
    return pd.DataFrame(hist), best_score

# (The original call that re-validated the finished PH-CVT seed-42 run is not needed here.)

In [ ]:
# =============================================================================
# CELL 13 (adapted) — Validation-only threshold calibration utilities
# =============================================================================
# The two functions and the candidate grid are copied verbatim from the original notebook,
# so new seeds are calibrated exactly like seeds 42/123/2026 (grid 0.25-0.75, step 0.05).
# The frozen seed-42 thresholds were calibrated on validation data earlier; they are LOADED
# here (never recomputed, never overwritten).
@torch.no_grad()
def calibrate_segmentation_thresholds(model,loader,candidates):
    intersections=np.zeros((3,len(candidates))); pred_sums=np.zeros((3,len(candidates))); target_sums=np.zeros(3)
    for batch in tqdm(loader,desc="Calibrating segmentation"):
        images=prepare_images(batch["image"]); targets=batch["regions"].to(DEVICE).bool(); probs=torch.sigmoid(model(images)["seg_logits"])
        target_sums+=targets.sum((0,2,3)).cpu().numpy()
        for j,t in enumerate(candidates):
            pred=probs>=t; intersections[:,j]+=(pred&targets).sum((0,2,3)).cpu().numpy(); pred_sums[:,j]+=pred.sum((0,2,3)).cpu().numpy()
    grid=(2*intersections+1e-8)/(pred_sums+target_sums[:,None]+1e-8); idx=grid.argmax(1)
    return [float(candidates[i]) for i in idx],grid

@torch.no_grad()
def collect_detection_outputs(model,loader):
    labels=[]; probs=[]
    for batch in tqdm(loader,desc="Collecting detection outputs"):
        images=prepare_images(batch["image"]); probs.extend(torch.sigmoid(model(images)["det_logit"]).cpu().numpy().tolist()); labels.extend(batch["has_tumor"].numpy().astype(int).tolist())
    return np.asarray(labels),np.asarray(probs)

candidates=np.arange(0.25,0.76,0.05)

_thr = json.loads(THRESHOLD_PATH.read_text())
seg_thresholds = [float(_thr['segmentation'][r]) for r in REGION_NAMES]
print('Frozen PH-CVT seed-42 validation thresholds:', dict(zip(REGION_NAMES, np.round(seg_thresholds, 2))))

# The randomly initialised instances created by the architecture cells are not needed on the GPU.
model.to('cpu')
globals().pop("lite_model", None)   # safe if this cell is re-run
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Comparison models and trained ablations; the original PH-CVT class above is unchanged.
class PlainConv(nn.Sequential):
    def __init__(self, ci, co):
        super().__init__(nn.Conv2d(ci,co,3,padding=1,bias=False), nn.BatchNorm2d(co), nn.ReLU(),
                         nn.Conv2d(co,co,3,padding=1,bias=False), nn.BatchNorm2d(co), nn.ReLU())

class SkipAttention(nn.Module):
    def __init__(self, channels):
        super().__init__(); hidden=max(channels//2,1)
        self.g=nn.Sequential(nn.Conv2d(channels,hidden,1),nn.BatchNorm2d(hidden))
        self.x=nn.Sequential(nn.Conv2d(channels,hidden,1),nn.BatchNorm2d(hidden))
        self.psi=nn.Sequential(nn.ReLU(),nn.Conv2d(hidden,1,1),nn.Sigmoid())
    def forward(self, g, x): return x*self.psi(self.g(g)+self.x(x))

class CNNBaseline(nn.Module):
    loss_weights=(1.,0.,0.)
    def __init__(self, name):
        super().__init__(); self.MODEL_NAME=name
        block=ResidualConvBlock if name=='ResUNet' else PlainConv
        widths=[32,64,128,256,512]
        self.enc=nn.ModuleList([block(ci,co) for ci,co in zip([4]+widths[:-1],widths)])
        self.up=nn.ModuleList([nn.ConvTranspose2d(ci,co,2,2) for ci,co in zip(widths[:0:-1],widths[-2::-1])])
        self.dec=nn.ModuleList([block(2*c,c) for c in widths[-2::-1]])
        self.att=nn.ModuleList([SkipAttention(c) for c in widths[-2::-1]]) if name=='AttentionUNet' else None
        self.head=nn.Conv2d(32,3,1)
    def forward(self,x):
        skips=[]
        for i, enc in enumerate(self.enc):
            x=enc(x if i==0 else F.max_pool2d(x,2)); skips.append(x)
        for i,(up,dec,skip) in enumerate(zip(self.up,self.dec,reversed(skips[:-1]))):
            x=up(x)
            if self.att is not None: skip=self.att[i](x,skip)
            x=dec(torch.cat([x,skip],1))
        return {'seg_logits':self.head(x)}

class AverageFusion(nn.Module):
    def __init__(self):
        super().__init__(); self.refine=ResidualConvBlock(384,384,.15)
    def forward(self,a,b): return self.refine((a+b)/2)

class PHCVTAblation(ProbabilisticHybridCNNViTUNet):
    def __init__(self, variant):
        super().__init__(4,3,CONFIG['image_size']); self.variant=variant; self.MODEL_NAME=variant
        self.loss_weights=(1.,.2,.25)
        if variant=='PHCVT_no_transformer':
            self.transformer=nn.Identity(); self.fusion=nn.Identity()
        if variant=='PHCVT_no_aspp': self.aspp=nn.Identity()
        if variant=='PHCVT_no_attention_fusion': self.fusion=AverageFusion()
        if variant in ('PHCVT_segmentation_only','PHCVT_no_auxiliary'):
            self.auxiliary_head=nn.Identity()
            self.loss_weights=(1.,0.,0. if variant=='PHCVT_segmentation_only' else .25)
        if variant=='PHCVT_segmentation_only': self.detection_head=nn.Identity()
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        cnn=self.aspp(self.cnn_bottleneck(self.pool(e4)))
        fused=cnn if self.variant=='PHCVT_no_transformer' else self.fusion(cnn,self.transformer(cnn))
        d4=self.dec4(fused,e4); d3=self.dec3(d4,e3); d2=self.dec2(d3,e2); d1=self.dec1(d2,e1)
        out={'seg_logits':self.segmentation_head(d1)}
        if self.loss_weights[1]: out['aux_logits']=F.interpolate(self.auxiliary_head(d3),size=x.shape[-2:],mode='bilinear',align_corners=False)
        if self.loss_weights[2]: out['det_logit']=self.detection_head(fused).squeeze(1)
        return out

# Official transformer implementations. Lock to the first fetched commit, then reuse the
# saved source bytes with SHA-256 verification. No network is needed on subsequent runs.
import urllib.request, importlib.util, types, copy
SOURCE_ROOT=STUDY_DIR/'baseline_sources'; SOURCE_ROOT.mkdir(exist_ok=True)

def official_source(repo, filenames, package_name):
    dest=SOURCE_ROOT/package_name; dest.mkdir(exist_ok=True)
    lock=dest/'source_lock.json'
    if lock.exists():
        record=json.loads(lock.read_text())
    else:
        request=urllib.request.Request(f'https://api.github.com/repos/{repo}/commits/main',headers={'User-Agent':'PHCVT-reproducible-study'})
        with urllib.request.urlopen(request,timeout=60) as response: commit=json.load(response)['sha']
        record={'repository':repo,'commit':commit,'files':{}}
        for filename in filenames:
            url=f'https://raw.githubusercontent.com/{repo}/{commit}/networks/{filename}'
            with urllib.request.urlopen(url,timeout=60) as response: data=response.read()
            (dest/filename).write_bytes(data)
            record['files'][filename]=hashlib.sha256(data).hexdigest()
        atomic_json(record,lock)
    for fn, sha in record['files'].items():
        if file_hash(dest/fn)!=sha: raise RuntimeError(f'Modified baseline source: {dest/fn}')
    pkg=types.ModuleType(package_name); pkg.__path__=[str(dest)]; sys.modules[package_name]=pkg
    return dest,record

class OfficialSegmentation(nn.Module):
    loss_weights=(1.,0.,0.)
    def __init__(self, core, name, provenance):
        super().__init__(); self.core=core; self.MODEL_NAME=name; self.provenance=provenance
    def forward(self,x): return {'seg_logits':self.core(x)}

def make_model(name):
    if name in ['UNet','ResUNet','AttentionUNet']: return CNNBaseline(name)
    if name == 'PHCVT_lite': return ProbabilisticHybridCNNViTUNetLite(4, 3, CONFIG['image_size'])
    if name.startswith('PHCVT_'): return PHCVTAblation(name)
    if name=='SwinUnet':
        dest,rec=official_source('HuCaoFighting/Swin-Unet', ['swin_transformer_unet_skip_expand_decoder_sys.py'], 'phcvt_swin_reference')
        from phcvt_swin_reference.swin_transformer_unet_skip_expand_decoder_sys import SwinTransformerSys
        # Window 5 divides the 40/20/10/5 feature grids at native 160 resolution.
        core=SwinTransformerSys(img_size=160,patch_size=4,in_chans=4,num_classes=3,
             embed_dim=96,depths=[2,2,2,2],depths_decoder=[1,2,2,2],num_heads=[3,6,12,24],
             window_size=5,mlp_ratio=4.,qkv_bias=True,drop_rate=0.,drop_path_rate=.1,
             ape=False,patch_norm=True,use_checkpoint=True,final_upsample='expand_first')
        rec=dict(rec, adaptation='4 input channels, 3 independent region logits, 160x160/window5; random initialization')
        return OfficialSegmentation(core,name,rec)
    if name=='TransUNet':
        dest,rec=official_source('Beckschen/TransUNet', ['vit_seg_modeling.py','vit_seg_configs.py','vit_seg_modeling_resnet_skip.py'], 'phcvt_transunet_reference')
        from phcvt_transunet_reference.vit_seg_modeling import VisionTransformer, CONFIGS
        cfg=copy.deepcopy(CONFIGS['R50-ViT-B_16']); cfg.n_classes=3; cfg.n_skip=3; cfg.patches.grid=(10,10)
        core=VisionTransformer(cfg,img_size=160,num_classes=3)
        old=core.transformer.embeddings.hybrid_model.root.conv
        core.transformer.embeddings.hybrid_model.root.conv=type(old)(4,old.out_channels,
            kernel_size=old.kernel_size,stride=old.stride,padding=old.padding,bias=False)
        rec=dict(rec, adaptation='R50-ViT-B_16; 4 input channels, 3 independent region logits; 160x160; random initialization')
        return OfficialSegmentation(core,name,rec)
    raise ValueError(name)

# Installation happens here, after proposed training; torch is never reinstalled.
if STUDY['run_comparisons'] and ALLOW_TRAINING and any(x in STUDY['models'] for x in ['SwinUnet','TransUNet']):
    import subprocess
    subprocess.run([sys.executable,'-m','pip','install','-q','timm==1.0.15','einops==0.8.1','ml-collections==1.0.0'],check=True)

In [ ]:
# Patient-level comparisons, calibration, efficiency and visual explanations.
def bootstrap_mean(values, seed=42):
    values=np.asarray(values,dtype=float); values=values[np.isfinite(values)]
    if not len(values): return [float('nan')]*3
    rng=np.random.default_rng(seed)
    means=np.array([rng.choice(values,len(values),replace=True).mean() for _ in range(STUDY['bootstrap_samples'])])
    return [float(values.mean()), *np.quantile(means,[.025,.975]).tolist()]

@torch.no_grad()
def assess_segmentation(net, loader, thresholds, folder, prefix='test'):
    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True); net.eval()
    total=empty_region_stats(); by_patient={}; seen=set()
    # Reliability bins: region-wise positive-class probability, full-image and foreground-balanced ECE.
    counts=np.zeros((3,2,15)); conf=np.zeros_like(counts); truth=np.zeros_like(counts)
    uncertainty_rows=[]
    th=torch.as_tensor(thresholds,device=DEVICE).view(1,3,1,1)
    for batch in tqdm(loader,desc=prefix+' segmentation'):
        x=prepare_images(batch['image']); y=batch['regions'].to(DEVICE)
        with autocast_context(): logits=net(x)['seg_logits']
        p=logits.float().sigmoid(); pred=p>=th; target=y>=.5
        add_stats(total,batch_region_statistics(logits,y,thresholds))
        for i,pid in enumerate(batch['patient_id']):
            key=(str(pid),int(batch['slice_idx'][i]))
            if key in seen: raise ValueError(f'Duplicate evaluation slice: {key}')
            seen.add(key)
            one=batch_region_statistics(logits[i:i+1],y[i:i+1],thresholds)
            if pid not in by_patient: by_patient[pid]=empty_region_stats()
            add_stats(by_patient[pid],one)
            q=p[i].clamp(1e-7,1-1e-7)
            entropy=-(q*q.log()+(1-q)*(1-q).log())
            uncertainty_rows.append({'patient_id':pid,'slice_idx':key[1],
                'mean_predictive_entropy':float(entropy.mean()),
                'mean_pixel_error':float((pred[i]!=target[i]).float().mean())})
        pp=p.cpu().numpy(); yy=target.cpu().numpy()
        for r in range(3):
            for cls in [0,1]:
                mask=yy[:,r]==bool(cls); vals=pp[:,r][mask]
                bins=np.minimum((vals*15).astype(int),14)
                counts[r,cls]+=np.bincount(bins,minlength=15)
                conf[r,cls]+=np.bincount(bins,weights=vals,minlength=15)
                truth[r,cls]+=np.bincount(bins,weights=np.full(len(vals),cls),minlength=15)
    if not seen: raise ValueError('Empty evaluation set')
    rows=[]
    for pid,st in by_patient.items():
        metrics=metrics_from_stats(st)
        for r,region in enumerate(REGION_NAMES):
            rows.append({'patient_id':pid,'region':region,**{k:float(v[r]) for k,v in metrics.items()},
                         'target_present':bool(st['target_sum'][r]>0)})
    patients=pd.DataFrame(rows); patients.to_csv(folder/f'{prefix}_patients.csv',index=False)
    micro=metrics_from_stats(total); summaries=[]
    for r,region in enumerate(REGION_NAMES):
        part=patients[patients.region==region]
        mean,lo,hi=bootstrap_mean(part.dice)
        summaries.append({'region':region,'patient_mean_dice':mean,'ci95_low':lo,'ci95_high':hi,
                          'micro_dice':float(micro['dice'][r]),'micro_iou':float(micro['iou'][r]),
                          'target_present_patient_dice':float(part.loc[part.target_present,'dice'].mean()),
                          'n_patients':len(part)})
    summary=pd.DataFrame(summaries); summary.to_csv(folder/f'{prefix}_summary.csv',index=False)
    pd.DataFrame(uncertainty_rows).groupby('patient_id')[['mean_predictive_entropy','mean_pixel_error']].mean().to_csv(folder/f'{prefix}_uncertainty_by_patient.csv')
    reliability=[]
    for r,region in enumerate(REGION_NAMES):
        for mode in ['full_image','foreground_balanced']:
            weights=np.ones(2) if mode=='full_image' else .5/np.maximum(counts[r].sum(1),1)
            n=(counts[r]*weights[:,None]).sum(0); c=(conf[r]*weights[:,None]).sum(0); t=(truth[r]*weights[:,None]).sum(0)
            cp=np.divide(c,n,out=np.zeros_like(c),where=n>0); tp=np.divide(t,n,out=np.zeros_like(t),where=n>0)
            ece=float((abs(cp-tp)*n).sum()/max(n.sum(),1e-12))
            for b in range(15): reliability.append({'region':region,'mode':mode,'bin':b,'weight':n[b],'confidence':cp[b],'observed_frequency':tp[b],'ece':ece})
    rel=pd.DataFrame(reliability); rel.to_csv(folder/f'{prefix}_reliability.csv',index=False)
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for ax,r in zip(axes,REGION_NAMES):
        for mode in rel['mode'].unique():
            part=rel[(rel.region==r)&(rel['mode']==mode)&(rel.weight>0)]
            ax.plot(part.confidence,part.observed_frequency,'o-',label=f'{mode}: ECE={part.ece.iloc[0]:.4f}')
        ax.plot([0,1],[0,1],'k--'); ax.set(title=r,xlabel='Predicted probability',ylabel='Observed foreground frequency'); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(folder/f'{prefix}_calibration.png',dpi=180); plt.close(fig)
    atomic_json({'patients':len(by_patient),'slices':len(seen),'empty_region_convention':'both empty=1; report target-present subset too',
                 'thresholds':list(thresholds),'inference':'deterministic single checkpoint'},folder/f'{prefix}_evaluation_protocol.json')
    return patients,summary

@torch.no_grad()
def measure_efficiency(net, repeats=50):
    net.eval(); x=torch.randn(1,4,160,160,device=DEVICE)
    if CONFIG['use_channels_last'] and DEVICE.type=='cuda': x=x.contiguous(memory_format=torch.channels_last)
    for _ in range(10):
        with autocast_context(): net(x)
    if DEVICE.type=='cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    times=[]
    for _ in range(repeats):
        start=time.perf_counter()
        with autocast_context(): out=net(x)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        times.append((time.perf_counter()-start)*1000)
        del out
    return {'parameters':sum(p.numel() for p in net.parameters()),'latency_median_ms':float(np.median(times)),
            'latency_p95_ms':float(np.percentile(times,95)),
            'peak_allocated_mb':torch.cuda.max_memory_allocated()/2**20 if DEVICE.type=='cuda' else None,
            'device':torch.cuda.get_device_name() if DEVICE.type=='cuda' else 'CPU',
            'precision':'AMP' if AMP_ENABLED else 'float32','timing_batch_size':1,'timing_repeats':repeats}

def cam_layer(net):
    if hasattr(net,'dec1'): return net.dec1.block.conv2
    if isinstance(net,CNNBaseline): return net.dec[-1]
    if net.MODEL_NAME=='SwinUnet': return net.core.layers_up[-1].blocks[-1].norm2
    if net.MODEL_NAME=='TransUNet': return net.core.decoder.blocks[-1]
    raise ValueError('No Grad-CAM target layer configured')

def segmentation_gradcam(net,x,region=0):
    net.eval(); activations=[]
    checkpoint_flags={m:m.use_checkpoint for m in net.modules() if hasattr(m,'use_checkpoint')}
    for m in checkpoint_flags: m.use_checkpoint=False
    handle=cam_layer(net).register_forward_hook(lambda module,args,out:activations.append(out))
    try:
        with torch.enable_grad():
            out=net(x.requires_grad_(True))['seg_logits']
            roi=(out[:,region].detach().sigmoid()>=.5).float()
            if not roi.any(): roi=torch.ones_like(roi)
            score=(out[:,region]*roi).sum()/roi.sum().clamp_min(1)
            act=activations[-1]; grad=torch.autograd.grad(score,act)[0]
            if act.ndim==3:
                b,n,c=act.shape; side=math.isqrt(n)
                if side*side!=n: raise ValueError('Non-square token map in Grad-CAM')
                act=act.transpose(1,2).reshape(b,c,side,side); grad=grad.transpose(1,2).reshape(b,c,side,side)
            cam=(grad.mean((2,3),keepdim=True)*act).sum(1,keepdim=True).relu()
            cam=F.interpolate(cam,size=x.shape[-2:],mode='bilinear',align_corners=False)
            cam=cam/(cam.amax((2,3),keepdim=True)+1e-8)
            return cam[0,0].detach().cpu().numpy()
    finally:
        handle.remove(); net.zero_grad(set_to_none=True)
        for m,flag in checkpoint_flags.items(): m.use_checkpoint=flag

@torch.no_grad()
def segmentation_mc(net,x,passes):
    states={m:m.training for m in net.modules()}; net.eval()
    dropout_count=0
    for m in net.modules():
        if isinstance(m,(nn.Dropout,nn.Dropout2d,nn.Dropout3d)) and m.p>0: m.train(); dropout_count+=1
    mean=None; m2=None; mean_ent=None
    try:
        for k in range(passes):
            p=net(x)['seg_logits'].float().sigmoid().clamp(1e-7,1-1e-7)
            ent=-(p*p.log()+(1-p)*(1-p).log())
            if mean is None: mean=p.clone(); m2=torch.zeros_like(p); mean_ent=ent.clone()
            else:
                delta=p-mean; mean+=delta/(k+1); m2+=delta*(p-mean); mean_ent+=(ent-mean_ent)/(k+1)
        entropy=-(mean*mean.log()+(1-mean)*(1-mean).log())
        return mean, m2/passes, entropy, (entropy-mean_ent).clamp_min(0),dropout_count
    finally:
        for m,training in states.items(): m.training=training

def visual_report(net,dataset,thresholds,folder,prefix='test',n=3):
    folder=Path(folder); folder.mkdir(exist_ok=True,parents=True)
    # Fixed evenly spaced dataset indices, never select only high-Dice cases.
    indices=np.linspace(0,len(dataset)-1,min(n,len(dataset))).round().astype(int)
    for idx in indices:
        sample=dataset[int(idx)]; x=prepare_images(sample['image'].unsqueeze(0))
        mean,var,ent,mi,n_drop=segmentation_mc(net,x,CONFIG['mc_dropout_passes'])
        pred=mean[0].cpu().numpy()>=np.asarray(thresholds)[:,None,None]
        target=sample['regions'].numpy()>.5
        fig,axes=plt.subplots(3,7,figsize=(21,9))
        for r,region in enumerate(REGION_NAMES):
            cam=segmentation_gradcam(net,x.detach().clone(),r)
            maps=[sample['image'][3],target[r],pred[r],pred[r]!=target[r],cam,ent[0,r].cpu(),mi[0,r].cpu()]
            titles=['FLAIR','Ground truth','MC prediction','Error','Grad-CAM','Predictive entropy','Mutual information']
            for j,(m,title) in enumerate(zip(maps,titles)):
                if j==4: axes[r,j].imshow(sample['image'][3],cmap='gray')
                axes[r,j].imshow(m,cmap='gray' if j<3 else 'magma',alpha=.5 if j==4 else 1)
                axes[r,j].set_title(f'{region}: {title}',fontsize=9); axes[r,j].axis('off')
        fig.suptitle(f'{net.MODEL_NAME} | {sample["patient_id"]} / slice {sample["slice_idx"]} | MC dropout modules={n_drop}')
        fig.tight_layout(); fig.savefig(folder/f'{prefix}_xai_{idx:05d}.png',dpi=160); plt.show(); plt.close(fig)
        np.savez_compressed(folder/f'{prefix}_uncertainty_{idx:05d}.npz',mean=mean.cpu().numpy(),variance=var.cpu().numpy(),entropy=ent.cpu().numpy(),mutual_information=mi.cpu().numpy())

# ---- Reference folder of the frozen PH-CVT seed-42 model (new) ---------------------------
# The original notebook re-evaluated seed 42 here on every run. Its results are already on
# Drive, so they are simply reused; they are regenerated (inference only) only if missing.
proposed_study_dir = STUDY_DIR / 'PHCVT_proposed_seed42'
proposed_study_dir.mkdir(exist_ok=True)
if not all((proposed_study_dir / f).exists() for f in ['test_patients.csv', 'test_summary.csv', 'efficiency.json']):
    print('Seed-42 evaluation files missing: regenerating them from the frozen checkpoint (no training).')
    _ck = torch.load(BEST_MODEL_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(_ck['model_state'], strict=True); model.to(DEVICE).eval(); del _ck
    assess_segmentation(model, test_loader, seg_thresholds, proposed_study_dir)
    atomic_json(measure_efficiency(model), proposed_study_dir / 'efficiency.json')
    model.to('cpu'); gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('PH-CVT seed-42 reference results:', proposed_study_dir)

## Part 3 — Training the missing seeds (sessions A and B)
Each run trains into `extended_studies/<model>_seed<seed>/` and, when finished, is calibrated on validation data, evaluated on the test slices, profiled and visualised exactly like the earlier seeds. In **FINAL** this cell only loads completed runs (seconds) and never trains.

In [ ]:
# =============================================================================
# TRAINING / RESUME LOOP (original loop; additions are marked NEW)
# =============================================================================
# * A/B sessions: trains only the runs listed in TRAIN_PLANS[SESSION].
# * FINAL session: never trains. Completed runs are loaded from Drive in seconds; any planned
#   run that is not complete is listed in MISSING_RUNS and skipped.
# * Checkpoints: last_checkpoint.pt every epoch, best_model.pt on improvement,
#   partial_checkpoint.pt every 5 min, interrupted_checkpoint.pt on interruption.

# NEW — optional compute-budget guard (see MAX_TRAINING_HOURS in the session cell).
CURRENT_RUN = {'folder': None}
_session_start = time.monotonic()
def _budget_watchdog():
    while not IDLE['closed']:
        if time.monotonic() - _session_start > MAX_TRAINING_HOURS * 3600 and CURRENT_RUN['folder']:
            # fit_experiment() checks this file at every optimiser step and stops cleanly.
            Path(CURRENT_RUN['folder'], 'STOP_REQUESTED').write_text('MAX_TRAINING_HOURS reached')
            print('\nCompute budget reached: stopping after the current optimiser step.', flush=True)
            return
        time.sleep(30)
if ALLOW_TRAINING and MAX_TRAINING_HOURS:
    threading.Thread(target=_budget_watchdog, daemon=True).start()

# Execute comparisons sequentially; resume each run independently.
# Reduce STUDY['models'] to a subset if you want to split this work across Colab sessions.
# All seeds use the SAME fixed patient split (independent training seeds, not cross-validation).
def study_worker_seed(worker_id):
    worker_seed=torch.initial_seed()%2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

def fresh_study_train_loader(seed):
    sampler_generator=torch.Generator().manual_seed(seed)
    worker_generator=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(torch.as_tensor(sample_weights,dtype=torch.double),len(sample_weights),replacement=True,generator=sampler_generator)
    return DataLoader(train_dataset,sampler=sampler,generator=worker_generator,
                      worker_init_fn=study_worker_seed,**loader_kwargs)

STUDY_RESULTS=[{'model':'PHCVT_proposed','seed':42,'folder':str(proposed_study_dir),
                'checkpoint':str(BEST_MODEL_PATH),'thresholds':list(seg_thresholds)}]
def seeds_for(name):
    return STUDY['seed_plan'].get(name, STUDY['default_seeds'])

if STUDY['run_comparisons']:
    for name in STUDY['models']:
        for seed in seeds_for(name):
            folder=STUDY_DIR/f'{name}_seed{seed}'
            # Fast-resume: a model/seed already fully calibrated+tested+benchmarked on a
            # prior run needs no GPU time -- just reload its recorded result. Makes
            # re-running this cell after a disconnect cheap instead of restarting
            # completed models from scratch every time.
            done = (folder/'test_summary.csv').exists() and (folder/'efficiency.json').exists() \
                   and (folder/'calibrated_thresholds.json').exists() and (folder/'best_model.pt').exists()
            if done:
                th_map=json.loads((folder/'calibrated_thresholds.json').read_text())['segmentation']
                STUDY_RESULTS.append({'model':name,'seed':seed,'folder':str(folder),
                                      'checkpoint':str(folder/'best_model.pt'),
                                      'thresholds':[th_map[r] for r in REGION_NAMES]})
                print(f'[skip] {name} seed{seed} already evaluated; loaded from disk')
                continue
            if not ALLOW_TRAINING:                     # NEW: FINAL (or a finished plan) never trains
                MISSING_RUNS.append((name, seed)); print(f'[missing] {name} seed{seed} is not complete; skipped'); continue
            seed_everything(seed)
            train_loader=fresh_study_train_loader(seed)
            net=make_model(name).to(DEVICE)
            if CONFIG['use_channels_last'] and DEVICE.type=='cuda': net.to(memory_format=torch.channels_last)
            folder.mkdir(exist_ok=True)
            # Smoke check before committing GPU time; failures remain visible in the run status.
            net.eval()
            with torch.no_grad():
                check=net(torch.zeros(1,4,160,160,device=DEVICE))['seg_logits']
                assert check.shape==(1,3,160,160) and torch.isfinite(check).all()
            del check
            params=sum(p.numel() for p in net.parameters())
            print(name, f'seed{seed}:', f'{params/1e6:.2f} M parameters; up to {params*4*100/2**30:.1f} GiB for 100 epoch weight files')
            provenance={'model':name,'seed':seed,'initialization':'random; same-run checkpoint resume only',
                        'loss_weights':list(net.loss_weights),'split_sha256':file_hash(SPLIT_PATH),
                        'source':getattr(net,'provenance','notebook implementation'),
                        'max_epochs':100,'early_stopping_patience':15}
            atomic_json(provenance,folder/'protocol.json')
            (folder/'STOP_REQUESTED').unlink(missing_ok=True); CURRENT_RUN['folder']=str(folder)   # NEW: clear an old budget stop
            hist,score=fit_experiment(net,folder,name,seed)
            fig,ax=plt.subplots(1,2,figsize=(11,4))
            for col in ['train_loss','val_loss']: ax[0].plot(hist.epoch,hist[col],label=col)
            for col in ['train_mean_dice','val_mean_dice']: ax[1].plot(hist.epoch,hist[col],label=col)
            for a in ax: a.legend(); a.set_xlabel('Epoch'); a.grid(alpha=.2)
            fig.suptitle(f'{name} (seed {seed})'); fig.tight_layout(); fig.savefig(folder/'training_curves.png',dpi=180); plt.close(fig)
            th,_=calibrate_segmentation_thresholds(net,val_loader,candidates)
            atomic_json({'segmentation':dict(zip(REGION_NAMES,th)), 'source':'validation only'},folder/'calibrated_thresholds.json')
            assess_segmentation(net,test_loader,th,folder)
            atomic_json(measure_efficiency(net),folder/'efficiency.json')
            visual_report(net,test_dataset,th,folder,n=1)
            STUDY_RESULTS.append({'model':name,'seed':seed,'folder':str(folder),
                                  'checkpoint':str(folder/'best_model.pt'),'thresholds':th})
            atomic_json(STUDY_RESULTS,STUDY_DIR/f'completed_models_session{SESSION}.json')
            net.to('cpu'); del net; gc.collect()
            if DEVICE.type=='cuda': torch.cuda.empty_cache()

# NEW — session summary
if MISSING_RUNS:
    print('\nPlanned runs NOT yet complete (skipped, not trained in this session):')
    for _m, _s in MISSING_RUNS: print(f'  - {_m} seed {_s}')
if ALLOW_TRAINING:
    atomic_json({'session': SESSION, 'finished_unix_time': time.time(),
                 'runs': [(r['model'], r['seed']) for r in STUDY_RESULTS]},
                STUDY_DIR / f'session_{SESSION}_complete.json')
    print(f'\nSession {SESSION}: all planned runs are complete.')

## Part 4 — FINAL session: aggregation, full-volume evaluation, uncertainty, statistics, figures
All cells below do nothing in sessions A and B.

In [ ]:
# =============================================================================
# Selected-slice aggregation across ALL models and seeds (original code, FINAL only)
# =============================================================================
# Writes comparison_summary.csv, all_model_patient_metrics.csv, paired_patient_bootstrap.csv and
# comparison_dashboard.png. Run only once every parallel session has finished.
if SESSION == 'FINAL':
    comparison_rows=[]; patient_frames=[]
    for item in STUDY_RESULTS:
        folder=Path(item['folder']); p=pd.read_csv(folder/'test_patients.csv')
        s=pd.read_csv(folder/'test_summary.csv'); eff=json.loads((folder/'efficiency.json').read_text())
        values=p.groupby('patient_id').dice.mean(); mean,lo,hi=bootstrap_mean(values)
        row={'model':item['model'],'seed':item['seed'],'patient_mean_dice':mean,'ci95_low':lo,'ci95_high':hi,
             'micro_mean_dice':float(s.micro_dice.mean()),**eff}
        row.update({f'{r}_patient_dice':float(s.loc[s.region==r,'patient_mean_dice'].iloc[0]) for r in REGION_NAMES})
        comparison_rows.append(row); patient_frames.append(p.assign(model=item['model'],seed=item['seed']))
    comparison=pd.DataFrame(comparison_rows); comparison.to_csv(STUDY_DIR/'comparison_summary.csv',index=False)
    all_patients=pd.concat(patient_frames,ignore_index=True); all_patients.to_csv(STUDY_DIR/'all_model_patient_metrics.csv',index=False)
    print(comparison.round(4).to_string(index=False))

    # Paired patient bootstrap: CI is sampling uncertainty conditional on the trained models,
    # not variability across training seeds. Report every baseline, not just favorable ones.
    paired=[]
    for seed in sorted(set(r['seed'] for r in STUDY_RESULTS)):
        data=all_patients[all_patients.seed==seed].groupby(['patient_id','model']).dice.mean().unstack('model')
        for reference in ['PHCVT_proposed','PHCVT_segmentation_only']:
            if reference not in data: continue
            for other in data.columns:
                if other==reference: continue
                pair=data[[reference,other]].dropna()
                if len(pair)!=len(test_ids): raise ValueError('Paired study has missing patients')
                mean,lo,hi=bootstrap_mean(pair[reference]-pair[other])
                paired.append({'reference':reference,'comparator':other,'seed':seed,'n_patients':len(pair),
                               'mean_dice_difference':mean,'ci95_low':lo,'ci95_high':hi})
    pd.DataFrame(paired).to_csv(STUDY_DIR/'paired_patient_bootstrap.csv',index=False)

    fig,axes=plt.subplots(2,2,figsize=(15,11))
    labels=[f'{r.model} (s{r.seed})' for r in comparison.itertuples()]; xx=np.arange(len(labels))
    axes[0,0].bar(xx,comparison.patient_mean_dice)
    axes[0,0].errorbar(xx,comparison.patient_mean_dice,
        yerr=[np.maximum(comparison.patient_mean_dice-comparison.ci95_low,0),np.maximum(comparison.ci95_high-comparison.patient_mean_dice,0)],fmt='none',ecolor='black')
    axes[0,0].set_title('Patient mean Dice with patient bootstrap CI')
    axes[0,1].bar(xx,comparison.parameters/1e6); axes[0,1].set_title('Parameters (millions)')
    for ax in axes[0]: ax.set_xticks(xx,labels,rotation=75,ha='right'); ax.grid(axis='y',alpha=.2)
    axes[1,0].scatter(comparison.latency_median_ms,comparison.patient_mean_dice)
    for i,r in comparison.iterrows(): axes[1,0].annotate(r.model,(r.latency_median_ms,r.patient_mean_dice),fontsize=7)
    axes[1,0].set(xlabel='Median inference ms / slice',ylabel='Patient mean Dice',title='Accuracy and inference time')
    heat=comparison[[f'{r}_patient_dice' for r in REGION_NAMES]].to_numpy()
    im=axes[1,1].imshow(heat,vmin=0,vmax=1,aspect='auto',cmap='viridis')
    axes[1,1].set_yticks(xx,labels,fontsize=7); axes[1,1].set_xticks([0,1,2],REGION_NAMES)
    for i in range(len(labels)):
        for j in range(3): axes[1,1].text(j,i,f'{heat[i,j]:.3f}',ha='center',va='center',fontsize=7,color='white')
    fig.colorbar(im,ax=axes[1,1]); fig.tight_layout(); fig.savefig(STUDY_DIR/'comparison_dashboard.png',dpi=180); plt.show(); plt.close(fig)
else:
    print('Aggregation runs in the FINAL session only.')

In [ ]:
# =============================================================================
# NEW — FULL-VOLUME 3D EVALUATION + DEEP-ENSEMBLE / MC-DROPOUT UNCERTAINTY  (FINAL only)
# =============================================================================
# For every held-out test patient (188) and every completed checkpoint of each family:
#   1. load raw NIfTI and apply EXACTLY the training preprocessing: per-volume robust z-score
#      (robust_normalize from the cache cell) -> all axial slices -> bilinear resize to
#      160x160 -> float16 rounding (the training cache stores float16);
#   2. predict every slice (not only the 16+4 selected slices used during training);
#   3. resize probabilities back to the native 240x240 grid and score against the ORIGINAL
#      3D label volume: per-patient 3D Dice and HD95 (mm) for WT / TC / ET;
#   4. deep ensemble per family = mean of its seed probabilities; MC dropout (12 passes) for the
#      frozen PH-CVT seed-42 model;
#   5. patient-level uncertainty scores and voxel-level calibration histograms (brain voxels).
# Conventions (state them in the paper):
#   * Dice = 1 when prediction and label are both empty.
#   * HD95 = max of the two directed 95th-percentile surface distances (mm);
#     0 when both are empty, 373.13 mm when exactly one is empty (BraTS convention).
#   * Each single model uses its own validation-calibrated thresholds; an ensemble uses the
#     mean of its members' thresholds; MC dropout uses the seed-42 thresholds.
# Resumable: the state is written atomically after every patient.
import pickle, subprocess
from scipy import ndimage

FV_DIR = STUDY_DIR / 'full_volume'; FV_DIR.mkdir(parents=True, exist_ok=True)
FV_STATE = FV_DIR / 'fullvolume_state.pkl'
RAW_LOCAL = LOCAL_ROOT / 'raw_test'
PAPER_DIR = STUDY_DIR / 'paper_outputs'
PAPER_FIG = PAPER_DIR / 'figures'; PAPER_TAB = PAPER_DIR / 'tables'
for _p in (PAPER_FIG, PAPER_TAB): _p.mkdir(parents=True, exist_ok=True)
HD95_EMPTY_PENALTY_MM = 373.13
FV_BATCH = 32            # slices per forward pass (comfortable on a 16 GB T4)
CALIB_BINS = 15          # same number of reliability bins as the original assessment

# ----------------------------------------------------------------------------- helpers
def binary_entropy(p):
    p = p.clamp(1e-7, 1 - 1e-7)
    return -(p * p.log() + (1 - p) * (1 - p).log())

def _surface(mask):
    eroded = ndimage.binary_erosion(mask, structure=ndimage.generate_binary_structure(3, 1), border_value=0)
    return mask & ~eroded

def hd95_mm(pred, gt, spacing):
    """Symmetric HD95 in mm on a crop around both masks (identical result, ~50x faster)."""
    p_any, g_any = bool(pred.any()), bool(gt.any())
    if not p_any and not g_any: return 0.0
    if p_any != g_any: return HD95_EMPTY_PENALTY_MM
    idx = np.argwhere(pred | gt)
    lo = np.maximum(idx.min(0) - 2, 0); hi = np.minimum(idx.max(0) + 3, pred.shape)
    crop = tuple(slice(a, b) for a, b in zip(lo, hi))
    sp, sg = _surface(pred[crop]), _surface(gt[crop])
    d_to_g = ndimage.distance_transform_edt(~sg, sampling=spacing)[sp]
    d_to_p = ndimage.distance_transform_edt(~sp, sampling=spacing)[sg]
    return float(max(np.percentile(d_to_g, 95), np.percentile(d_to_p, 95)))

def member_list(family):
    """Completed checkpoints of one family: [(seed, checkpoint_path, thresholds)]."""
    out = []
    if family == 'PHCVT_proposed':
        out.append((42, BEST_MODEL_PATH, list(seg_thresholds)))
    for folder in sorted(STUDY_DIR.glob(f'{family}_seed*')):
        tail = folder.name[len(family) + len('_seed'):]
        if not tail.isdigit(): continue
        seed = int(tail)
        if family == 'PHCVT_proposed' and seed == 42: continue      # evaluation-only folder
        if not all((folder / f).exists() for f in ['best_model.pt', 'calibrated_thresholds.json', 'test_summary.csv']):
            print(f'  [incomplete run skipped] {folder.name}'); continue
        th = json.loads((folder / 'calibrated_thresholds.json').read_text())['segmentation']
        out.append((seed, folder / 'best_model.pt', [float(th[r]) for r in REGION_NAMES]))
    if ANALYSIS_SEEDS:                                    # like-for-like seed set for every model
        out = [t for t in out if t[0] in ANALYSIS_SEEDS]
    return sorted(out, key=lambda t: t[0])

def load_member(family, seed, ckpt):
    net = (ProbabilisticHybridCNNViTUNet(4, 3, CONFIG['image_size'])
           if (family == 'PHCVT_proposed' and seed == 42) else make_model(family))
    net.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False)['model_state'], strict=True)
    net = net.to(DEVICE).eval()
    if CONFIG['use_channels_last'] and DEVICE.type == 'cuda':
        net = net.to(memory_format=torch.channels_last)
    return net

def load_patient(pid):
    """Raw volumes -> training-identical 160x160 slices + native 3D targets."""
    d = RAW_LOCAL / pid
    raw_flair = nib.load(str(d / f'{pid}_flair.nii.gz')).get_fdata()
    vols = [robust_normalize(nib.load(str(d / f'{pid}_{m}.nii.gz')).get_fdata()) for m in ['t1', 't1ce', 't2']]
    vols.append(robust_normalize(raw_flair))
    seg_img = nib.load(str(d / f'{pid}_seg.nii.gz'))
    seg = np.asarray(seg_img.get_fdata()).astype(np.uint8); seg[seg == 4] = 3
    H, W, D = seg.shape
    x = torch.from_numpy(np.ascontiguousarray(np.stack(vols, 0).transpose(3, 0, 1, 2)))   # (D,4,H,W)
    x = F.interpolate(x, size=(CONFIG['image_size'],) * 2, mode='bilinear', align_corners=False)
    x = x.half().float()                                    # float16 rounding, as in the HDF5 cache
    gt = label_to_regions(seg).astype(bool)                 # (3,H,W,D) WT/TC/ET
    return {'pid': pid, 'x': x.to(DEVICE), 'gt_np': gt, 'gt': torch.from_numpy(gt).to(DEVICE),
            'brain': torch.from_numpy(raw_flair != 0).to(DEVICE), 'shape': (H, W, D),
            'spacing': tuple(float(z) for z in seg_img.header.get_zooms()[:3]),
            'flair': vols[3], 'seg': seg}

@torch.no_grad()
def predict_probs(net, x):
    out = []
    for i in range(0, x.shape[0], FV_BATCH):
        xb = x[i:i + FV_BATCH]
        if CONFIG['use_channels_last'] and DEVICE.type == 'cuda':
            xb = xb.contiguous(memory_format=torch.channels_last)
        with autocast_context():
            out.append(net(xb)['seg_logits'].float().sigmoid())
    return torch.cat(out)                                   # (D,3,160,160)

@torch.no_grad()
def predict_mc(net, x, passes):
    """MC dropout (same module selection as the original segmentation_mc)."""
    drop = [m for m in net.modules() if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)) and m.p > 0]
    net.eval()
    for m in drop: m.train()
    means, sq, ents = [], [], []
    try:
        for i in range(0, x.shape[0], FV_BATCH):
            xb = x[i:i + FV_BATCH]
            if CONFIG['use_channels_last'] and DEVICE.type == 'cuda':
                xb = xb.contiguous(memory_format=torch.channels_last)
            s1 = s2 = se = 0
            for _ in range(passes):
                with autocast_context():
                    p = net(xb)['seg_logits'].float().sigmoid()
                s1 = s1 + p; s2 = s2 + p * p; se = se + binary_entropy(p)
            means.append(s1 / passes); sq.append(s2 / passes); ents.append(se / passes)
    finally:
        net.eval()
    mean = torch.cat(means)
    return mean, (torch.cat(sq) - mean * mean).clamp_min(0), torch.cat(ents), len(drop)

def uncertainty_scores(p_mean, total_entropy, mutual_info=None, variance=None):
    """Patient scores = mean voxel uncertainty (averaged over WT/TC/ET) inside the tumour
    neighbourhood ROI (WT probability > 0.05). Averaging over the whole brain would be
    dominated by confident background."""
    roi = p_mean[:, 0] > 0.05
    n = int(roi.sum())
    def roi_mean(u):
        if u is None: return float('nan')
        return 0.0 if n == 0 else float(u.mean(1)[roi].mean())
    return {'unc_entropy': roi_mean(total_entropy), 'unc_mi': roi_mean(mutual_info),
            'unc_variance': roi_mean(variance), 'roi_voxels_160': n}

def score_prediction(p160, thresholds, case):
    """Native-resolution 3D metrics. Returns (row, native probabilities (3,H,W,D))."""
    H, W, D = case['shape']
    p = F.interpolate(p160, size=(H, W), mode='bilinear', align_corners=False).permute(1, 2, 3, 0)
    pred = p >= torch.tensor(thresholds, device=p.device, dtype=p.dtype).view(3, 1, 1, 1)
    gt = case['gt']
    inter = (pred & gt).sum((1, 2, 3)).double(); ps = pred.sum((1, 2, 3)).double(); gs = gt.sum((1, 2, 3)).double()
    dice = torch.where(ps + gs > 0, 2 * inter / (ps + gs).clamp_min(1), torch.ones_like(inter)).cpu().numpy()
    pred_np = pred.cpu().numpy()
    hd = [hd95_mm(pred_np[r], case['gt_np'][r], case['spacing']) for r in range(3)]
    ml = float(np.prod(case['spacing'])) / 1000.0
    row = {}
    for r, name in enumerate(REGION_NAMES):
        row[f'{name}_dice'] = float(dice[r]); row[f'{name}_hd95'] = hd[r]
        row[f'{name}_present'] = bool(gs[r] > 0)
        row[f'{name}_pred_ml'] = float(ps[r]) * ml; row[f'{name}_true_ml'] = float(gs[r]) * ml
    row['mean_dice'] = float(np.mean(dice)); row['mean_hd95'] = float(np.mean(hd))
    return row, p

def add_calibration(calib, key, p_native, case):
    """Streaming reliability histograms over brain voxels, split by true class."""
    acc = calib.setdefault(key, {'n': np.zeros((3, 2, CALIB_BINS)), 'conf': np.zeros((3, 2, CALIB_BINS))})
    brain = case['brain']
    for r in range(3):
        pr = p_native[r][brain]; t = case['gt'][r][brain]
        b = (pr * CALIB_BINS).long().clamp_(0, CALIB_BINS - 1)
        for cls in (0, 1):
            m = t == bool(cls)
            acc['n'][r, cls] += torch.bincount(b[m], minlength=CALIB_BINS).double().cpu().numpy()
            acc['conf'][r, cls] += torch.bincount(b[m], weights=pr[m].float(), minlength=CALIB_BINS).double().cpu().numpy()

def calibration_table(calib):
    """Same ECE definitions as the original assess_segmentation (full and class-balanced)."""
    rows = []
    for key, acc in calib.items():
        for r, region in enumerate(REGION_NAMES):
            for mode in ['full_brain', 'class_balanced']:
                w = np.ones(2) if mode == 'full_brain' else 0.5 / np.maximum(acc['n'][r].sum(1), 1)
                n = (acc['n'][r] * w[:, None]).sum(0); c = (acc['conf'][r] * w[:, None]).sum(0)
                t = (acc['n'][r, 1] * w[1])
                cp = np.divide(c, n, out=np.zeros_like(c), where=n > 0)
                tp = np.divide(t, n, out=np.zeros_like(t), where=n > 0)
                ece = float((np.abs(cp - tp) * n).sum() / max(n.sum(), 1e-12))
                for b in range(CALIB_BINS):
                    rows.append({'method': key, 'region': region, 'mode': mode, 'bin': b, 'weight': n[b],
                                 'confidence': cp[b], 'observed_frequency': tp[b], 'ece': ece})
    return pd.DataFrame(rows)

def save_state(state):
    tmp = FV_STATE.with_suffix('.tmp')
    with open(tmp, 'wb') as f: pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, FV_STATE)

def ensure_transformer_baseline_deps(families):
    if any(f in ('SwinUnet', 'TransUNet') for f in families):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.15', 'einops==0.8.1',
                        'ml-collections==1.0.0'], check=True)

# ----------------------------------------------------------------------------- main
if RUN_FULL_VOLUME:
    have = [pid for pid in test_ids if raw_patient_complete(pid)]
    if len(have) != len(test_ids):
        raise FileNotFoundError(f'Only {len(have)}/{len(test_ids)} raw test patients on Drive. '
                                'Run with EXTRACT_RAW_TEST = True first.')
    RAW_LOCAL.mkdir(parents=True, exist_ok=True)
    for pid in tqdm([p for p in test_ids if not raw_patient_complete(p, RAW_LOCAL)], desc='Copy raw test to local disk'):
        shutil.copytree(RAW_TEST_DRIVE / pid, RAW_LOCAL / pid, dirs_exist_ok=True)

    FV_FAMILIES = list(dict.fromkeys(FULLVOLUME_FAMILIES + FULLVOLUME_EXTRA_FAMILIES))
    ensure_transformer_baseline_deps(FV_FAMILIES)
    MEMBERS = {}
    for fam in FV_FAMILIES:
        MEMBERS[fam] = member_list(fam)
        print(f'{fam}: seeds {[s for s, _, _ in MEMBERS[fam]]}')
    MEMBERS = {f: m for f, m in MEMBERS.items() if m}
    signature = {'members': {f: [(s, str(c), os.path.getsize(c), th) for s, c, th in m] for f, m in MEMBERS.items()},
                 'mc': (RUN_MC_DROPOUT_FULLVOLUME, MC_PASSES_FULLVOLUME),
                 'hd95_penalty': HD95_EMPTY_PENALTY_MM, 'bins': CALIB_BINS}
    state = None
    if FV_STATE.exists():
        with open(FV_STATE, 'rb') as f: state = pickle.load(f)
        if state.get('signature') != signature:
            stale = FV_STATE.with_name(f'fullvolume_state_stale_{int(time.time())}.pkl')
            os.replace(FV_STATE, stale)
            print('Checkpoint set changed since the last run: old state archived to', stale.name)
            state = None
    if state is None:
        state = {'signature': signature, 'done': [], 'rows': [], 'calib': {}, 'seconds': []}
    print(f"Full-volume state: {len(state['done'])}/{len(test_ids)} patients already evaluated")

    NETS = {(f, s): (load_member(f, s, c), th) for f, m in MEMBERS.items() for s, c, th in m}
    todo = [pid for pid in test_ids if pid not in set(state['done'])]
    for pid in tqdm(todo, desc='Full-volume patients'):
        t0 = time.time(); case = load_patient(pid); rows = []
        for fam, members in MEMBERS.items():
            s1 = s2 = sent = 0; ths = []
            for seed, _, th in members:
                net = NETS[(fam, seed)][0]
                p = predict_probs(net, case['x']); ent = binary_entropy(p)
                row, p_nat = score_prediction(p, th, case)
                row.update(uncertainty_scores(p, ent))
                row.update(patient_id=pid, family=fam, member=str(seed), variant='single', n_members=1)
                rows.append(row)
                if fam == 'PHCVT_proposed' and seed == 42:
                    add_calibration(state['calib'], 'PH-CVT seed 42 (deterministic)', p_nat, case)
                del p_nat
                s1 = s1 + p; s2 = s2 + p * p; sent = sent + ent; ths.append(th)
                del p, ent
            k = len(members)
            if k >= 2:
                mean = s1 / k; var = (s2 / k - mean * mean).clamp_min(0)
                total = binary_entropy(mean); mi = (total - sent / k).clamp_min(0)
                row, p_nat = score_prediction(mean, list(np.mean(ths, axis=0)), case)
                row.update(uncertainty_scores(mean, total, mi, var))
                row.update(patient_id=pid, family=fam, member='ensemble', variant='ensemble', n_members=k)
                rows.append(row); add_calibration(state['calib'], f'{fam} {k}-seed ensemble', p_nat, case)
                del mean, var, total, mi, p_nat
            del s1, s2, sent
        if RUN_MC_DROPOUT_FULLVOLUME and ('PHCVT_proposed', 42) in NETS:
            mean, var, exp_ent, n_drop = predict_mc(NETS[('PHCVT_proposed', 42)][0], case['x'], MC_PASSES_FULLVOLUME)
            total = binary_entropy(mean); mi = (total - exp_ent).clamp_min(0)
            row, p_nat = score_prediction(mean, list(seg_thresholds), case)
            row.update(uncertainty_scores(mean, total, mi, var))
            row.update(patient_id=pid, family='PHCVT_proposed', member='mc_dropout', variant='mc_dropout',
                       n_members=MC_PASSES_FULLVOLUME)
            rows.append(row); add_calibration(state['calib'], 'PH-CVT seed 42 MC dropout', p_nat, case)
            del mean, var, exp_ent, total, mi, p_nat
        state['rows'].extend(rows); state['done'].append(pid); state['seconds'].append(time.time() - t0)
        save_state(state)
        del case
        if len(state['done']) % 10 == 0 and DEVICE.type == 'cuda': torch.cuda.empty_cache()

    FV = pd.DataFrame(state['rows'])
    FV.to_csv(FV_DIR / 'full_volume_patient_rows.csv', index=False)
    CAL = calibration_table(state['calib']); CAL.to_csv(FV_DIR / 'full_volume_calibration_bins.csv', index=False)
    print(f"Full-volume evaluation complete: {FV.patient_id.nunique()} patients, "
          f"median {np.median(state['seconds']):.1f} s/patient")
    quick = (FV.groupby(['family', 'variant'])[['WT_dice', 'TC_dice', 'ET_dice', 'mean_dice', 'mean_hd95']]
               .mean().round(4))
    print(quick.to_string())

    # ------------------------------------------------------- qualitative full-volume figure
    # Patients are chosen systematically at the 10th / 50th / 90th percentile of the PH-CVT
    # ensemble Dice (never hand-picked). Slice = axial slice with the largest true WT area.
    def label_rgba(wt, tc, et):
        rgba = np.zeros(wt.shape + (4,))
        rgba[wt & ~tc] = (0.20, 0.80, 0.20, 0.55)   # oedema
        rgba[tc & ~et] = (0.90, 0.20, 0.20, 0.65)   # necrotic / non-enhancing core
        rgba[et] = (1.00, 0.85, 0.10, 0.75)         # enhancing tumour
        return rgba
    ref_variant = 'ensemble' if (FV.query("family=='PHCVT_proposed'").variant == 'ensemble').any() else 'single'
    ref = FV[(FV.family == 'PHCVT_proposed') & (FV.variant == ref_variant)]
    if ref_variant == 'single': ref = ref[ref.member == '42']
    ref = ref.sort_values('mean_dice').reset_index(drop=True)
    picks = []
    for q in (0.10, 0.50, 0.90):
        pid = ref.iloc[int(round(q * (len(ref) - 1)))].patient_id
        if pid not in picks: picks.append(pid)
    cmp_family = next((f for f in ['UNet', 'ResUNet', 'AttentionUNet'] if f in MEMBERS), None)
    fig, axes = plt.subplots(len(picks), 7, figsize=(24, 3.6 * len(picks)), squeeze=False)
    for i, pid in enumerate(picks):
        case = load_patient(pid)
        d = int(case['gt_np'][0].sum((0, 1)).argmax())
        def family_mean(fam):
            ps = [predict_probs(NETS[(fam, s)][0], case['x'][d:d + 1]) for s, _, _ in MEMBERS[fam]]
            st = torch.stack(ps)                                            # (k,1,3,160,160)
            mean = st.mean(0); mi = (binary_entropy(mean) - binary_entropy(st).mean(0)).clamp_min(0)
            th = np.mean([t for _, _, t in MEMBERS[fam]], axis=0)
            return mean, mi, th
        ph_mean, ph_mi, ph_th = family_mean('PHCVT_proposed')
        up = lambda t: F.interpolate(t, size=case['shape'][:2], mode='bilinear', align_corners=False)[0].cpu().numpy()
        ph_pred = up(ph_mean) >= ph_th[:, None, None]
        maps = {'mi_ens': up(ph_mi).mean(0)}
        if RUN_MC_DROPOUT_FULLVOLUME and ('PHCVT_proposed', 42) in NETS:
            m_mean, _, m_ent, _ = predict_mc(NETS[('PHCVT_proposed', 42)][0], case['x'][d:d + 1], MC_PASSES_FULLVOLUME)
            maps['mi_mc'] = up((binary_entropy(m_mean) - m_ent).clamp_min(0)).mean(0)
        gt = case['gt_np'][:, :, :, d]
        flair = np.rot90(case['flair'][:, :, d]); rot = np.rot90
        panels = [('FLAIR', None), ('Ground truth', label_rgba(*gt)),
                  ('PH-CVT ensemble', label_rgba(*ph_pred))]
        err = np.zeros(gt[0].shape + (4,))
        err[ph_pred[0] & ~gt[0]] = (1, 0.1, 0.1, 0.8); err[~ph_pred[0] & gt[0]] = (0.1, 0.4, 1, 0.8)
        panels.append(('WT error (red FP, blue FN)', err))
        for j, (title, overlay) in enumerate(panels):
            ax = axes[i, j]; ax.imshow(flair, cmap='gray')
            if overlay is not None: ax.imshow(rot(overlay))
            ax.set_title(f'{title}' if j else f'{pid} | slice {d}', fontsize=9); ax.axis('off')
        im = axes[i, 4].imshow(rot(maps['mi_ens']), cmap='magma'); axes[i, 4].set_title('Ensemble MI (epistemic)', fontsize=9)
        fig.colorbar(im, ax=axes[i, 4], fraction=0.046)
        if 'mi_mc' in maps:
            im = axes[i, 5].imshow(rot(maps['mi_mc']), cmap='magma'); axes[i, 5].set_title('MC-dropout MI', fontsize=9)
            fig.colorbar(im, ax=axes[i, 5], fraction=0.046)
        if cmp_family:
            c_mean, _, c_th = family_mean(cmp_family)
            axes[i, 6].imshow(flair, cmap='gray'); axes[i, 6].imshow(rot(label_rgba(*(up(c_mean) >= c_th[:, None, None]))))
            axes[i, 6].set_title(f'{cmp_family} ensemble', fontsize=9)
        for j in (4, 5, 6): axes[i, j].axis('off')
        del case
    fig.suptitle('Full-volume qualitative results (patients at the 10th/50th/90th percentile of PH-CVT ensemble Dice)\n'
                 'green = oedema, red = necrotic core, yellow = enhancing tumour', fontsize=11)
    fig.tight_layout(); fig.savefig(PAPER_FIG / 'fig_qualitative_full_volume.png', dpi=220); plt.show(); plt.close(fig)

    for key in list(NETS): NETS[key][0].to('cpu')
    del NETS; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('Full-volume evaluation not requested in this session.')

In [ ]:
# =============================================================================
# NEW — Same-GPU efficiency profiling (FINAL only)
# =============================================================================
# Latencies stored with each run were measured on whatever GPU trained it (A100 for older
# seeds, T4 for new ones), so they are not comparable. Here ONE checkpoint per model is
# re-profiled in THIS runtime with the original measure_efficiency() (batch 1, AMP,
# 10 warm-up + 50 timed passes). Report only this table in the paper, naming the GPU.
if RUN_EFFICIENCY:
    eff_rows = []
    wanted = [m for m in ALL_MODELS]
    if any(m in ('SwinUnet', 'TransUNet') for m in wanted):
        try:
            ensure_transformer_baseline_deps(wanted)
        except Exception as exc:
            print('timm installation failed; transformer baselines will be skipped:', repr(exc))
    for fam in wanted:
        members = member_list(fam)
        if not members:
            print(f'  {fam}: no completed checkpoint, skipped'); continue
        seed, ckpt, _ = members[0]
        try:
            net = load_member(fam, seed, ckpt)
            eff = measure_efficiency(net)
            eff_rows.append({'model': fam, 'profiled_seed': seed, **eff})
            print(f"  {fam:28s} {eff['parameters']/1e6:7.2f} M  median {eff['latency_median_ms']:7.2f} ms  "
                  f"p95 {eff['latency_p95_ms']:7.2f} ms")
            net.to('cpu'); del net
        except Exception as exc:
            print(f'  {fam}: profiling failed ({exc!r}); skipped')
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    EFF = pd.DataFrame(eff_rows)
    EFF.to_csv(PAPER_TAB / 'table_efficiency_same_gpu.csv', index=False)
    print('Saved:', PAPER_TAB / 'table_efficiency_same_gpu.csv')
else:
    print('Same-GPU efficiency profiling not requested in this session.')

In [ ]:
# =============================================================================
# NEW — Statistics, uncertainty analysis, figures and paper tables (FINAL only)
# =============================================================================
# Reads only files on Drive, so it can be re-run on its own at any time.
# Primary unit of analysis = patient (n = 188). Seeds are averaged within patient before
# paired testing, and only seeds available for BOTH models of a comparison are used.
#   * Wilcoxon signed-rank on per-patient Dice differences, Holm correction across comparators
#   * Patient bootstrap 95% CI of the mean difference (2,000 resamples)
#   * Equivalence (TOST-style): 90% bootstrap CI entirely inside +/- EQUIVALENCE_MARGIN
#   * Uncertainty: Spearman(uncertainty, 1 - Dice), failure AUROC/AUPRC (Dice < FAILURE_DICE),
#     risk-coverage curves and AURC, each with patient-bootstrap CIs
from scipy import stats as sps
from sklearn.metrics import roc_auc_score, average_precision_score

N_BOOT = 2000
PRETTY = {'PHCVT_proposed': 'PH-CVT (proposed)', 'PHCVT_lite': 'PH-CVT-Lite', 'UNet': 'U-Net',
          'ResUNet': 'ResU-Net', 'AttentionUNet': 'Attention U-Net', 'SwinUnet': 'Swin-Unet',
          'TransUNet': 'TransUNet', 'PHCVT_segmentation_only': 'PH-CVT seg-only',
          'PHCVT_no_transformer': 'PH-CVT w/o Transformer', 'PHCVT_no_aspp': 'PH-CVT w/o ASPP',
          'PHCVT_no_attention_fusion': 'PH-CVT w/o attn. fusion', 'PHCVT_no_auxiliary': 'PH-CVT w/o aux. loss'}
ORDER = list(PRETTY)
REF = 'PHCVT_proposed'
pretty = lambda m: PRETTY.get(m, m)
ordered = lambda models: [m for m in ORDER if m in set(models)] + sorted(set(models) - set(ORDER))

# ----------------------------------------------------------------------------- helpers
def seed_table(df, cols):
    """Mean over patients per (model, seed), then mean / SD across seeds per model."""
    per_seed = df.groupby(['model', 'seed'])[cols].mean().reset_index()
    rows = []
    for m in ordered(per_seed.model.unique()):
        part = per_seed[per_seed.model == m]
        row = {'model': m, 'label': pretty(m), 'n_seeds': part.seed.nunique(),
               'seeds': ','.join(str(s) for s in sorted(part.seed.unique()))}
        for c in cols:
            row[f'{c}_mean'] = part[c].mean(); row[f'{c}_sd'] = part[c].std(ddof=1) if len(part) > 1 else np.nan
        rows.append(row)
    return pd.DataFrame(rows), per_seed

def bootstrap_ci(x, stat=np.mean, n=N_BOOT, seed=42, q=(2.5, 97.5)):
    x = np.asarray(x, float); rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n, len(x)))
    vals = np.array([stat(x[i]) for i in idx])
    return np.nanpercentile(vals, q)

def holm(pvals):
    p = np.asarray(pvals, float); adj = np.full_like(p, np.nan)
    ok = np.where(np.isfinite(p))[0]
    order = ok[np.argsort(p[ok])]; running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(order) - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

def paired_vs_reference(df, value='mean_dice', reference=REF, margin=EQUIVALENCE_MARGIN, protocol=''):
    """df columns: model, seed, patient_id, value. Returns one row per comparator."""
    if reference not in set(df.model): return pd.DataFrame()
    ref_seeds = set(df.loc[df.model == reference, 'seed'])
    rows = []
    for other in ordered(df.model.unique()):
        if other == reference: continue
        seeds = sorted(ref_seeds & set(df.loc[df.model == other, 'seed']))
        if not seeds: continue
        sub = df[df.seed.isin(seeds) & df.model.isin([reference, other])]
        wide = sub.groupby(['patient_id', 'model'])[value].mean().unstack('model').dropna()
        diff = (wide[reference] - wide[other]).to_numpy()
        lo95, hi95 = bootstrap_ci(diff); lo90, hi90 = bootstrap_ci(diff, q=(5, 95))
        try: p = sps.wilcoxon(diff).pvalue
        except ValueError: p = np.nan
        rows.append({'protocol': protocol, 'reference': pretty(reference), 'comparator': pretty(other),
                     'comparator_key': other, 'matched_seeds': ','.join(map(str, seeds)), 'n_patients': len(diff),
                     'mean_difference': diff.mean(), 'ci95_low': lo95, 'ci95_high': hi95,
                     'ci90_low': lo90, 'ci90_high': hi90, 'wilcoxon_p': p,
                     'equivalent_within_margin': bool(-margin < lo90 and hi90 < margin)})
    out = pd.DataFrame(rows)
    if len(out):
        out['wilcoxon_p_holm'] = holm(out.wilcoxon_p)
        def verdict(r):
            # Statistical significance and practical equivalence are reported separately:
            # a tiny but consistent difference can be both significant and within the margin.
            sig = ('reference significantly higher' if r.mean_difference > 0 else 'reference significantly lower') \
                  if r.wilcoxon_p_holm < 0.05 else 'no significant difference'
            eq = f'equivalent within ±{margin:g}' if r.equivalent_within_margin else 'equivalence not shown'
            return f'{sig}; {eq}'
        out['interpretation'] = out.apply(verdict, axis=1)
    return out

def risk_coverage(dice, score, coverages):
    order = np.argsort(score, kind='stable')                 # most confident patients first
    return np.array([dice[order[:max(1, int(round(c * len(dice))))]].mean() for c in coverages])

def aurc(dice, score):
    risk = (1 - dice)[np.argsort(score, kind='stable')]
    return float(np.mean(np.cumsum(risk) / np.arange(1, len(risk) + 1)))

def uncertainty_metrics(dice, score, failure_threshold):
    dice = np.asarray(dice, float); score = np.asarray(score, float)
    fail = (dice < failure_threshold).astype(int)
    rho = sps.spearmanr(score, 1 - dice).statistic
    rng = np.random.default_rng(42); boots = {'rho': [], 'auroc': [], 'auprc': [], 'aurc': []}
    for _ in range(N_BOOT):
        i = rng.integers(0, len(dice), len(dice))
        boots['rho'].append(sps.spearmanr(score[i], 1 - dice[i]).statistic)
        boots['aurc'].append(aurc(dice[i], score[i]))
        if 0 < fail[i].sum() < len(i):
            boots['auroc'].append(roc_auc_score(fail[i], score[i])); boots['auprc'].append(average_precision_score(fail[i], score[i]))
    two_classes = 0 < fail.sum() < len(fail)
    ci = lambda v: np.nanpercentile(v, [2.5, 97.5]) if len(v) else [np.nan, np.nan]
    res = {'n_patients': len(dice), 'n_failures': int(fail.sum()), 'mean_dice': dice.mean(),
           'spearman_rho': rho, 'rho_ci_low': ci(boots['rho'])[0], 'rho_ci_high': ci(boots['rho'])[1],
           'auroc': roc_auc_score(fail, score) if two_classes else np.nan,
           'auroc_ci_low': ci(boots['auroc'])[0], 'auroc_ci_high': ci(boots['auroc'])[1],
           'auprc': average_precision_score(fail, score) if two_classes else np.nan,
           'auprc_ci_low': ci(boots['auprc'])[0], 'auprc_ci_high': ci(boots['auprc'])[1],
           'failure_prevalence': fail.mean(), 'aurc': aurc(dice, score),
           'aurc_ci_low': ci(boots['aurc'])[0], 'aurc_ci_high': ci(boots['aurc'])[1],
           'aurc_oracle': aurc(dice, -dice)}
    return res

def savefig(fig, name):
    fig.tight_layout(); fig.savefig(PAPER_FIG / name, dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

def box(ax, data, labels, title, ylabel, logy=False):
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6)
    for patch, lab in zip(bp['boxes'], labels):
        patch.set_facecolor('#d95f02' if lab == pretty(REF) else '#7570b3'); patch.set_alpha(0.55)
    for k, d in enumerate(data):
        ax.scatter(np.random.default_rng(k).normal(k + 1, 0.05, len(d)), d, s=4, alpha=0.25, color='k')
    ax.set_xticks(range(1, len(labels) + 1), labels, rotation=35, ha='right'); ax.set_title(title); ax.set_ylabel(ylabel)
    if logy: ax.set_yscale('log')
    ax.grid(axis='y', alpha=0.25)

# ----------------------------------------------------------------------------- driver
if RUN_STATISTICS:
    SUMMARY_LINES = []
    # ---------------- 1. selected-slice protocol (original evaluation, all models) -----------
    sel_raw = pd.read_csv(STUDY_DIR / 'all_model_patient_metrics.csv')
    SEL = (sel_raw.pivot_table(index=['model', 'seed', 'patient_id'], columns='region', values='dice')
                  .rename(columns={r: f'{r}_dice' for r in REGION_NAMES}).reset_index())
    SEL['mean_dice'] = SEL[[f'{r}_dice' for r in REGION_NAMES]].mean(axis=1)
    if ANALYSIS_SEEDS: SEL = SEL[SEL.seed.isin(ANALYSIS_SEEDS)]          # like-for-like seeds
    dice_cols = [f'{r}_dice' for r in REGION_NAMES] + ['mean_dice']
    T_SEL, SEL_SEED = seed_table(SEL, dice_cols)
    T_SEL.to_csv(PAPER_TAB / 'table1_selected_slice_seed_summary.csv', index=False)
    P_SEL = paired_vs_reference(SEL, protocol='selected-slice')
    P_SEL.to_csv(PAPER_TAB / 'table3a_paired_selected_slice.csv', index=False)
    print('Selected-slice protocol (mean ± SD across seeds of patient-mean Dice):')
    print(T_SEL[['label', 'n_seeds', 'mean_dice_mean', 'mean_dice_sd', 'WT_dice_mean', 'TC_dice_mean', 'ET_dice_mean']].round(4).to_string(index=False))

    # ---------------- 2. full-volume protocol -------------------------------------------------
    fv_path = STUDY_DIR / 'full_volume' / 'full_volume_patient_rows.csv'
    FVR = pd.read_csv(fv_path, dtype={'member': str}) if fv_path.exists() else None
    if FVR is not None:
        single = FVR[FVR.variant == 'single'].copy()
        single['model'] = single.family; single['seed'] = single.member.astype(int)
        if ANALYSIS_SEEDS: single = single[single.seed.isin(ANALYSIS_SEEDS)]
        hd_cols = [f'{r}_hd95' for r in REGION_NAMES] + ['mean_hd95']
        T_FV, FV_SEED = seed_table(single, dice_cols + hd_cols)
        med = (single.groupby(['model', 'patient_id'])[hd_cols].mean().groupby('model').median()
                     .add_suffix('_patient_median'))
        T_FV = T_FV.merge(med, left_on='model', right_index=True, how='left')
        T_FV.to_csv(PAPER_TAB / 'table2_full_volume_seed_summary.csv', index=False)
        P_FV = paired_vs_reference(single, protocol='full-volume 3D')
        P_FV_HD = paired_vs_reference(single, value='mean_hd95', protocol='full-volume 3D HD95', margin=1.0)
        ens = FVR[FVR.variant == 'ensemble'].copy(); ens['model'] = ens.family; ens['seed'] = 0
        P_ENS = paired_vs_reference(ens, protocol='full-volume seed-ensemble')
        if len(P_ENS): P_ENS['matched_seeds'] = 'all completed seeds (ensemble)'
        pd.concat([P_FV, P_FV_HD, P_ENS]).to_csv(PAPER_TAB / 'table3b_paired_full_volume.csv', index=False)
        ens_rows = []
        for fam in ordered(ens.model.unique()):
            part = ens[ens.model == fam]
            row = {'model': fam, 'label': pretty(fam), 'n_members': int(part.n_members.iloc[0])}
            for c in dice_cols + hd_cols:
                row[c] = part[c].mean(); row[f'{c}_ci_low'], row[f'{c}_ci_high'] = bootstrap_ci(part[c])
            ens_rows.append(row)
        T_ENS = pd.DataFrame(ens_rows); T_ENS.to_csv(PAPER_TAB / 'table2b_full_volume_ensembles.csv', index=False)
        print('\nFull-volume 3D protocol (single models, mean ± SD across seeds):')
        print(T_FV[['label', 'n_seeds', 'mean_dice_mean', 'mean_dice_sd', 'WT_dice_mean', 'TC_dice_mean',
                    'ET_dice_mean', 'mean_hd95_mean']].round(4).to_string(index=False))
        if len(T_ENS):
            print('\nFull-volume seed ensembles:')
            print(T_ENS[['label', 'n_members', 'mean_dice', 'mean_dice_ci_low', 'mean_dice_ci_high', 'mean_hd95']].round(4).to_string(index=False))
        print('\nPaired comparisons vs PH-CVT (reference - comparator; positive favours PH-CVT):')
        show = ['protocol', 'comparator', 'matched_seeds', 'mean_difference', 'ci95_low', 'ci95_high', 'wilcoxon_p_holm', 'interpretation']
        print(pd.concat([P_SEL, P_FV, P_ENS])[show].round(4).to_string(index=False))
    else:
        print('\nFull-volume results not found; only selected-slice statistics were produced.')
        print(P_SEL[['comparator', 'mean_difference', 'ci95_low', 'ci95_high', 'wilcoxon_p_holm', 'interpretation']].round(4).to_string(index=False))

    # ---------------- 3. figures: accuracy -----------------------------------------------------
    main_fv = FV_SEED if FVR is not None else None
    if FVR is not None:
        models = ordered(single.model.unique()); labels = [pretty(m) for m in models]
        pat = single.groupby(['model', 'patient_id'])[dice_cols + hd_cols].mean()
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, c, t in zip(axes, dice_cols, ['WT', 'TC', 'ET', 'Mean of WT/TC/ET']):
            box(ax, [pat.loc[m][c].to_numpy() for m in models], labels, f'{t} — full-volume 3D Dice', 'Patient Dice (seed-averaged)')
        savefig(fig, 'fig1_full_volume_dice_boxplots.png')
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        for ax, c, t in zip(axes, hd_cols, REGION_NAMES):
            box(ax, [pat.loc[m][c].to_numpy() for m in models], labels, f'{t} — HD95', 'HD95 (mm, seed-averaged)', logy=True)
        savefig(fig, 'fig2_full_volume_hd95_boxplots.png')

    forest = pd.concat([P_SEL] + ([P_FV, P_ENS] if FVR is not None else []))
    if len(forest):
        protos = list(dict.fromkeys(forest.protocol)); comps = list(dict.fromkeys(forest.comparator))
        fig, ax = plt.subplots(figsize=(9, 0.32 * len(forest) + 0.25 * len(comps) + 1.5))
        colors = ['#1b9e77', '#d95f02', '#7570b3']; y = 0; yt, yl = [], []; labelled = set()
        for c in comps:
            for k, pr in enumerate(protos):
                r = forest[(forest.comparator == c) & (forest.protocol == pr)]
                if not len(r): continue
                r = r.iloc[0]
                ax.errorbar(r.mean_difference, y, xerr=[[r.mean_difference - r.ci95_low], [r.ci95_high - r.mean_difference]],
                            fmt='o', color=colors[k % 3], capsize=3, label=None if pr in labelled else pr)
                labelled.add(pr)
                yt.append(y); yl.append(f'{c}'); y += 1
            y += 0.6
        ax.axvline(0, color='k', lw=1); ax.axvspan(-EQUIVALENCE_MARGIN, EQUIVALENCE_MARGIN, color='grey', alpha=0.15,
                                                    label=f'equivalence margin ±{EQUIVALENCE_MARGIN}')
        ax.set_yticks(yt, yl, fontsize=8); ax.invert_yaxis()
        ax.set_xlabel('Mean patient Dice difference, PH-CVT − comparator (95% bootstrap CI)')
        ax.legend(fontsize=8, loc='lower right'); ax.grid(axis='x', alpha=0.25)
        savefig(fig, 'fig3_paired_differences_forest.png')

    fig, axes = plt.subplots(1, 2 if FVR is not None else 1, figsize=(16 if FVR is not None else 8, 5), squeeze=False)
    for ax, (seed_df, title) in zip(axes[0], [(SEL_SEED, 'Selected-slice protocol'), (main_fv, 'Full-volume 3D protocol')]):
        if seed_df is None: continue
        ms = ordered(seed_df.model.unique())
        for k, m in enumerate(ms):
            v = seed_df[seed_df.model == m].mean_dice.to_numpy()
            ax.scatter(np.full(len(v), k), v, color='#d95f02' if m == REF else '#7570b3', s=30, zorder=3)
            ax.hlines(v.mean(), k - 0.3, k + 0.3, color='k')
        ax.set_xticks(range(len(ms)), [pretty(m) for m in ms], rotation=35, ha='right')
        ax.set_ylabel('Mean patient Dice per training seed'); ax.set_title(f'{title}: seed-to-seed variability'); ax.grid(axis='y', alpha=0.25)
    savefig(fig, 'fig4_seed_variability.png')

    if FVR is not None:
        both = T_SEL[['model', 'mean_dice_mean']].merge(T_FV[['model', 'mean_dice_mean']], on='model', suffixes=('_sel', '_fv'))
        fig, ax = plt.subplots(figsize=(10, 4.5)); xx = np.arange(len(both))
        ax.bar(xx - 0.2, both.mean_dice_mean_sel, 0.4, label='Selected slices (16 tumour + 4 near-tumour per patient)')
        ax.bar(xx + 0.2, both.mean_dice_mean_fv, 0.4, label='Full 3D volume (all 155 slices, native grid)')
        ax.set_xticks(xx, [pretty(m) for m in both.model], rotation=30, ha='right'); ax.set_ylim(0.6, 1.0)
        ax.set_ylabel('Mean patient Dice'); ax.set_title('Effect of the evaluation protocol'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.25)
        savefig(fig, 'fig5_protocol_comparison.png')

    eff_path = PAPER_TAB / 'table_efficiency_same_gpu.csv'
    if eff_path.exists():
        EFFT = pd.read_csv(eff_path)
        acc = (T_FV if FVR is not None else T_SEL)[['model', 'mean_dice_mean']]
        e = EFFT.merge(acc, on='model')
        if len(e):
            fig, ax = plt.subplots(figsize=(8, 5.5))
            ax.scatter(e.latency_median_ms, e.mean_dice_mean, s=e.parameters / 2e5, alpha=0.6,
                       c=['#d95f02' if m == REF else '#7570b3' for m in e.model])
            for r in e.itertuples():
                ax.annotate(f'{pretty(r.model)}\n{r.parameters/1e6:.1f} M', (r.latency_median_ms, r.mean_dice_mean), fontsize=7)
            ax.set_xlabel(f'Median latency per 160×160 slice (ms, batch 1, {e.device.iloc[0]})')
            ax.set_ylabel('Mean patient Dice (' + ('full volume' if FVR is not None else 'selected slices') + ')')
            ax.set_title('Accuracy vs inference cost (bubble area ∝ parameters)'); ax.grid(alpha=0.25)
            savefig(fig, 'fig6_accuracy_vs_efficiency.png')

    hist_files = [(REF, 42, OUTPUT_DIR / 'training_history.csv')] + [
        (p.parent.name.rsplit('_seed', 1)[0], int(p.parent.name.rsplit('_seed', 1)[1]), p)
        for p in STUDY_DIR.glob('*_seed*/training_history.csv') if p.parent.name.rsplit('_seed', 1)[1].isdigit()]
    if ANALYSIS_SEEDS: hist_files = [h for h in hist_files if h[1] in ANALYSIS_SEEDS]
    fams = ordered({f for f, _, _ in hist_files})
    if fams:
        ncol = 4; nrow = int(np.ceil(len(fams) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.4 * nrow), squeeze=False)
        for ax, fam in zip(axes.flat, fams):
            for f, s, p in sorted(hist_files, key=lambda t: t[1]):
                if f != fam: continue
                h = pd.read_csv(p); ax.plot(h.epoch, h.val_mean_dice, label=f'seed {s}', lw=1.2)
            ax.set_title(pretty(fam)); ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Dice (selected slices)')
            ax.legend(fontsize=7); ax.grid(alpha=0.25)
        for ax in list(axes.flat)[len(fams):]: ax.axis('off')
        savefig(fig, 'figS1_training_curves_all_seeds.png')

    # ---------------- 4. uncertainty -------------------------------------------------------------
    if FVR is not None:
        METHODS = []
        def add(name, frame, col, transform=None):
            frame = frame.dropna(subset=['mean_dice'])
            if not len(frame): return
            sc = frame[col].to_numpy(float) if transform is None else transform(frame)
            if np.all(np.isnan(sc)): return
            METHODS.append((name, frame.mean_dice.to_numpy(float), sc))
        ph = FVR[FVR.family == REF]
        add('PH-CVT deterministic: predictive entropy', ph[(ph.variant == 'single') & (ph.member == '42')], 'unc_entropy')
        add('PH-CVT MC dropout: mutual information', ph[ph.variant == 'mc_dropout'], 'unc_mi')
        add('PH-CVT MC dropout: predictive entropy', ph[ph.variant == 'mc_dropout'], 'unc_entropy')
        add('PH-CVT seed ensemble: mutual information', ph[ph.variant == 'ensemble'], 'unc_mi')
        add('PH-CVT seed ensemble: predictive entropy', ph[ph.variant == 'ensemble'], 'unc_entropy')
        add('Reference score: small predicted tumour (−log WT volume)', ph[ph.variant == 'ensemble'], None,
            transform=lambda fr: -np.log1p(fr.WT_pred_ml.to_numpy(float)))
        for fam in ordered(FVR.family.unique()):
            if fam == REF: continue
            add(f'{pretty(fam)} seed ensemble: mutual information', FVR[(FVR.family == fam) & (FVR.variant == 'ensemble')], 'unc_mi')
        U = pd.DataFrame([{'method': n, **uncertainty_metrics(d, s, FAILURE_DICE)} for n, d, s in METHODS])
        U.to_csv(PAPER_TAB / 'table4_uncertainty_full_volume.csv', index=False)
        print(f'\nUncertainty (full volume; failure = patient mean Dice < {FAILURE_DICE}):')
        print(U[['method', 'spearman_rho', 'rho_ci_low', 'rho_ci_high', 'auroc', 'auroc_ci_low', 'auroc_ci_high',
                 'n_failures', 'aurc', 'aurc_oracle']].round(3).to_string(index=False))

        cov = np.linspace(1.0, 0.5, 11)
        fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
        for n, d, s in METHODS:
            axes[0].plot(cov * 100, risk_coverage(d, s, cov), marker='o', ms=3, label=n)
        d_or = next(d for n, d, s in METHODS if 'seed ensemble' in n and n.startswith('PH-CVT')) if any(
            'seed ensemble' in n and n.startswith('PH-CVT') for n, _, _ in METHODS) else METHODS[0][1]
        axes[0].plot(cov * 100, risk_coverage(d_or, -d_or, cov), 'k--', label='Oracle (PH-CVT ensemble)')
        axes[0].invert_xaxis(); axes[0].set_xlabel('Coverage: % of patients retained (rest referred to expert)')
        axes[0].set_ylabel('Mean Dice of retained patients'); axes[0].set_title('Risk–coverage (uncertainty-based referral)')
        axes[0].legend(fontsize=7); axes[0].grid(alpha=0.25)
        yy = np.arange(len(U))
        axes[1].errorbar(U.auroc, yy, xerr=[U.auroc - U.auroc_ci_low, U.auroc_ci_high - U.auroc], fmt='o', capsize=3)
        axes[1].axvline(0.5, color='k', lw=1, ls='--'); axes[1].set_yticks(yy, U.method, fontsize=7); axes[1].invert_yaxis()
        axes[1].set_xlabel(f'AUROC for detecting failures (Dice < {FAILURE_DICE}), 95% CI'); axes[1].set_title('Failure detection')
        axes[1].grid(axis='x', alpha=0.25)
        savefig(fig, 'fig7_uncertainty_risk_coverage_failure_detection.png')

        sel_methods = [m for m in METHODS if 'mutual information' in m[0] or 'deterministic' in m[0]][:4]
        fig, axes = plt.subplots(1, len(sel_methods), figsize=(5 * len(sel_methods), 4.5), squeeze=False)
        for ax, (n, d, s) in zip(axes[0], sel_methods):
            ax.scatter(s, 1 - d, s=10, alpha=0.6); rho = sps.spearmanr(s, 1 - d).statistic
            ax.axhline(1 - FAILURE_DICE, color='r', ls='--', lw=1)
            ax.set_title(f'{n}\nSpearman ρ = {rho:.2f}', fontsize=8); ax.set_xlabel('Patient uncertainty score'); ax.set_ylabel('1 − patient Dice')
            ax.grid(alpha=0.25)
        savefig(fig, 'fig8_uncertainty_vs_error.png')

        cal_path = STUDY_DIR / 'full_volume' / 'full_volume_calibration_bins.csv'
        if cal_path.exists():
            CAL = pd.read_csv(cal_path)
            ece = CAL.groupby(['method', 'region', 'mode']).ece.first().unstack(['mode', 'region']).round(4)
            ece.to_csv(PAPER_TAB / 'table5_calibration_ece_full_volume.csv')
            print('\nExpected calibration error (brain voxels, full volume):'); print(ece.to_string())
            fig, axes = plt.subplots(2, 3, figsize=(17, 10))
            for row, mode in enumerate(['full_brain', 'class_balanced']):
                for ax, region in zip(axes[row], REGION_NAMES):
                    for meth in CAL.method.unique():
                        part = CAL[(CAL.method == meth) & (CAL.region == region) & (CAL['mode'] == mode) & (CAL.weight > 0)]
                        ax.plot(part.confidence, part.observed_frequency, 'o-', ms=3, label=f'{meth} (ECE {part.ece.iloc[0]:.3f})')
                    ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set_title(f'{region} — {mode.replace("_", " ")}')
                    ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed tumour frequency')
                    ax.legend(fontsize=6); ax.grid(alpha=0.25)
            savefig(fig, 'fig9_reliability_full_volume.png')

    # ---------------- 5. plain-text summary for writing the manuscript -------------------------
    with open(PAPER_DIR / 'results_summary.md', 'w') as f:
        f.write('# Automatically generated results summary\n\n')
        f.write(f'Test patients: {SEL.patient_id.nunique()} | failure definition: patient mean Dice < {FAILURE_DICE} | '
                f'equivalence margin: ±{EQUIVALENCE_MARGIN}\n\n')
        f.write('## Selected-slice protocol\n```\n' + T_SEL.round(4).to_string(index=False) + '\n```\n\n')
        f.write('## Paired comparisons (selected slices)\n```\n' + P_SEL.round(4).to_string(index=False) + '\n```\n\n')
        if FVR is not None:
            f.write('## Full-volume 3D protocol\n```\n' + T_FV.round(4).to_string(index=False) + '\n```\n\n')
            f.write('## Seed ensembles\n```\n' + T_ENS.round(4).to_string(index=False) + '\n```\n\n')
            f.write('## Paired comparisons (full volume)\n```\n' + pd.concat([P_FV, P_FV_HD, P_ENS]).round(4).to_string(index=False) + '\n```\n\n')
            f.write('## Uncertainty\n```\n' + U.round(4).to_string(index=False) + '\n```\n')
    print('\nTables :', PAPER_TAB); print('Figures:', PAPER_FIG); print('Summary:', PAPER_DIR / 'results_summary.md')
else:
    print('Statistics not requested in this session.')

### NEW — Modality-contribution analysis
Each MRI sequence is removed in turn at test time (full volumes, same checkpoints and thresholds) to measure how much each model relies on it for WT, TC and ET. This is a quantitative counterpart to Grad-CAM modality maps such as those in TransXAI (Zeineldin et al., *Sci Rep* 2024).

In [ ]:
# =============================================================================
# NEW — Modality-contribution analysis (inference only, FINAL session)
# =============================================================================
# Question: how much does each model rely on each MRI sequence, per tumour region?
# Method  : for every test patient (full volume, same preprocessing and scoring as the
#           full-volume cell) one input channel at a time is replaced by 0. After the
#           per-volume z-score, 0 equals the mean brain intensity, so the sequence keeps no
#           contrast information. Nothing else changes: same checkpoints, same validation
#           thresholds, no retraining.
# Output  : dDice = Dice(all four sequences) - Dice(sequence removed), per patient and
#           region, averaged over seeds within patient; bootstrap 95% CIs over patients;
#           paired Wilcoxon tests (Holm-corrected) of PH-CVT's dDice vs each baseline's.
#           This gives a quantitative counterpart to qualitative Grad-CAM modality maps.
# Resumable: state saved after every patient; a changed checkpoint set restarts cleanly.
MOD_DIR = STUDY_DIR / 'modality_ablation'; MOD_DIR.mkdir(parents=True, exist_ok=True)
MOD_STATE = MOD_DIR / 'modality_ablation_state.pkl'
MODALITY_NAMES = ['T1', 'T1ce', 'T2', 'FLAIR']          # channel order of the training cache
CONDITIONS = [('all sequences', None)] + [(f'without {m}', i) for i, m in enumerate(MODALITY_NAMES)]

@torch.no_grad()
def dice_native(p160, thresholds, case):
    """3D Dice at the native grid (same rules as score_prediction, without HD95)."""
    H, W, D = case['shape']
    p = F.interpolate(p160, size=(H, W), mode='bilinear', align_corners=False).permute(1, 2, 3, 0)
    pred = p >= torch.tensor(thresholds, device=p.device, dtype=p.dtype).view(3, 1, 1, 1)
    gt = case['gt']
    inter = (pred & gt).sum((1, 2, 3)).double(); ps = pred.sum((1, 2, 3)).double(); gs = gt.sum((1, 2, 3)).double()
    return torch.where(ps + gs > 0, 2 * inter / (ps + gs).clamp_min(1), torch.ones_like(inter)).cpu().numpy()

def analyse_modality_ablation(MA, reference=REF):
    """MA: long table patient_id, family, seed, condition, WT, TC, ET -> (summary, tests)."""
    long = MA.melt(id_vars=['patient_id', 'family', 'seed', 'condition'], value_vars=REGION_NAMES,
                   var_name='region', value_name='dice')
    per_pat = long.groupby(['family', 'patient_id', 'region', 'condition']).dice.mean().unstack('condition')
    full = per_pat['all sequences']
    rows, deltas = [], {}
    for fam in ordered(MA.family.unique()):
        for m in MODALITY_NAMES:
            for region in REGION_NAMES:
                f_ = full.loc[(fam, slice(None), region)]
                d_ = per_pat[f'without {m}'].loc[(fam, slice(None), region)]
                delta = (f_ - d_)
                delta.index = delta.index.get_level_values('patient_id')
                deltas[(fam, m, region)] = delta
                lo, hi = bootstrap_ci(delta.to_numpy())
                rows.append({'model': fam, 'label': pretty(fam), 'removed_sequence': m, 'region': region,
                             'dice_all_sequences': f_.mean(), 'dice_without': d_.mean(),
                             'delta_dice': delta.mean(), 'delta_ci95_low': lo, 'delta_ci95_high': hi,
                             'n_seeds': MA[MA.family == fam].seed.nunique(), 'n_patients': len(delta)})
    summary = pd.DataFrame(rows)
    tests = []
    if reference in set(MA.family):
        for fam in ordered(MA.family.unique()):
            if fam == reference: continue
            for m in MODALITY_NAMES:
                for region in REGION_NAMES:
                    a = deltas[(reference, m, region)]; b = deltas[(fam, m, region)]
                    diff = (a - b.reindex(a.index)).dropna().to_numpy()
                    try: p = sps.wilcoxon(diff).pvalue
                    except ValueError: p = np.nan
                    lo, hi = bootstrap_ci(diff)
                    tests.append({'reference': pretty(reference), 'comparator': pretty(fam), 'removed_sequence': m,
                                  'region': region, 'reference_minus_comparator_delta': diff.mean(),
                                  'ci95_low': lo, 'ci95_high': hi, 'wilcoxon_p': p})
    tests = pd.DataFrame(tests)
    if len(tests): tests['wilcoxon_p_holm'] = holm(tests.wilcoxon_p)
    return summary, tests

def plot_modality_ablation(summary):
    fams = ordered(summary.model.unique())
    vmax = max(0.05, float(summary.delta_dice.max())); vmin = min(0.0, float(summary.delta_dice.min()))
    fig, axes = plt.subplots(1, len(fams), figsize=(3.6 * len(fams) + 1.5, 4.2), squeeze=False)
    for k, (ax, fam) in enumerate(zip(axes[0], fams)):
        mat = (summary[summary.model == fam].pivot(index='removed_sequence', columns='region', values='delta_dice')
                      .reindex(index=MODALITY_NAMES, columns=REGION_NAMES))
        im = ax.imshow(mat.to_numpy(), cmap='Reds', vmin=vmin, vmax=vmax)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat.iloc[i, j]
                ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9,
                        color='white' if v > 0.6 * vmax else 'black')
        ax.set_xticks(range(3), REGION_NAMES); ax.set_yticks(range(4), [f'without {m}' for m in MODALITY_NAMES] if k == 0 else [''] * 4)
        ax.set_title(pretty(fam), fontsize=10)
    fig.colorbar(im, ax=axes[0].tolist(), fraction=0.02, label='Dice drop (all sequences − sequence removed)')
    fig.suptitle('Contribution of each MRI sequence to each tumour region (full-volume 3D Dice)', fontsize=11)
    fig.savefig(PAPER_FIG / 'fig10_modality_ablation_heatmaps.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
    w = 0.8 / len(fams); xx = np.arange(len(MODALITY_NAMES))
    for ax, region in zip(axes, REGION_NAMES):
        for k, fam in enumerate(fams):
            part = summary[(summary.model == fam) & (summary.region == region)].set_index('removed_sequence').reindex(MODALITY_NAMES)
            ax.bar(xx + (k - (len(fams) - 1) / 2) * w, part.delta_dice, w,
                   yerr=[part.delta_dice - part.delta_ci95_low, part.delta_ci95_high - part.delta_dice],
                   capsize=2, label=pretty(fam), color='#d95f02' if fam == REF else None, alpha=0.85)
        ax.set_xticks(xx, [f'without\n{m}' for m in MODALITY_NAMES]); ax.set_title(region)
        ax.axhline(0, color='k', lw=0.8); ax.grid(axis='y', alpha=0.25)
    axes[0].set_ylabel('Dice drop (95% bootstrap CI)'); axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(PAPER_FIG / 'fig11_modality_ablation_bars.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

if RUN_MODALITY_ABLATION:
    have = [pid for pid in test_ids if raw_patient_complete(pid)]
    if len(have) != len(test_ids):
        raise FileNotFoundError(f'Only {len(have)}/{len(test_ids)} raw test patients on Drive; run the extraction first.')
    RAW_LOCAL.mkdir(parents=True, exist_ok=True)
    for pid in [p for p in test_ids if not raw_patient_complete(p, RAW_LOCAL)]:
        shutil.copytree(RAW_TEST_DRIVE / pid, RAW_LOCAL / pid, dirs_exist_ok=True)

    MA_MEMBERS = {}
    for fam in MODALITY_ABLATION_FAMILIES:
        mem = member_list(fam)
        if MODALITY_ABLATION_MAX_SEEDS: mem = mem[:MODALITY_ABLATION_MAX_SEEDS]
        if mem: MA_MEMBERS[fam] = mem
        print(f'{fam}: seeds {[s for s, _, _ in mem]}')
    signature = {f: [(s, str(c), os.path.getsize(c), th) for s, c, th in m] for f, m in MA_MEMBERS.items()}
    state = None
    if MOD_STATE.exists():
        with open(MOD_STATE, 'rb') as fh: state = pickle.load(fh)
        if state.get('signature') != signature:
            os.replace(MOD_STATE, MOD_STATE.with_name(f'modality_ablation_state_stale_{int(time.time())}.pkl'))
            print('Checkpoint set changed: previous modality-ablation state archived.'); state = None
    if state is None: state = {'signature': signature, 'done': [], 'rows': []}
    print(f"Modality ablation: {len(state['done'])}/{len(test_ids)} patients already done")

    MA_NETS = {(f, s): (load_member(f, s, c), th) for f, m in MA_MEMBERS.items() for s, c, th in m}
    done = set(state['done'])
    for pid in tqdm([p for p in test_ids if p not in done], desc='Modality ablation patients'):
        case = load_patient(pid); rows = []
        for cond, ch in CONDITIONS:
            x = case['x']
            if ch is not None:
                x = x.clone(); x[:, ch] = 0.0
            for (fam, seed), (net, th) in MA_NETS.items():
                d = dice_native(predict_probs(net, x), th, case)
                rows.append({'patient_id': pid, 'family': fam, 'seed': seed, 'condition': cond,
                             'WT': float(d[0]), 'TC': float(d[1]), 'ET': float(d[2])})
            del x
        state['rows'].extend(rows); state['done'].append(pid)
        tmp = MOD_STATE.with_suffix('.tmp')
        with open(tmp, 'wb') as fh: pickle.dump(state, fh, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, MOD_STATE)
        del case
    for key in list(MA_NETS): MA_NETS[key][0].to('cpu')
    del MA_NETS; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    MA = pd.DataFrame(state['rows']); MA.to_csv(MOD_DIR / 'modality_ablation_patient_rows.csv', index=False)

    # Sanity check: the "all sequences" condition must reproduce the full-volume Dice.
    fv_csv = FV_DIR / 'full_volume_patient_rows.csv'
    if fv_csv.exists():
        fvc = pd.read_csv(fv_csv, dtype={'member': str}); fvc = fvc[fvc.variant == 'single'].copy()
        fvc['seed'] = fvc.member.astype(int)
        chk = MA[MA.condition == 'all sequences'].merge(fvc, on=['patient_id', 'family', 'seed'])
        if len(chk):
            gap = max(float((chk[r] - chk[f'{r}_dice']).abs().max()) for r in REGION_NAMES)
            print(f'Consistency with full-volume cell: max |dDice| = {gap:.2e} over {len(chk)} rows '
                  '(tiny non-zero values come from non-deterministic GPU kernels)')

    MA_SUM, MA_TEST = analyse_modality_ablation(MA)
    MA_SUM.to_csv(PAPER_TAB / 'table6_modality_ablation.csv', index=False)
    MA_TEST.to_csv(PAPER_TAB / 'table6b_modality_reliance_vs_phcvt.csv', index=False)
    print('\nDice drop when one sequence is removed (mean over patients, seeds averaged within patient):')
    print(MA_SUM.pivot_table(index=['label', 'removed_sequence'], columns='region', values='delta_dice')
                .reindex(columns=REGION_NAMES).round(3).to_string())
    if len(MA_TEST):
        sig = MA_TEST[MA_TEST.wilcoxon_p_holm < 0.05]
        print(f'\nPH-CVT vs baselines: {len(sig)}/{len(MA_TEST)} modality×region reliance differences significant after Holm.')
    plot_modality_ablation(MA_SUM)
    with open(PAPER_DIR / 'results_summary_modality_ablation.md', 'w') as fh:
        fh.write('# Modality ablation (Dice drop when a sequence is removed)\n```\n' +
                 MA_SUM.round(4).to_string(index=False) + '\n```\n')
else:
    print('Modality ablation not requested in this session.')

## Part 5 — EXTERNAL session: frozen validation on OpenNeuro ds007045 (Kaggle mirror)
337 preoperative IDH-wildtype glioblastomas from 8 hospitals (Filimonova et al., *Sci Data* 13:1213, 2026; OpenNeuro DOI 10.18112/openneuro.ds007045.v1.0.1). Set `SESSION = "EXTERNAL"` and choose **Run all** (after FINAL, so the internal-vs-external comparison is available).

* **Step 1** downloads once from Kaggle to local disk, copies only `processing_MNI152_skull-stripped_N4` and `segmentation` to Drive, maps every subject's four sequences and mask, and shows an orientation check image — **look at it before trusting the numbers**.
* **Step 2** runs every checkpoint frozen (no fine-tuning, BraTS-2021 validation thresholds), resumable per subject.
* **Step 3** writes tables and figures to `paper_outputs/tables/external` and `paper_outputs/figures/external`.

Report the reference standard as *expert-reviewed semi-automatic segmentations*, and cite both the data paper and the OpenNeuro DOI.

In [ ]:
# =============================================================================
# NEW — EXTERNAL COHORT, STEP 1: download, file mapping, geometry harmonisation, QC
# =============================================================================
# Cohort : OpenNeuro ds007045 (Filimonova et al., Sci Data 13:1213, 2026), Kaggle mirror.
#          337 preoperative IDH-wildtype glioblastomas, 8 hospitals, 1.5 T / 3 T,
#          Siemens / Philips / GE. Masks: automatic CNN + neuroradiologist review
#          (labels 1 necrotic core, 2 oedema/non-enhancing, 3 enhancing = BraTS convention).
# Steps  : 1) download once from Kaggle to LOCAL disk, copy only the two needed derivative
#             folders to Drive, delete the local copy (idempotent, marker file);
#          2) map every subject's T1 / T1ce / T2 / FLAIR / mask automatically (BIDS names:
#             run-01_T1w = pre-contrast, run-02_T1w = post-contrast) and report ambiguities;
#          3) helper functions that bring each volume into the BraTS array orientation and a
#             240 x 240 in-plane grid by zero-PADDING (never stretching), so one pixel keeps the
#             same physical size as in training (1 mm -> 1.5 mm after the 240->160 resize).
import re, glob

EXT_DIR = DRIVE_BASE / 'external' / EXTERNAL_NAME
EXT_IMG = EXT_DIR / 'derivatives' / EXTERNAL_IMAGE_FOLDER
EXT_SEG = EXT_DIR / 'derivatives' / EXTERNAL_MASK_FOLDER
EXT_MARKER = EXT_DIR / 'COPY_COMPLETE.json'
EXT_OUT = STUDY_DIR / 'external' / EXTERNAL_NAME
EXT_FIG = PAPER_FIG / 'external'; EXT_TAB = PAPER_TAB / 'external'
for _p in (EXT_OUT, EXT_FIG, EXT_TAB): _p.mkdir(parents=True, exist_ok=True)
EXT_MODS = ['t1', 't1ce', 't2', 'flair']          # channel order of the training cache

# ----------------------------------------------------------------------------- 1) download
def print_tree(root, depth=3, limit=80):
    shown = 0
    for p in sorted(root.rglob('*')):
        d = len(p.relative_to(root).parts)
        if p.is_dir() and d <= depth:
            print('  ' * (d - 1) + f'{p.name}/'); shown += 1
            if shown >= limit: print('  ...'); break

def acquire_external():
    if EXT_MARKER.exists():
        print('External cohort already on Drive:', json.loads(EXT_MARKER.read_text())); return
    print(f"Free local disk: {shutil.disk_usage('/content').free / 1e9:.1f} GB (the Kaggle archive is unpacked there)")
    configure_kaggle_auth()
    os.environ['KAGGLEHUB_CACHE'] = str(LOCAL_ROOT / 'kaggle_external_cache')
    root = Path(kagglehub.dataset_download(EXTERNAL_KAGGLE_HANDLE))
    print('Downloaded to', root)
    found = {k: next((p for p in root.rglob(k) if p.is_dir()), None) for k in (EXTERNAL_IMAGE_FOLDER, EXTERNAL_MASK_FOLDER)}
    if not all(found.values()):
        print_tree(root)
        raise FileNotFoundError(f'Folders {list(found)} not found in the Kaggle copy (tree printed above).')
    for name, src in found.items():
        print('Copying to Drive:', name)
        shutil.copytree(src, EXT_DIR / 'derivatives' / name, dirs_exist_ok=True)
    for meta in ['participants.tsv', 'participants.json', 'README.txt', 'README', 'dataset_description.json']:
        hit = next(root.rglob(meta), None)
        if hit is not None: shutil.copy2(hit, EXT_DIR / hit.name)
    n_img = sum(1 for p in EXT_IMG.iterdir() if p.is_dir() and p.name.startswith('sub-'))
    atomic_json({'source': f'kaggle:{EXTERNAL_KAGGLE_HANDLE}', 'original': 'OpenNeuro ds007045 v1.0.1',
                 'subjects_with_images': n_img, 'copied_unix_time': time.time()}, EXT_MARKER)
    shutil.rmtree(root, ignore_errors=True)
    print(f'Copied {n_img} subjects to {EXT_DIR}; local download deleted.')

# ----------------------------------------------------------------------------- 2) file mapping
def classify_sequence(filename):
    """BIDS-aware sequence classification; returns t1 / t1ce / t2 / flair / None."""
    stem = filename.lower().replace('.nii.gz', '').replace('.nii', '')
    for mod, key in EXTERNAL_FILE_OVERRIDES.items():
        if key.lower() in stem: return mod
    tokens = re.split(r'[_.]', stem)
    if any('flair' in t for t in tokens): return 'flair'
    if any(t in ('t2w', 't2') or t.startswith('t2w') for t in tokens): return 't2'
    if any(t.startswith('t1') for t in tokens):
        post = any(t in ('run-02', 'run-2', 't1ce', 't1c', 't1gd', 't1post', 'post', 'postcontrast', 'gd', 'contrast')
                   or t.startswith('ce-') for t in tokens)
        pre = any(t in ('run-01', 'run-1', 'pre', 'precontrast') for t in tokens)
        if post and not pre: return 't1ce'
        return 't1'
    return None

def site_lookup():
    """Hospital per subject: participants.tsv column if present, else the letters of the subject ID."""
    tsv = EXT_DIR / 'participants.tsv'
    if tsv.exists():
        pt = pd.read_csv(tsv, sep='\t', dtype=str)
        col = next((c for c in pt.columns if re.search(r'site|center|centre|hospital|institution', c, re.I)), None)
        if col is not None and 'participant_id' in pt.columns:
            return dict(zip(pt.participant_id, pt[col])), f'participants.tsv:{col}'
    return None, 'subject-id prefix'

def subject_site(sub, table):
    if table and sub in table and isinstance(table[sub], str): return table[sub]
    m = re.match(r'sub-([A-Za-z]+)', sub)
    return m.group(1).upper() if m else 'unknown'

def build_external_index():
    subs = sorted(p.name for p in EXT_IMG.iterdir() if p.is_dir() and p.name.startswith('sub-'))
    table, site_source = site_lookup()
    rows = []
    for sub in subs:
        row = {'subject': sub, 'site': subject_site(sub, table), 'status': 'ok', 'problem': ''}
        files = [f for f in (EXT_IMG / sub).rglob('*.nii*') if f.is_file()]
        for mod in EXT_MODS:
            hits = [f for f in files if classify_sequence(f.name) == mod]
            if len(hits) == 1: row[mod] = str(hits[0])
            else:
                row[mod] = ''; row['status'] = 'excluded'
                row['problem'] += f'{mod}: {len(hits)} candidate(s) {[h.name for h in hits]}; '
        segs = [f for f in list((EXT_SEG / sub).rglob('*.nii*')) + [Path(p) for p in glob.glob(str(EXT_SEG / f'{sub}*.nii*'))]
                if f.is_file()]
        segs = sorted(set(segs))
        if len(segs) > 1:
            pref = [f for f in segs if re.search(r'seg|mask|label|dseg', f.name, re.I)]
            segs = pref if len(pref) == 1 else segs
        if len(segs) == 1: row['seg'] = str(segs[0])
        else:
            row['seg'] = ''; row['status'] = 'excluded'; row['problem'] += f'mask: {len(segs)} candidate(s); '
        rows.append(row)
    idx = pd.DataFrame(rows)
    idx.to_csv(EXT_OUT / 'external_file_index.csv', index=False)
    print(f'Subjects: {len(idx)} | usable: {(idx.status == "ok").sum()} | excluded: {(idx.status != "ok").sum()} '
          f'| site source: {site_source}')
    print('Subjects per site:', idx[idx.status == 'ok'].site.value_counts().to_dict())
    if (idx.status != 'ok').any():
        print('Examples of problems (set EXTERNAL_FILE_OVERRIDES if these are systematic):')
        print(idx[idx.status != 'ok'][['subject', 'problem']].head(8).to_string(index=False))
    ex = idx[idx.status == 'ok'].head(1)
    if len(ex):
        print('Example mapping:', {m: Path(ex.iloc[0][m]).name for m in EXT_MODS + ['seg']})
    return idx

# ----------------------------------------------------------------------------- 3) geometry
_AX = {'R': (0, 1), 'L': (0, -1), 'A': (1, 1), 'P': (1, -1), 'S': (2, 1), 'I': (2, -1)}

def axcodes_from_affine(affine):
    """Direction each voxel axis points to (nibabel RAS+ letter convention)."""
    A = np.asarray(affine, float)[:3, :3]; used, codes = set(), []
    for i in range(3):
        w = next(int(k) for k in np.argsort(-np.abs(A[:, i])) if int(k) not in used); used.add(w)
        codes.append(('R', 'A', 'S')[w] if A[w, i] > 0 else ('L', 'P', 'I')[w])
    return tuple(codes)

def reorient_array(arr, src_codes, dst_codes):
    """Permute / flip voxel axes so the array has the destination orientation."""
    src = [_AX[c] for c in src_codes]; dst = [_AX[c] for c in dst_codes]
    perm = [next(i for i, (w, _) in enumerate(src) if w == wd) for wd, _ in dst]
    out = np.transpose(arr, perm)
    for j, (_, sd) in enumerate(dst):
        if src[perm[j]][1] != sd: out = np.flip(out, axis=j)
    return np.ascontiguousarray(out), perm

def pad_or_crop_inplane(arrays, brain, target=240):
    """Zero-pad (or, if larger, crop around the brain centre) axes 0 and 1 to target x target."""
    out = list(arrays); info = []
    for ax in (0, 1):
        n = out[0].shape[ax]
        if n <= target:
            before = (target - n) // 2; after = target - n - before
            pad = [(0, 0)] * out[0].ndim; pad[ax] = (before, after)
            out = [np.pad(a, pad) for a in out]; info.append(('pad', before, after))
        else:
            coords = np.where(brain.any(axis=tuple(i for i in range(brain.ndim) if i != ax)))[0]
            centre = int((coords.min() + coords.max()) // 2) if len(coords) else n // 2
            start = int(np.clip(centre - target // 2, 0, n - target))
            sl = [slice(None)] * out[0].ndim; sl[ax] = slice(start, start + target)
            out = [a[tuple(sl)] for a in out]; info.append(('crop', start, n - start - target))
            brain = brain[tuple(sl)]
    return out, info

def external_brain_mask(vols):
    """Union of non-zero voxels. If the background is not exactly zero (unexpected for the
    skull-stripped folder), the border median of each sequence is treated as background."""
    brain = np.any(np.stack([v != 0 for v in vols]), axis=0)
    if brain.mean() > 0.6:
        masks = []
        for v in vols:
            border = np.concatenate([v[[0, -1]].ravel(), v[:, [0, -1]].ravel(), v[:, :, [0, -1]].ravel()])
            bg = float(np.median(border)); tol = 1e-3 * (float(np.abs(v).max()) + 1e-6)
            masks.append(np.abs(v - bg) > tol)
        brain = np.any(np.stack(masks), axis=0)
    return brain

def load_external_case(row, ref_codes):
    """External subject -> same dict as load_patient() (training-identical preprocessing)."""
    vols = []
    for m in EXT_MODS:
        img = nib.load(row[m]); a = np.asarray(img.dataobj, dtype=np.float32)
        if a.ndim == 4: a = a[..., 0]
        a, _ = reorient_array(a, axcodes_from_affine(img.affine), ref_codes); vols.append(a)
    seg_img = nib.load(row['seg']); seg = np.rint(np.asarray(seg_img.dataobj, dtype=np.float32))
    if seg.ndim == 4: seg = seg[..., 0]
    seg, perm = reorient_array(seg, axcodes_from_affine(seg_img.affine), ref_codes)
    seg = seg.astype(np.int16); seg[seg == 4] = 3; seg = np.clip(seg, 0, 3).astype(np.uint8)
    shapes = {v.shape for v in vols} | {seg.shape}
    if len(shapes) != 1: raise ValueError(f'shape mismatch between sequences/mask: {shapes}')
    spacing = tuple(float(z) for z in np.asarray(seg_img.header.get_zooms()[:3])[perm])
    brain = external_brain_mask(vols); brain_fraction = float(brain.mean())
    for v in vols: v[~brain] = 0
    (*vols, seg, brain_u8), geom = pad_or_crop_inplane(vols + [seg, brain.astype(np.uint8)], brain)
    brain = brain_u8.astype(bool)
    vols = [robust_normalize(v) for v in vols]                    # identical to training
    H, W, D = seg.shape
    x = torch.from_numpy(np.ascontiguousarray(np.stack(vols, 0).transpose(3, 0, 1, 2)))
    x = F.interpolate(x, size=(CONFIG['image_size'],) * 2, mode='bilinear', align_corners=False).half().float()
    gt = label_to_regions(seg).astype(bool)
    qc = {'shape_after': (H, W, D), 'spacing': spacing, 'brain_fraction': brain_fraction, 'geometry': str(geom),
          'labels': sorted(np.unique(seg).tolist()), 'wt_ml': float(gt[0].sum() * np.prod(spacing) / 1000)}
    return {'pid': row['subject'], 'x': x.to(DEVICE), 'gt_np': gt, 'gt': torch.from_numpy(gt).to(DEVICE),
            'brain': torch.from_numpy(brain).to(DEVICE), 'shape': (H, W, D), 'spacing': spacing,
            'flair': vols[3], 'seg': seg}, qc

# ----------------------------------------------------------------------------- run step 1
if RUN_EXTERNAL:
    acquire_external()
    EXT_INDEX = build_external_index()
    ref_file = next(RAW_TEST_DRIVE.rglob('*_flair.nii.gz'), None)
    if ref_file is None:
        raise FileNotFoundError('A BraTS 2021 raw test volume is needed to define the training orientation. '
                                'Keep EXTRACT_RAW_TEST = True (it runs earlier in this session).')
    EXT_REF_CODES = axcodes_from_affine(nib.load(str(ref_file)).affine)
    print('BraTS training orientation:', EXT_REF_CODES, '| reference file:', ref_file.name)
    # Quality control on three subjects: geometry, background, labels, tumour volume.
    for _, r in EXT_INDEX[EXT_INDEX.status == 'ok'].head(3).iterrows():
        img = nib.load(r['flair'])
        _case, _qc = load_external_case(r, EXT_REF_CODES)
        print(f"  {r['subject']} [{r['site']}] original {img.shape} {axcodes_from_affine(img.affine)} "
              f"{np.round(img.header.get_zooms()[:3], 2)} -> {_qc['shape_after']} | brain {_qc['brain_fraction']:.2f} "
              f"| labels {_qc['labels']} | WT {_qc['wt_ml']:.1f} mL | {_qc['geometry']}")
        del _case
    # Visual orientation check: external vs BraTS axial slice through the tumour centre.
    _r = EXT_INDEX[EXT_INDEX.status == 'ok'].iloc[0]; _case, _ = load_external_case(_r, EXT_REF_CODES)
    _b = None
    if raw_patient_complete(test_ids[0]):                         # one BraTS case for side-by-side display
        shutil.copytree(RAW_TEST_DRIVE / test_ids[0], RAW_LOCAL / test_ids[0], dirs_exist_ok=True)
        _b = load_patient(test_ids[0])
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    dz = int(_case['gt_np'][0].sum((0, 1)).argmax())
    axes[0].imshow(np.rot90(_case['flair'][:, :, dz]), cmap='gray'); axes[0].contour(np.rot90(_case['gt_np'][0][:, :, dz]), colors='y', linewidths=0.8)
    axes[0].set_title(f"External {_r['subject']} (after harmonisation)")
    if _b is not None:
        db = int(_b['gt_np'][0].sum((0, 1)).argmax())
        axes[1].imshow(np.rot90(_b['flair'][:, :, db]), cmap='gray'); axes[1].contour(np.rot90(_b['gt_np'][0][:, :, db]), colors='y', linewidths=0.8)
        axes[1].set_title(f'BraTS 2021 {test_ids[0]} (training orientation)')
    for a in axes: a.axis('off')
    fig.suptitle('Orientation check: both brains must face the same way (nose up, same left/right)')
    fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_orientation_check.png', dpi=150); plt.show(); plt.close(fig)
    del _case, _b
else:
    print('External cohort not requested in this session.')

In [ ]:
# =============================================================================
# NEW — EXTERNAL COHORT, STEP 2: frozen full-volume inference (resumable)
# =============================================================================
# Exactly the internal full-volume protocol, applied to every usable external subject:
#   * every completed seed of each family (own validation thresholds), seed ensembles
#     (mean of member thresholds) and 12-pass MC dropout for PH-CVT seed 42;
#   * 3D Dice and HD95 (mm) on the harmonised grid, patient-level uncertainty scores and
#     voxel-level calibration histograms.
# Nothing is tuned on external data. State is saved after every subject; subjects that fail
# to load are logged (reason) and retried on the next run.
EXT_STATE = EXT_OUT / 'external_state.pkl'

if RUN_EXTERNAL:
    usable = EXT_INDEX[EXT_INDEX.status == 'ok'].reset_index(drop=True)
    EXT_MEMBERS = {}
    ensure_transformer_baseline_deps(EXTERNAL_FAMILIES)
    for fam in EXTERNAL_FAMILIES:
        mem = member_list(fam)
        if EXTERNAL_MAX_SEEDS: mem = mem[:EXTERNAL_MAX_SEEDS]
        if mem: EXT_MEMBERS[fam] = mem
        print(f'{fam}: seeds {[s for s, _, _ in mem]}')
    signature = {'members': {f: [(s, str(c), os.path.getsize(c), th) for s, c, th in m] for f, m in EXT_MEMBERS.items()},
                 'mc': (RUN_MC_DROPOUT_FULLVOLUME, MC_PASSES_FULLVOLUME), 'ref_codes': EXT_REF_CODES,
                 'image_folder': EXTERNAL_IMAGE_FOLDER, 'overrides': dict(EXTERNAL_FILE_OVERRIDES)}
    state = None
    if EXT_STATE.exists():
        with open(EXT_STATE, 'rb') as fh: state = pickle.load(fh)
        if state.get('signature') != signature:
            os.replace(EXT_STATE, EXT_STATE.with_name(f'external_state_stale_{int(time.time())}.pkl'))
            print('Checkpoint set or preprocessing changed: previous external state archived.'); state = None
    if state is None:
        state = {'signature': signature, 'done': [], 'rows': [], 'calib': {}, 'qc': {}, 'failed': {}, 'seconds': []}
    done = set(state['done'])
    todo = usable[~usable.subject.isin(done)]
    print(f"External inference: {len(done)}/{len(usable)} subjects done, {len(todo)} to go")

    EXT_NETS = {(f, s): (load_member(f, s, c), th) for f, m in EXT_MEMBERS.items() for s, c, th in m}
    for _, r in tqdm(todo.iterrows(), total=len(todo), desc='External subjects'):
        t0 = time.time()
        try:
            case, qc = load_external_case(r, EXT_REF_CODES)
        except Exception as exc:
            state['failed'][r.subject] = repr(exc); print(f'  [load failed] {r.subject}: {exc!r}')
            continue
        state['failed'].pop(r.subject, None)
        rows = []; tag = {'patient_id': r.subject, 'site': r.site}
        for fam, members in EXT_MEMBERS.items():
            s1 = s2 = sent = 0; ths = []
            for seed, _, th in members:
                p = predict_probs(EXT_NETS[(fam, seed)][0], case['x']); ent = binary_entropy(p)
                row, p_nat = score_prediction(p, th, case)
                row.update(uncertainty_scores(p, ent)); row.update(tag, family=fam, member=str(seed), variant='single', n_members=1)
                rows.append(row)
                if fam == 'PHCVT_proposed' and seed == 42:
                    add_calibration(state['calib'], 'PH-CVT seed 42 (deterministic)', p_nat, case)
                del p_nat
                s1 = s1 + p; s2 = s2 + p * p; sent = sent + ent; ths.append(th); del p, ent
            k = len(members)
            if k >= 2:
                mean = s1 / k; var = (s2 / k - mean * mean).clamp_min(0)
                total = binary_entropy(mean); mi = (total - sent / k).clamp_min(0)
                row, p_nat = score_prediction(mean, list(np.mean(ths, axis=0)), case)
                row.update(uncertainty_scores(mean, total, mi, var)); row.update(tag, family=fam, member='ensemble', variant='ensemble', n_members=k)
                rows.append(row); add_calibration(state['calib'], f'{fam} {k}-seed ensemble', p_nat, case)
                del mean, var, total, mi, p_nat
            del s1, s2, sent
        if RUN_MC_DROPOUT_FULLVOLUME and ('PHCVT_proposed', 42) in EXT_NETS:
            mean, var, exp_ent, _ = predict_mc(EXT_NETS[('PHCVT_proposed', 42)][0], case['x'], MC_PASSES_FULLVOLUME)
            total = binary_entropy(mean); mi = (total - exp_ent).clamp_min(0)
            row, p_nat = score_prediction(mean, list(seg_thresholds), case)
            row.update(uncertainty_scores(mean, total, mi, var))
            row.update(tag, family='PHCVT_proposed', member='mc_dropout', variant='mc_dropout', n_members=MC_PASSES_FULLVOLUME)
            rows.append(row); add_calibration(state['calib'], 'PH-CVT seed 42 MC dropout', p_nat, case)
            del mean, var, exp_ent, total, mi, p_nat
        state['rows'].extend(rows); state['done'].append(r.subject); state['qc'][r.subject] = qc
        state['seconds'].append(time.time() - t0)
        tmp = EXT_STATE.with_suffix('.tmp')
        with open(tmp, 'wb') as fh: pickle.dump(state, fh, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, EXT_STATE)
        del case
        if len(state['done']) % 10 == 0 and DEVICE.type == 'cuda': torch.cuda.empty_cache()

    EXT_FV = pd.DataFrame(state['rows'])
    EXT_FV.to_csv(EXT_OUT / 'external_patient_rows.csv', index=False)
    calibration_table(state['calib']).to_csv(EXT_OUT / 'external_calibration_bins.csv', index=False)
    pd.DataFrame([{'subject': k, **v} for k, v in state['qc'].items()]).to_csv(EXT_OUT / 'external_qc.csv', index=False)
    if state['failed']:
        pd.DataFrame([{'subject': k, 'reason': v} for k, v in state['failed'].items()]).to_csv(EXT_OUT / 'external_failed.csv', index=False)
        print(f"{len(state['failed'])} subjects could not be loaded (see external_failed.csv).")
    print(f"External inference complete: {EXT_FV.patient_id.nunique()} subjects, "
          f"median {np.median(state['seconds']) if state['seconds'] else float('nan'):.1f} s/subject")
    print(EXT_FV.groupby(['family', 'variant'])[['WT_dice', 'TC_dice', 'ET_dice', 'mean_dice', 'mean_hd95']].mean().round(4).to_string())

    # ---------------- qualitative external cases (10th / 50th / 90th percentile, never hand-picked)
    ref_rows = EXT_FV[(EXT_FV.family == 'PHCVT_proposed') & (EXT_FV.variant == ('ensemble' if 'ensemble' in set(EXT_FV.variant) else 'single'))]
    if 'ensemble' not in set(ref_rows.variant): ref_rows = ref_rows[ref_rows.member == '42']
    ref_rows = ref_rows.sort_values('mean_dice').reset_index(drop=True)
    picks = list(dict.fromkeys(ref_rows.iloc[int(round(q * (len(ref_rows) - 1)))].patient_id for q in (0.1, 0.5, 0.9)))
    cmp_family = next((f for f in ['UNet', 'ResUNet', 'AttentionUNet'] if f in EXT_MEMBERS), None)
    def _rgba(wt, tc, et):
        o = np.zeros(wt.shape + (4,)); o[wt & ~tc] = (0.2, 0.8, 0.2, 0.55); o[tc & ~et] = (0.9, 0.2, 0.2, 0.65); o[et] = (1, 0.85, 0.1, 0.75)
        return o
    fig, axes = plt.subplots(len(picks), 6, figsize=(21, 3.6 * len(picks)), squeeze=False)
    for i, pid in enumerate(picks):
        r = usable[usable.subject == pid].iloc[0]; case, _ = load_external_case(r, EXT_REF_CODES)
        d = int(case['gt_np'][0].sum((0, 1)).argmax()); xs = case['x'][d:d + 1]
        up = lambda t: F.interpolate(t, size=case['shape'][:2], mode='bilinear', align_corners=False)[0].cpu().numpy()
        def fam_pred(fam):
            st = torch.stack([predict_probs(EXT_NETS[(fam, s)][0], xs) for s, _, _ in EXT_MEMBERS[fam]])
            mean = st.mean(0); mi = (binary_entropy(mean) - binary_entropy(st).mean(0)).clamp_min(0)
            th = np.mean([t for _, _, t in EXT_MEMBERS[fam]], axis=0)
            return up(mean) >= th[:, None, None], up(mi).mean(0)
        ph_pred, ph_mi = fam_pred('PHCVT_proposed'); gt = case['gt_np'][:, :, :, d]
        flair = np.rot90(case['flair'][:, :, d]); err = np.zeros(gt[0].shape + (4,))
        err[ph_pred[0] & ~gt[0]] = (1, .1, .1, .8); err[~ph_pred[0] & gt[0]] = (.1, .4, 1, .8)
        dice_here = float(ref_rows[ref_rows.patient_id == pid].mean_dice.iloc[0])
        panels = [(f'{pid} [{r.site}] slice {d}', None), ('Reference mask', _rgba(*gt)),
                  (f'PH-CVT ensemble (case Dice {dice_here:.2f})', _rgba(*ph_pred)), ('WT error (red FP, blue FN)', err)]
        for j, (title, ov) in enumerate(panels):
            axes[i, j].imshow(flair, cmap='gray')
            if ov is not None: axes[i, j].imshow(np.rot90(ov))
            axes[i, j].set_title(title, fontsize=9); axes[i, j].axis('off')
        im = axes[i, 4].imshow(np.rot90(ph_mi), cmap='magma'); axes[i, 4].set_title('Ensemble MI', fontsize=9); axes[i, 4].axis('off')
        fig.colorbar(im, ax=axes[i, 4], fraction=0.046)
        axes[i, 5].imshow(flair, cmap='gray'); axes[i, 5].axis('off')
        if cmp_family:
            c_pred, _ = fam_pred(cmp_family); axes[i, 5].imshow(np.rot90(_rgba(*c_pred))); axes[i, 5].set_title(f'{cmp_family} ensemble', fontsize=9)
        del case
    fig.suptitle('External cohort (ds007045): cases at the 10th/50th/90th percentile of PH-CVT ensemble Dice\n'
                 'green = oedema/non-enhancing, red = necrotic core, yellow = enhancing', fontsize=11)
    fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig_qualitative.png', dpi=220); plt.show(); plt.close(fig)

    for key in list(EXT_NETS): EXT_NETS[key][0].to('cpu')
    del EXT_NETS; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
else:
    print('External inference not requested in this session.')

In [ ]:
# =============================================================================
# NEW — EXTERNAL COHORT, STEP 3: statistics, generalisation gap, uncertainty, figures
# =============================================================================
# Reads files only (re-runnable on its own). Outputs go to paper_outputs/tables/external and
# paper_outputs/figures/external:
#   E1 single models (mean ± SD across seeds)      E2 seed ensembles (patient-bootstrap CI)
#   E3 paired comparisons vs PH-CVT (Wilcoxon + Holm, bootstrap CI, equivalence ±margin)
#   E4 per-hospital results                        E5 internal -> external performance drop,
#      and whether PH-CVT's drop differs from each baseline's (bootstrap over both cohorts)
#   E6 uncertainty on external data + uncertainty shift internal -> external (Mann-Whitney)
#   E7 calibration (ECE) on external data
def _single(df):
    s = df[df.variant == 'single'].copy(); s['model'] = s.family; s['seed'] = s.member.astype(int); return s

def _patient_mean(df, col='mean_dice'):
    """Seed-averaged value per (model, patient)."""
    return df.groupby(['model', 'patient_id'])[col].mean()

def generalisation_gap(int_single, ext_single, reference=REF, n_boot=N_BOOT, seed=42):
    """Drop = internal mean Dice - external mean Dice (seed-averaged per patient).
    Bootstrap resamples patients within each cohort; the SAME resampled patients are used for
    every model, so differences between models' drops are paired (more precise and fair)."""
    rng = np.random.default_rng(seed)
    wi = _patient_mean(int_single).unstack('model'); we = _patient_mean(ext_single).unstack('model')
    models = [m for m in ordered(ext_single.model.unique()) if m in wi.columns and m in we.columns]
    wi, we = wi[models].dropna(), we[models].dropna()
    obs = wi.mean() - we.mean()
    ii = rng.integers(0, len(wi), size=(n_boot, len(wi))); ie = rng.integers(0, len(we), size=(n_boot, len(we)))
    A, B = wi.to_numpy(), we.to_numpy()
    boot = A[ii].mean(1) - B[ie].mean(1)                      # (n_boot, n_models)
    gap = pd.DataFrame({'model': models, 'label': [pretty(m) for m in models],
                        'internal_dice': wi.mean().to_numpy(), 'external_dice': we.mean().to_numpy(),
                        'drop': obs.to_numpy(), 'drop_ci95_low': np.percentile(boot, 2.5, axis=0),
                        'drop_ci95_high': np.percentile(boot, 97.5, axis=0),
                        'relative_drop_pct': 100 * obs.to_numpy() / wi.mean().to_numpy(),
                        'n_internal': len(wi), 'n_external': len(we)})
    cmp = []
    if reference in models:
        r = models.index(reference)
        for k, m in enumerate(models):
            if m == reference: continue
            d = boot[:, r] - boot[:, k]
            cmp.append({'reference': pretty(reference), 'comparator': pretty(m),
                        'drop_difference': float(obs[reference] - obs[m]),
                        'ci95_low': np.percentile(d, 2.5), 'ci95_high': np.percentile(d, 97.5),
                        'bootstrap_p': min(1.0, 2 * min(np.mean(d <= 0), np.mean(d >= 0)))})
    cmp = pd.DataFrame(cmp)
    if len(cmp): cmp['bootstrap_p_holm'] = holm(cmp.bootstrap_p)
    return gap, cmp

def site_table(ext_df):
    base = ext_df[ext_df.variant == 'ensemble'] if (ext_df.variant == 'ensemble').any() else ext_df[ext_df.variant == 'single']
    base = base.assign(model=base.family)
    per = base.groupby(['site', 'model', 'patient_id'])[['WT_dice', 'TC_dice', 'ET_dice', 'mean_dice', 'mean_hd95']].mean()
    out = per.groupby(['site', 'model']).agg(['mean']).droplevel(1, axis=1).reset_index()
    out['n_patients'] = per.groupby(['site', 'model']).size().to_numpy()
    out['label'] = out.model.map(pretty)
    return out

def uncertainty_methods(df):
    m = []
    def add(name, frame, col=None, fn=None):
        frame = frame.dropna(subset=['mean_dice'])
        if not len(frame): return
        sc = fn(frame) if fn else frame[col].to_numpy(float)
        if np.all(np.isnan(sc)): return
        m.append((name, frame, sc))
    ph = df[df.family == REF]
    add('PH-CVT deterministic: entropy', ph[(ph.variant == 'single') & (ph.member == '42')], 'unc_entropy')
    add('PH-CVT MC dropout: MI', ph[ph.variant == 'mc_dropout'], 'unc_mi')
    add('PH-CVT seed ensemble: MI', ph[ph.variant == 'ensemble'], 'unc_mi')
    add('PH-CVT seed ensemble: entropy', ph[ph.variant == 'ensemble'], 'unc_entropy')
    add('Reference: small predicted tumour', ph[ph.variant == 'ensemble'], fn=lambda fr: -np.log1p(fr.WT_pred_ml.to_numpy(float)))
    for fam in ordered(df.family.unique()):
        if fam != REF: add(f'{pretty(fam)} seed ensemble: MI', df[(df.family == fam) & (df.variant == 'ensemble')], 'unc_mi')
    return m

def analyse_external(EXT, INT=None):
    dice_cols = [f'{r}_dice' for r in REGION_NAMES] + ['mean_dice']; hd_cols = [f'{r}_hd95' for r in REGION_NAMES] + ['mean_hd95']
    res = {}; es = _single(EXT)
    res['E1'], res['E1_seed'] = seed_table(es, dice_cols + hd_cols)
    ens = EXT[EXT.variant == 'ensemble'].assign(model=lambda d: d.family, seed=0); rows = []
    for fam in ordered(ens.model.unique()):
        part = ens[ens.model == fam]; row = {'model': fam, 'label': pretty(fam), 'n_members': int(part.n_members.iloc[0]), 'n_patients': len(part)}
        for c in dice_cols + hd_cols:
            row[c] = part[c].mean(); row[f'{c}_ci_low'], row[f'{c}_ci_high'] = bootstrap_ci(part[c])
        rows.append(row)
    res['E2'] = pd.DataFrame(rows)
    p1 = paired_vs_reference(es, protocol='external single models')
    p2 = paired_vs_reference(es, value='mean_hd95', protocol='external single models HD95', margin=1.0)
    p3 = paired_vs_reference(ens, protocol='external seed ensembles')
    if len(p3): p3['matched_seeds'] = 'all completed seeds (ensemble)'
    res['E3'] = pd.concat([p1, p2, p3], ignore_index=True)
    res['E4'] = site_table(EXT)
    methods = uncertainty_methods(EXT)
    res['E6'] = pd.DataFrame([{'method': n, **uncertainty_metrics(fr.mean_dice.to_numpy(float), sc, FAILURE_DICE)} for n, fr, sc in methods])
    res['methods'] = methods
    if INT is not None:
        res['E5'], res['E5_compare'] = generalisation_gap(_single(INT), es)
        shift = []; int_methods = {n: (fr, sc) for n, fr, sc in uncertainty_methods(INT)}
        for n, fr, sc in methods:
            if n in int_methods and not n.startswith('Reference'):
                a = int_methods[n][1]; b = sc
                shift.append({'method': n, 'internal_median': np.nanmedian(a), 'external_median': np.nanmedian(b),
                              'ratio_external_to_internal': np.nanmedian(b) / max(np.nanmedian(a), 1e-12),
                              'mann_whitney_p': sps.mannwhitneyu(a[~np.isnan(a)], b[~np.isnan(b)]).pvalue})
        res['E6_shift'] = pd.DataFrame(shift)
        if len(res['E6_shift']): res['E6_shift']['mann_whitney_p_holm'] = holm(res['E6_shift'].mann_whitney_p)
    bad = res['E1'][res['E1'].TC_dice_mean > res['E1'].WT_dice_mean]
    if len(bad):
        print('NOTE: mean TC Dice exceeds WT Dice for', list(bad.label), '- check label mapping before reporting.')
    return res

def plot_external(res, EXT, INT=None):
    es = _single(EXT); models = ordered(es.model.unique()); labels = [pretty(m) for m in models]
    pat = es.groupby(['model', 'patient_id'])[[f'{r}_dice' for r in REGION_NAMES] + ['mean_dice']].mean()
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, c, t in zip(axes, [f'{r}_dice' for r in REGION_NAMES] + ['mean_dice'], REGION_NAMES + ['Mean']):
        box(ax, [pat.loc[m][c].to_numpy() for m in models], labels, f'External — {t} 3D Dice', 'Patient Dice (seed-averaged)')
    fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig1_dice_boxplots.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    if 'E5' in res and len(res['E5']):
        g = res['E5']; fig, ax = plt.subplots(figsize=(8, 0.6 * len(g) + 1.8)); y = np.arange(len(g))
        ax.hlines(y, g.external_dice, g.internal_dice, color='grey', lw=2)
        ax.scatter(g.internal_dice, y, color='#1b9e77', s=50, label=f'Internal test (BraTS 2021, n={int(g.n_internal.iloc[0])})', zorder=3)
        ax.scatter(g.external_dice, y, color='#d95f02', s=50, label=f'External (ds007045, n={int(g.n_external.iloc[0])})', zorder=3)
        for yi, r in zip(y, g.itertuples()):
            ax.text(max(r.internal_dice, r.external_dice) + 0.004, yi, f'drop {r.drop:.3f} [{r.drop_ci95_low:.3f}, {r.drop_ci95_high:.3f}]', va='center', fontsize=8)
        lo_x, hi_x = float(min(g.external_dice.min(), g.internal_dice.min())), float(max(g.external_dice.max(), g.internal_dice.max()))
        ax.set_xlim(lo_x - 0.01, hi_x + 0.45 * (hi_x - lo_x + 0.02))
        ax.set_yticks(y, g.label); ax.invert_yaxis(); ax.set_xlabel('Mean patient Dice (full volume)')
        ax.set_title('Generalisation gap: internal → external (drop, 95% CI)'); ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2); ax.grid(axis='x', alpha=0.25)
        fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig2_generalisation_gap.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    st = res['E4']
    if len(st):
        mat = st.pivot(index='label', columns='site', values='mean_dice').reindex([pretty(m) for m in ordered(st.model.unique())])
        n = st.drop_duplicates('site').set_index('site').n_patients
        fig, ax = plt.subplots(figsize=(1.3 * mat.shape[1] + 3, 0.6 * mat.shape[0] + 2))
        im = ax.imshow(mat.to_numpy(), cmap='viridis', vmin=np.nanmin(mat.to_numpy()) - 0.02, vmax=np.nanmax(mat.to_numpy()))
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, f'{mat.iloc[i, j]:.3f}', ha='center', va='center', fontsize=8, color='white')
        ax.set_xticks(range(mat.shape[1]), [f'{s}\n(n={int(n.get(s, 0))})' for s in mat.columns], fontsize=8)
        ax.set_yticks(range(mat.shape[0]), mat.index, fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.03, label='Mean patient Dice (seed ensemble)')
        ax.set_title('External performance per hospital'); fig.tight_layout()
        fig.savefig(EXT_FIG / 'ext_fig3_per_site.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    U = res['E6']
    if len(U):
        cov = np.linspace(1.0, 0.5, 11); fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
        for n, fr, sc in res['methods']:
            axes[0].plot(cov * 100, risk_coverage(fr.mean_dice.to_numpy(float), sc, cov), marker='o', ms=3, label=n)
        axes[0].invert_xaxis(); axes[0].set_xlabel('Coverage: % of patients retained'); axes[0].set_ylabel('Mean Dice of retained patients')
        axes[0].set_title('External: risk–coverage'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.25)
        yy = np.arange(len(U))
        axes[1].errorbar(U.auroc, yy, xerr=[U.auroc - U.auroc_ci_low, U.auroc_ci_high - U.auroc], fmt='o', capsize=3)
        axes[1].axvline(0.5, color='k', ls='--', lw=1); axes[1].set_yticks(yy, U.method, fontsize=7); axes[1].invert_yaxis()
        axes[1].set_xlabel(f'AUROC for failures (Dice < {FAILURE_DICE}), 95% CI'); axes[1].set_title('External: failure detection'); axes[1].grid(axis='x', alpha=0.25)
        fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig4_uncertainty.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    if INT is not None and 'E6_shift' in res and len(res['E6_shift']):
        im_int = {n: sc for n, fr, sc in uncertainty_methods(INT)}; sel = [(n, sc) for n, fr, sc in res['methods'] if n in im_int and not n.startswith('Reference')]
        fig, ax = plt.subplots(figsize=(1.6 * len(sel) + 3, 5)); data, ticks = [], []
        for k, (n, sc) in enumerate(sel):
            data += [im_int[n][~np.isnan(im_int[n])], sc[~np.isnan(sc)]]; ticks += [f'{n}\ninternal', 'external']
        bp = ax.boxplot(data, showfliers=False, patch_artist=True)
        for k, patch in enumerate(bp['boxes']): patch.set_facecolor('#1b9e77' if k % 2 == 0 else '#d95f02'); patch.set_alpha(0.55)
        ax.set_xticks(range(1, len(ticks) + 1), ticks, rotation=40, ha='right', fontsize=7); ax.set_yscale('log')
        ax.set_ylabel('Patient uncertainty score (log)'); ax.set_title('Does uncertainty rise under domain shift?'); ax.grid(axis='y', alpha=0.25)
        fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig5_uncertainty_shift.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

    cal_path = EXT_OUT / 'external_calibration_bins.csv'
    if cal_path.exists():
        CAL = pd.read_csv(cal_path)
        CAL.groupby(['method', 'region', 'mode']).ece.first().unstack(['mode', 'region']).round(4).to_csv(EXT_TAB / 'ext_table7_calibration_ece.csv')
        fig, axes = plt.subplots(1, 3, figsize=(17, 5))
        for ax, region in zip(axes, REGION_NAMES):
            for meth in CAL.method.unique():
                part = CAL[(CAL.method == meth) & (CAL.region == region) & (CAL['mode'] == 'class_balanced') & (CAL.weight > 0)]
                ax.plot(part.confidence, part.observed_frequency, 'o-', ms=3, label=f'{meth} (ECE {part.ece.iloc[0]:.3f})')
            ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set_title(f'External — {region} (class-balanced)')
            ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed frequency'); ax.legend(fontsize=6); ax.grid(alpha=0.25)
        fig.tight_layout(); fig.savefig(EXT_FIG / 'ext_fig6_reliability.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)

if RUN_EXTERNAL:
    EXT = pd.read_csv(EXT_OUT / 'external_patient_rows.csv', dtype={'member': str})
    int_path = STUDY_DIR / 'full_volume' / 'full_volume_patient_rows.csv'
    INT = pd.read_csv(int_path, dtype={'member': str}) if int_path.exists() else None
    if INT is None: print('Internal full-volume results not found (run FINAL first): generalisation-gap analyses skipped.')
    RES = analyse_external(EXT, INT)
    names = {'E1': 'ext_table1_single_models', 'E2': 'ext_table2_ensembles', 'E3': 'ext_table3_paired_vs_phcvt',
             'E4': 'ext_table4_per_site', 'E5': 'ext_table5_generalisation_gap', 'E5_compare': 'ext_table5b_gap_vs_phcvt',
             'E6': 'ext_table6_uncertainty', 'E6_shift': 'ext_table6b_uncertainty_shift'}
    for key, name in names.items():
        if key in RES and isinstance(RES[key], pd.DataFrame): RES[key].to_csv(EXT_TAB / f'{name}.csv', index=False)
    print('External single models (mean ± SD across seeds):')
    print(RES['E1'][['label', 'n_seeds', 'mean_dice_mean', 'mean_dice_sd', 'WT_dice_mean', 'TC_dice_mean', 'ET_dice_mean', 'mean_hd95_mean']].round(4).to_string(index=False))
    if 'E5' in RES:
        print('\nGeneralisation gap (internal − external mean Dice):')
        print(RES['E5'][['label', 'internal_dice', 'external_dice', 'drop', 'drop_ci95_low', 'drop_ci95_high', 'relative_drop_pct']].round(4).to_string(index=False))
        if len(RES['E5_compare']): print(RES['E5_compare'].round(4).to_string(index=False))
    print('\nPaired comparisons on external data (reference − comparator):')
    print(RES['E3'][['protocol', 'comparator', 'mean_difference', 'ci95_low', 'ci95_high', 'wilcoxon_p_holm', 'interpretation']].round(4).to_string(index=False))
    print('\nUncertainty on external data:')
    print(RES['E6'][['method', 'spearman_rho', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'n_failures', 'aurc']].round(3).to_string(index=False))
    plot_external(RES, EXT, INT)
    with open(PAPER_DIR / 'results_summary_external.md', 'w') as fh:
        fh.write(f'# External validation: {EXTERNAL_NAME} (frozen models)\n\n')
        for key, name in names.items():
            if key in RES and isinstance(RES[key], pd.DataFrame) and len(RES[key]):
                fh.write(f'## {name}\n```\n{RES[key].round(4).to_string(index=False)}\n```\n\n')
    print('\nExternal tables :', EXT_TAB); print('External figures:', EXT_FIG)
else:
    print('External analysis not requested in this session.')

### Outputs for the manuscript
`extended_studies/paper_outputs/` contains `tables/` (CSV), `figures/` (300-dpi PNG), `results_summary.md` and a zip of everything. Report latency only from `table_efficiency_same_gpu.csv`, and state the evaluation conventions printed in the full-volume cell (Dice/HD95 empty-mask rules, threshold sources, 160×160 inference with native-grid scoring).

In [ ]:
# =============================================================================
# NEW — Package outputs and release the GPU
# =============================================================================
if SESSION in ('FINAL', 'EXTERNAL') and PAPER_DIR.exists():
    archive = shutil.make_archive(str(STUDY_DIR / 'paper_outputs'), 'zip', root_dir=PAPER_DIR)
    print('Paper figures and tables:', PAPER_DIR)
    print('Zip archive             :', archive)
    for p in sorted(PAPER_DIR.rglob('*')):
        if p.is_file(): print('  -', p.relative_to(PAPER_DIR))
print(f'Session {SESSION} finished.')
if RELEASE_RUNTIME_WHEN_DONE:
    release_runtime(f'Session {SESSION}: all requested stages finished')

# Part M. BraTS-Africa with all five seeds (inference only, no training)
Reviewers will ask why the BraTS-Africa results (Table 3, Fig. 3) use only the seed-42 checkpoint while UCSF-PDGM uses all
five seeds. Part M applies the remaining seeds (123, 2026, 31415, 27182) of PH-CVT, U-Net, ResU-Net and Attention U-Net,
plus seed 42 again as a reproduction check, to the same 94 BraTS-Africa cases with the same preprocessing, frozen
validation thresholds and full-volume Dice. It then repeats the Table 3 paired statistics on five-seed means.

**How to run (one runtime, A100 or T4):** in Cell 0 set `SESSION = "AFRICA5"`, then **Runtime ▸ Run all**.
Training, FINAL, EXTERNAL and Parts K/L are skipped; Run all stops at the end of Part M on purpose.

**Why it is fast:** each case is loaded once and passed through all 20 checkpoints held on the GPU, only Dice is computed
(no HD95, no MC dropout), and progress is saved every 5 cases, so a disconnect resumes where it stopped.
Optional: `M_MC_ALL_SEEDS = True` also runs 12-pass MC dropout for the four extra PH-CVT seeds to check that the
external failure-detection AUROC (0.811) is not specific to seed 42. This roughly triples the run time.

Outputs go to `outputs/extended_studies/external_brats_africa_5seed/` on Drive. Send me `M2_*.csv`.

In [ ]:
# ---------- M1. BraTS-Africa, all five seeds, deterministic full-volume Dice (GPU, resumable) ----------
M_MC_ALL_SEEDS = False          # True: also MC dropout (K = 12) for PH-CVT seeds other than 42 (about 3x slower)
M_FAMILIES = ['PHCVT_proposed', 'UNet', 'ResUNet', 'AttentionUNet']
if SESSION == 'AFRICA5':
    import glob, os, subprocess, sys, time, nibabel as nib, torch, torch.nn.functional as F
    OUTM = STUDY_DIR / 'external_brats_africa_5seed'; OUTM.mkdir(parents=True, exist_ok=True)
    EXCLUDED = {'BraTS-SSA-00230-000'}

    def find_ssa_roots(base):
        counts = {}
        for p in glob.glob(f'{base}/**/BraTS-SSA-*-000/*seg.nii*', recursive=True):
            root = str(Path(p).parent.parent); counts[root] = counts.get(root, 0) + 1
        return counts
    roots = find_ssa_roots('/content/drive/MyDrive/AI_Brain_Tumor') or find_ssa_roots('/content/kaggle_cache')
    if not roots:
        print('Downloading the BraTS-Africa Kaggle mirror used in the paper (shahriyarhasib/brats-africa-dataset) ...')
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle_API')
        except Exception as e:
            print(f'Colab secret "kaggle_API" not available ({type(e).__name__}); trying ~/.kaggle/kaggle.json')
        os.environ.setdefault('KAGGLEHUB_CACHE', '/content/kaggle_cache')
        try:
            import kagglehub
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True); import kagglehub
        roots = find_ssa_roots(kagglehub.dataset_download('shahriyarhasib/brats-africa-dataset'))
    assert roots, 'BraTS-Africa not found on Drive or Kaggle.'
    SSA_ROOT = Path(max(roots, key=roots.get))
    CASES = sorted(p.name for p in SSA_ROOT.glob('BraTS-SSA-*-000') if p.name not in EXCLUDED)
    assert len(CASES) == 94, f'expected 94 BraTS-Africa cases, found {len(CASES)}'

    def load_ssa(pid):
        """Identical to K2: per-volume robust z-score, bilinear 160x160, float16 rounding, native 3D labels."""
        d = SSA_ROOT / pid
        get = lambda m: nib.load(str(next(d.glob(f'{pid}-{m}.nii*'))))
        raw_flair = get('t2f').get_fdata()
        vols = [robust_normalize(get(m).get_fdata()) for m in ('t1n', 't1c', 't2w')] + [robust_normalize(raw_flair)]
        seg_img = get('seg'); seg = np.asarray(seg_img.get_fdata()).astype(np.uint8); seg[seg == 4] = 3
        H, W, D = seg.shape
        x = torch.from_numpy(np.ascontiguousarray(np.stack(vols, 0).transpose(3, 0, 1, 2)))
        x = F.interpolate(x, size=(CONFIG['image_size'],) * 2, mode='bilinear', align_corners=False).half().float()
        return {'pid': pid, 'x': x.to(DEVICE), 'gt': torch.from_numpy(label_to_regions(seg).astype(bool)).to(DEVICE),
                'shape': (H, W, D)}

    @torch.no_grad()
    def dice_only(p160, thresholds, case):
        """Dice part of score_prediction (native grid, Dice = 1 when both empty); HD95 skipped for speed."""
        H, W, D = case['shape']
        p = F.interpolate(p160, size=(H, W), mode='bilinear', align_corners=False).permute(1, 2, 3, 0)
        pred = p >= torch.tensor(thresholds, device=p.device, dtype=p.dtype).view(3, 1, 1, 1)
        gt = case['gt']
        inter = (pred & gt).sum((1, 2, 3)).double(); ps = pred.sum((1, 2, 3)).double(); gs = gt.sum((1, 2, 3)).double()
        dice = torch.where(ps + gs > 0, 2 * inter / (ps + gs).clamp_min(1), torch.ones_like(inter)).cpu().numpy()
        return {f'{r}_dice': float(dice[i]) for i, r in enumerate(REGION_NAMES)} | {'mean_dice': float(dice.mean())}

    MEMBERS = {f: member_list(f) for f in M_FAMILIES}
    for f, m in MEMBERS.items():
        print(f'{f}: seeds {[s for s, _, _ in m]}')
        assert len(m) == 5, f'{f}: expected 5 completed seeds (is ANALYSIS_SEEDS = None in Cell 0?)'
    M_NETS = {(f, s): (load_member(f, s, c), th) for f, m in MEMBERS.items() for s, c, th in m}

    OUT_ROWS = OUTM / 'M1_patient_seed_dice.csv'
    rows = pd.read_csv(OUT_ROWS).to_dict('records') if OUT_ROWS.exists() else []
    n_expected = 5 * len(M_FAMILIES) + len(M_FAMILIES) + (4 if M_MC_ALL_SEEDS else 0)
    per_pid = pd.Series([r['patient_id'] for r in rows]).value_counts() if rows else pd.Series(dtype=int)
    done = set(per_pid[per_pid >= n_expected].index)
    rows = [r for r in rows if r['patient_id'] in done]          # drop partial cases; they are redone
    todo = [c for c in CASES if c not in done]
    print(f'{len(done)}/94 cases done, {len(todo)} to go'); t0 = time.time()
    for i, pid in enumerate(todo):
        case = load_ssa(pid)
        for fam, members in MEMBERS.items():
            s1 = 0; ths = []
            for seed, _, th in members:
                net = M_NETS[(fam, seed)][0]
                p = predict_probs(net, case['x']); s1 = s1 + p; ths.append(th)
                rows.append({'patient_id': pid, 'family': fam, 'seed': seed, 'variant': 'single', **dice_only(p, th, case)})
                if M_MC_ALL_SEEDS and fam == 'PHCVT_proposed' and seed != 42:
                    mean, var, exp_ent, _ = predict_mc(net, case['x'], MC_PASSES_FULLVOLUME)
                    total = binary_entropy(mean); mi = (total - exp_ent).clamp_min(0)
                    u = uncertainty_scores(mean, total, mi, var)
                    rows.append({'patient_id': pid, 'family': fam, 'seed': seed, 'variant': 'mc_dropout',
                                 **dice_only(mean, th, case), 'unc_mi': u['unc_mi'], 'unc_entropy': u['unc_entropy']})
                    del mean, var, exp_ent, total, mi
                del p
            ens_th = list(np.mean(ths, axis=0))                      # internal convention: mean of member thresholds
            rows.append({'patient_id': pid, 'family': fam, 'seed': 0, 'variant': 'ensemble',
                         **dice_only(s1 / len(members), ens_th, case)})
        del case, s1; torch.cuda.empty_cache()
        if i % 5 == 4 or i == len(todo) - 1:
            pd.DataFrame(rows).to_csv(OUT_ROWS, index=False)
            el = time.time() - t0; print(f'{len(done) + i + 1}/94 cases  {el / 60:.1f} min  (~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left)')
    M1 = pd.DataFrame(rows); M1.to_csv(OUT_ROWS, index=False)
    chk = M1[(M1.variant == 'single') & (M1.seed == 42)].groupby('family')[[f'{r}_dice' for r in REGION_NAMES]].mean().round(3)
    print('\nReproduction check, seed 42 (should match Table 3: PH-CVT WT 0.890, U-Net WT 0.891, ResU-Net 0.890, Attention U-Net 0.893):')
    print(chk.to_string())

In [ ]:
# ---------- M2. Table 3 statistics on five-seed means (CPU) ----------
if SESSION == 'AFRICA5':
    from sklearn.metrics import roc_auc_score
    M1 = pd.read_csv(OUTM / 'M1_patient_seed_dice.csv')
    assert M1.patient_id.nunique() == 94, 'M1 has not finished all 94 cases'
    REG = [f'{r}_dice' for r in REGION_NAMES]
    single = M1[M1.variant == 'single']
    # (a) per-seed cohort means -> mean ± SD across the five seeds (as internal Table 1)
    per_seed = single.groupby(['family', 'seed'])[REG + ['mean_dice']].mean()
    a = per_seed.groupby('family').agg(['mean', 'std']).round(4); a.to_csv(OUTM / 'M2_seed_mean_sd.csv'); print(a.to_string())
    # (b) paired bootstrap on per-patient five-seed means (10,000 resamples, Holm across regions per comparator)
    def holm(p):
        p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0
        for k, i in enumerate(o): run = max(run, (len(p) - k) * p[i]); adj[i] = min(1, run)
        return adj
    def paired(df, ref='PHCVT_proposed', n=10_000, seed=0):
        out = []; rng = np.random.default_rng(seed)
        wide = {r: df.pivot(index='patient_id', columns='family', values=r) for r in REG}
        idx = rng.integers(0, 94, (n, 94))
        for comp in ['UNet', 'ResUNet', 'AttentionUNet']:
            res = []
            for r in REG:
                d = (wide[r][ref] - wide[r][comp]).values; b = d[idx].mean(1)
                res.append(dict(comparator=comp, region=r.split('_')[0], diff=round(d.mean(), 4),
                                ci95_low=round(np.percentile(b, 2.5), 4), ci95_high=round(np.percentile(b, 97.5), 4),
                                ci90_low=round(np.percentile(b, 5), 4), ci90_high=round(np.percentile(b, 95), 4),
                                p_boot=max(2 * min((b <= 0).mean(), (b >= 0).mean()), 1 / n)))
            for row, ph in zip(res, holm([x['p_boot'] for x in res])):
                row['p_holm'] = round(ph, 4); row['equivalent_pm0.01'] = bool(row['ci90_low'] > -0.01 and row['ci90_high'] < 0.01)
            out += res
        return pd.DataFrame(out)
    seed_mean = single.groupby(['patient_id', 'family'])[REG].mean().reset_index()
    b5 = paired(seed_mean); b5.to_csv(OUTM / 'M2_paired_5seed_mean.csv', index=False)
    print('\nPaired PH-CVT minus comparator, per-patient five-seed means:'); print(b5.to_string(index=False))
    be = paired(M1[M1.variant == 'ensemble']); be.to_csv(OUTM / 'M2_paired_5seed_ensemble.csv', index=False)
    print('\nPaired PH-CVT minus comparator, five-seed ensembles:'); print(be.to_string(index=False))
    # (c) per-seed paired differences: is the seed-42 TC/ET deficit typical?
    ps = []
    for s in sorted(single.seed.unique()):
        w = single[single.seed == s].pivot(index='patient_id', columns='family', values=REG)
        for comp in ['UNet', 'ResUNet', 'AttentionUNet']:
            ps.append({'seed': s, 'comparator': comp, **{r.split('_')[0]: round((w[r]['PHCVT_proposed'] - w[r][comp]).mean(), 4) for r in REG}})
    ps = pd.DataFrame(ps); ps.to_csv(OUTM / 'M2_per_seed_diff.csv', index=False); print('\nPer-seed mean differences:'); print(ps.to_string(index=False))
    # (d) internal (five-seed full-volume) vs external (five-seed) relative decline
    fv = pd.read_csv(STUDY_DIR / 'full_volume' / 'full_volume_patient_rows.csv')
    fvs = fv[fv.variant == 'single'].groupby(['family', 'member'])[REG].mean().groupby('family').mean()
    ext = per_seed.groupby('family')[REG].mean()
    dec = ((fvs.loc[M_FAMILIES] - ext.loc[M_FAMILIES]) / fvs.loc[M_FAMILIES] * 100).round(1)
    dec.to_csv(OUTM / 'M2_internal_external_decline_pct.csv'); print('\nRelative decline internal -> external (%), both five-seed means:'); print(dec.to_string())
    # (e) optional: MC-dropout failure detection per PH-CVT seed
    mc = M1[M1.variant == 'mc_dropout']
    if len(mc):
        au = [{'seed': s, 'failures': int((g.mean_dice < 0.75).sum()),
               'AUROC_MI': round(roc_auc_score((g.mean_dice < 0.75).astype(int), g.unc_mi), 3)} for s, g in mc.groupby('seed')]
        au = pd.DataFrame(au); au.to_csv(OUTM / 'M2_mc_auroc_per_seed.csv', index=False)
        print('\nMC-dropout MI AUROC per extra seed (seed 42 in the paper: 0.811):'); print(au.to_string(index=False))
    print(f'\nSaved to {OUTM}')

In [ ]:
# ---------- M4. Does the transformer-and-fusion block cause the external TC/ET deficit? (GPU, ~2-4 min on a T4) ----------
# Applies the ablation without transformer and gated fusion (PHCVT_no_transformer, 3 seeds: 42, 123, 2026) to the same
# 94 cases and compares it with PH-CVT and U-Net on the same three seeds (rows from M1). Inference only.
if SESSION == 'AFRICA5':
    ABL = 'PHCVT_no_transformer'
    OUT_ABL = OUTM / 'M4_ablation_patient_dice.csv'
    abl_members = member_list(ABL)
    print(f'{ABL}: seeds {[s for s, _, _ in abl_members]}'); assert len(abl_members) >= 3, 'ablation checkpoints not found'
    if OUT_ABL.exists():
        M4 = pd.read_csv(OUT_ABL)
    else:
        abl_nets = [(s, load_member(ABL, s, c), th) for s, c, th in abl_members]; rows4 = []; t0 = time.time()
        for i, pid in enumerate(CASES):
            case = load_ssa(pid)
            for s, net, th in abl_nets:
                rows4.append({'patient_id': pid, 'family': ABL, 'seed': s, 'variant': 'single',
                              **dice_only(predict_probs(net, case['x']), th, case)})
            del case; torch.cuda.empty_cache()
            if i % 20 == 19: print(f'{i + 1}/94  {(time.time() - t0) / 60:.1f} min')
        M4 = pd.DataFrame(rows4); M4.to_csv(OUT_ABL, index=False); del abl_nets
    seeds = sorted(M4.seed.unique())
    M1 = pd.read_csv(OUTM / 'M1_patient_seed_dice.csv')
    both = pd.concat([M1[(M1.variant == 'single') & M1.seed.isin(seeds)], M4])
    sm = both.groupby(['patient_id', 'family'])[REG].mean().reset_index()
    print('\nSeed means (same seeds):'); print(sm.groupby('family')[REG].mean().round(4).to_string())
    rng = np.random.default_rng(0); idx = rng.integers(0, 94, (10_000, 94)); out = []
    for a, b in [(ABL, 'PHCVT_proposed'), (ABL, 'UNet'), ('PHCVT_proposed', 'UNet')]:
        res = []
        for r in REG:
            w = sm.pivot(index='patient_id', columns='family', values=r); dd = (w[a] - w[b]).values; bb = dd[idx].mean(1)
            res.append(dict(model=a, comparator=b, region=r.split('_')[0], diff=round(dd.mean(), 4),
                            ci95_low=round(np.percentile(bb, 2.5), 4), ci95_high=round(np.percentile(bb, 97.5), 4),
                            ci90_low=round(np.percentile(bb, 5), 4), ci90_high=round(np.percentile(bb, 95), 4),
                            p_boot=max(2 * min((bb <= 0).mean(), (bb >= 0).mean()), 1e-4)))
        p = np.array([x['p_boot'] for x in res]); o = np.argsort(p); run = 0
        for k, i in enumerate(o): run = max(run, (3 - k) * p[i]); res[i]['p_holm'] = round(min(1, run), 4)
        out += res
    M4s = pd.DataFrame(out); M4s.to_csv(OUTM / 'M4_ablation_paired.csv', index=False)
    print(f'\nPaired differences on seeds {seeds} (10,000 resamples, Holm across regions):'); print(M4s.to_string(index=False))

In [ ]:
# ---------- M3. Stop Run all here in the AFRICA5 session (Parts K and L are not needed) ----------
if SESSION == 'AFRICA5':
    raise SystemExit('Part M finished. Run all stops here on purpose; send the M2_*.csv files.')

# Part K. Same uncertainty rule on both cohorts (review items M4 and M5)
The paper's internal MC-dropout AUROC (0.903) uses the tumour-neighbourhood rule in `uncertainty_scores()`
(mean voxel MI inside WT probability > 0.05, full volume). The external value (0.578) came from
`external_mc_dropout_by_patient.csv`, which averages slice-level MI (`n_slices`, `patient_mi_mean`), the rule that gives
0.645 internally. K1 checks this without a GPU. K2 recomputes BraTS-Africa with the paper's rule using the frozen
PH-CVT seed-42 model, and K3 compares internal and external with one rule.

Run after the FINAL session cells (they define NETS, predict_mc, score_prediction, uncertainty_scores).
K1 also runs on its own after mounting Drive.

**Note:** K1 was already answered by the revision-analyses run (internal 0.645 vs external 0.578 with the slice-average rule). You can skip K1 and run K2 and K3 directly.


In [ ]:
# ---------- K1. Which rule produced which number (no GPU) ----------
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
BASE = Path('/content/drive/MyDrive/AI_Brain_Tumor/PH_CVT_BraTS1251/outputs/extended_studies')
OUTK = BASE / 'revision_partK'; OUTK.mkdir(exist_ok=True)
FV_ROWS = BASE / 'full_volume' / 'full_volume_patient_rows.csv'
EXT_SLICE = BASE / 'PHCVT_proposed_seed42' / 'external' / 'external_mc_dropout_by_patient.csv'
INT_SLICE = BASE / 'publication_reliability' / 'patient_uncertainty_failure_data.csv'

fv = pd.read_csv(FV_ROWS)
INT_ROI = fv[(fv.family == 'PHCVT_proposed') & (fv.variant == 'mc_dropout')][['patient_id', 'mean_dice', 'unc_mi', 'unc_entropy']]
INT_DET = fv[(fv.family == 'PHCVT_proposed') & (fv.variant == 'single') & (fv.member.astype(str) == '42')][['patient_id', 'unc_entropy']]
ext_s = pd.read_csv(EXT_SLICE); int_s = pd.read_csv(INT_SLICE)
def auc(d, s, t=0.75):
    y = (np.asarray(d) < t).astype(int); return roc_auc_score(y, s), int(y.sum()), len(y)
for name, d, s in [('internal, tumour-neighbourhood rule (paper Table 4)', INT_ROI.mean_dice, INT_ROI.unc_mi),
                   ('internal, slice-average rule', int_s.patient_mean_dice, int_s.mean_mutual_information),
                   ('external, slice-average rule (paper 0.578)', ext_s.dice_mean, ext_s.patient_mi_mean)]:
    a, f, n = auc(d, s); print(f'{name}: MI AUROC {a:.3f}, failures {f}/{n}, Spearman {spearmanr(s, 1 - np.asarray(d))[0]:.3f}')
# Expected: 0.903, 0.645, 0.578. If so, the paper compares 0.903 and 0.578 across two different rules.

In [ ]:
# ---------- K0. Reload the models Parts K and L need (the full-volume cell frees them at its end) ----------
# PH-CVT seed 42 for K2 and L1, and every completed U-Net seed for the L2 ensemble timing.
NETS = {}
for fam in ('PHCVT_proposed', 'UNet'):
    for s, c, th in member_list(fam):
        if fam == 'PHCVT_proposed' and s != 42: continue
        NETS[(fam, s)] = (load_member(fam, s, c), th)
print('Loaded for Parts K and L:', sorted(NETS))
assert ('PHCVT_proposed', 42) in NETS, 'PH-CVT seed 42 checkpoint not found'
assert sum(1 for k in NETS if k[0] == 'UNet') == 5, 'expected 5 U-Net seeds for the L2 ensemble timing'


In [ ]:
# ---------- K2. BraTS-Africa with the paper's rule (GPU, resumable; ~1 min per patient for T = 12) ----------
import glob, os, subprocess, sys, time, nibabel as nib, torch, torch.nn.functional as F
EXCLUDED = {'BraTS-SSA-00230-000'}

def find_ssa_roots(base):
    """Folders that hold BraTS-SSA-*-000 case folders, with the number of cases in each."""
    counts = {}
    for p in glob.glob(f'{base}/**/BraTS-SSA-*-000/*seg.nii*', recursive=True):
        root = str(Path(p).parent.parent); counts[root] = counts.get(root, 0) + 1
    return counts

# 1) Drive (AI_Brain_Tumor), 2) an earlier Kaggle download in this runtime, 3) download from Kaggle now
roots = find_ssa_roots('/content/drive/MyDrive/AI_Brain_Tumor') or find_ssa_roots('/content/kaggle_cache')
if not roots:
    print('BraTS-Africa is not on Drive: downloading the Kaggle mirror used in the paper (shahriyarhasib/brats-africa-dataset) ...')
    try:
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle_API')      # same Colab secret as the 3D notebook
    except Exception as e:
        print(f'Colab secret "kaggle_API" not available ({type(e).__name__}); trying ~/.kaggle/kaggle.json')
    os.environ.setdefault('KAGGLEHUB_CACHE', '/content/kaggle_cache')    # fast local disk, not Drive
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True); import kagglehub
    roots = find_ssa_roots(kagglehub.dataset_download('shahriyarhasib/brats-africa-dataset'))
print('BraTS-Africa folders found (cases):', roots)
assert roots, 'BraTS-Africa not found on Drive or Kaggle; send this output before running anything else.'
SSA_ROOT = Path(max(roots, key=roots.get))       # the folder with the most cases (the 95 glioma cases)
CASES = sorted(p.name for p in SSA_ROOT.glob('BraTS-SSA-*-000') if p.name not in EXCLUDED)
print(SSA_ROOT, '|', len(CASES), 'cases')        # expected 94
assert len(CASES) == 94, f'expected 94 BraTS-Africa cases, found {len(CASES)}'

def load_ssa(pid):
    d = SSA_ROOT / pid
    get = lambda m: nib.load(str(next(d.glob(f'{pid}-{m}.nii*'))))
    raw_flair = get('t2f').get_fdata()
    vols = [robust_normalize(get(m).get_fdata()) for m in ('t1n', 't1c', 't2w')] + [robust_normalize(raw_flair)]
    seg_img = get('seg'); seg = np.asarray(seg_img.get_fdata()).astype(np.uint8); seg[seg == 4] = 3
    H, W, D = seg.shape
    x = torch.from_numpy(np.ascontiguousarray(np.stack(vols, 0).transpose(3, 0, 1, 2)))
    x = F.interpolate(x, size=(CONFIG['image_size'],) * 2, mode='bilinear', align_corners=False).half().float()
    gt = label_to_regions(seg).astype(bool)
    return {'pid': pid, 'x': x.to(DEVICE), 'gt_np': gt, 'gt': torch.from_numpy(gt).to(DEVICE),
            'brain': torch.from_numpy(raw_flair != 0).to(DEVICE), 'shape': (H, W, D),
            'spacing': tuple(float(z) for z in seg_img.header.get_zooms()[:3])}

OUT_EXT = OUTK / 'external_roi_uncertainty.csv'
rows = pd.read_csv(OUT_EXT).to_dict('records') if OUT_EXT.exists() else []
done = {r['patient_id'] for r in rows}
net = NETS[('PHCVT_proposed', 42)][0]; t0 = time.time()
for i, pid in enumerate([c for c in CASES if c not in done]):
    case = load_ssa(pid)
    p = predict_probs(net, case['x']); det = uncertainty_scores(p, binary_entropy(p))
    mean, var, exp_ent, _ = predict_mc(net, case['x'], MC_PASSES_FULLVOLUME)
    total = binary_entropy(mean); mi = (total - exp_ent).clamp_min(0)
    row, _ = score_prediction(mean, list(seg_thresholds), case)
    row.update(uncertainty_scores(mean, total, mi, var)); row.update(patient_id=pid, det_entropy=det['unc_entropy'])
    rows.append(row); del case, p, mean, var, exp_ent, total, mi
    if i % 5 == 0:
        pd.DataFrame(rows).to_csv(OUT_EXT, index=False); print(f'{len(rows)}/{len(CASES)}  {time.time() - t0:.0f}s')
    torch.cuda.empty_cache()
EXT_ROI = pd.DataFrame(rows); EXT_ROI.to_csv(OUT_EXT, index=False)
print('external mean Dice', EXT_ROI.mean_dice.mean().round(4), '(MC mean prediction; the paper reports WT 0.890 / TC 0.755 / ET 0.716)')

In [ ]:
# ---------- K3. Internal vs external with ONE rule ----------
def boot(y, s, n=5000, seed=0):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i): out.append(roc_auc_score(y[i], s[i]))
    return np.array(out)
EXT_ROI = pd.read_csv(OUTK / 'external_roi_uncertainty.csv')
int_det = INT_ROI[['patient_id', 'mean_dice']].merge(INT_DET, on='patient_id')
rows = []
for t in (0.70, 0.75, 0.80):
    for coh, df, cols in (('BraTS 2021 test', INT_ROI, (('unc_mi', 'MC dropout MI'), ('unc_entropy', 'MC dropout entropy'))),
                          ('BraTS 2021 test', int_det, (('unc_entropy', 'deterministic entropy'),)),
                          ('BraTS-Africa', EXT_ROI, (('unc_mi', 'MC dropout MI'), ('unc_entropy', 'MC dropout entropy'), ('det_entropy', 'deterministic entropy')))):
        y = (df.mean_dice < t).astype(int).values
        for c, lab in cols:
            b = boot(y, df[c].values, 2000)
            rows.append(dict(threshold=t, cohort=coh, score=lab, failures=f'{y.sum()}/{len(y)}', AUROC=round(roc_auc_score(y, df[c]), 3),
                             CI=f'{np.percentile(b, 2.5):.2f}–{np.percentile(b, 97.5):.2f}',
                             spearman=round(spearmanr(df[c], 1 - df.mean_dice)[0], 3)))
K3 = pd.DataFrame(rows); K3.to_csv(OUTK / 'K3_threshold_sensitivity_same_rule.csv', index=False); print(K3.to_string(index=False))
yi = (INT_ROI.mean_dice < 0.75).astype(int).values; ye = (EXT_ROI.mean_dice < 0.75).astype(int).values
obs = roc_auc_score(yi, INT_ROI.unc_mi) - roc_auc_score(ye, EXT_ROI.unc_mi)
d = boot(yi, INT_ROI.unc_mi.values, 5000, 1) - boot(ye, EXT_ROI.unc_mi.values, 5000, 2)
p = max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / 5000)
print(f'\nMC dropout MI, tumour-neighbourhood rule: internal − external AUROC = {obs:+.3f} '
      f'(95% CI {np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f}), bootstrap P = {p:.4f}')

# Part L. Detection head on all slices, and measured MC-dropout latency
Replaces revision cells C and F. Run in the FINAL-study session, after the full-volume cell has
defined NETS, load_patient, test_ids, autocast_context, FV_BATCH and BEST_MODEL_PATH.

In [ ]:
# ---------- L1. Detection head on every axial slice of the 188 test volumes ----------
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
OUTL = STUDY_DIR / 'revision_partL'; OUTL.mkdir(exist_ok=True)
net = NETS[('PHCVT_proposed', 42)][0].eval()

# detection threshold calibrated on validation (falls back to 0.5 if no file is found)
DET_THR = 0.5
for f in [Path(BEST_MODEL_PATH).parent / 'calibrated_thresholds.json', STUDY_DIR / 'PHCVT_proposed_seed42' / 'calibrated_thresholds.json']:
    if f.exists():
        js = json.loads(f.read_text()); print(f, '->', {k: v for k, v in js.items() if k != 'segmentation'})
        for k in ('detection', 'det', 'detection_threshold'):
            if k in js: DET_THR = float(js[k] if not isinstance(js[k], dict) else list(js[k].values())[0])
        break
print('Detection threshold used:', DET_THR)

OUT_L1 = OUTL / 'detection_all_slices_rows.csv'
rows = pd.read_csv(OUT_L1).to_dict('records') if OUT_L1.exists() else []
done = {r['patient_id'] for r in rows}
for i, pid in enumerate([p for p in test_ids if p not in done]):
    case = load_patient(pid); x = case['x']; logits = []
    with torch.no_grad():
        for s in range(0, x.shape[0], FV_BATCH):
            with autocast_context():
                logits.append(net(x[s:s + FV_BATCH])['det_logit'].float().flatten().cpu())
    prob = torch.sigmoid(torch.cat(logits)).numpy()
    vox = case['seg'].reshape(-1, case['seg'].shape[2]).astype(bool).sum(0)      # tumour voxels per axial slice
    rows += [dict(patient_id=pid, slice=k, prob=float(prob[k]), tumour_voxels=int(vox[k])) for k in range(len(prob))]
    del case
    if i % 10 == 0: pd.DataFrame(rows).to_csv(OUT_L1, index=False); print(len({r['patient_id'] for r in rows}), '/', len(test_ids))
L1 = pd.DataFrame(rows); L1.to_csv(OUT_L1, index=False)

res = []
for rule, y in (('>=1 tumour voxel', L1.tumour_voxels.values >= 1), ('>=20 tumour voxels', L1.tumour_voxels.values >= 20)):
    for thr in sorted({DET_THR, 0.5}):
        pred = L1.prob.values >= thr
        tp, fp = (pred & y).sum(), (pred & ~y).sum(); fn, tn = (~pred & y).sum(), (~pred & ~y).sum()
        prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        res.append(dict(positive=rule, threshold=thr, slices=len(y), prevalence=round(y.mean(), 3),
                        accuracy=round((tp + tn) / len(y), 3), precision=round(prec, 3), recall=round(rec, 3),
                        specificity=round(tn / max(tn + fp, 1), 3), F1=round(2 * prec * rec / max(prec + rec, 1e-9), 3),
                        ROC_AUC=round(roc_auc_score(y, L1.prob), 3), PR_AUC=round(average_precision_score(y, L1.prob), 3)))
L1_SUM = pd.DataFrame(res); L1_SUM.to_csv(OUTL / 'detection_all_slices_summary.csv', index=False); print(L1_SUM.to_string(index=False))

In [ ]:
# ---------- L2. Measured latency: PH-CVT MC dropout (T = 12) vs 5-member U-Net ensemble ----------
import time
@torch.no_grad()
def time_fn(fn, n=50, warm=10):
    x = torch.randn(1, 4, 160, 160, device=DEVICE)
    for _ in range(warm): fn(x)
    torch.cuda.synchronize(); ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(x); torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))
drop = [m for m in net.modules() if isinstance(m, (torch.nn.Dropout, torch.nn.Dropout2d, torch.nn.Dropout3d)) and m.p > 0]
def single(x):
    net.eval()
    with autocast_context(): net(x)
def mc12_seq(x):
    net.eval(); [m.train() for m in drop]
    with autocast_context():
        for _ in range(12): net(x)
    net.eval()
def mc12_batched(x):
    net.eval(); [m.train() for m in drop]
    with autocast_context(): net(x.expand(12, -1, -1, -1))
    net.eval()
unets = [NETS[k][0] for k in sorted(NETS) if k[0] == 'UNet']
def ens5(x):
    with autocast_context():
        for u in unets: u(x)
lat = dict(gpu=torch.cuda.get_device_name(0), phcvt_single_ms=time_fn(single), phcvt_mc12_sequential_ms=time_fn(mc12_seq),
           phcvt_mc12_batched_ms=time_fn(mc12_batched), unet_members=len(unets), unet_ensemble_ms=time_fn(ens5) if unets else float('nan'))
pd.DataFrame([lat]).to_csv(OUTL / 'latency_mc_vs_ensemble.csv', index=False)
for k, v in lat.items(): print(f'{k}: {v:.2f}' if isinstance(v, float) else f'{k}: {v}')

In [ ]:
# ---------- Final: release the A100 once everything above has finished ----------
# Writes all pending Drive files first, then disconnects the GPU so it stops using compute units.
# The outputs stay visible in this browser tab; press Ctrl+S afterwards to keep them.
import time
from google.colab import drive, runtime
print('All cells finished. Saving pending Drive writes, then releasing the A100 ...', flush=True)
drive.flush_and_unmount()
time.sleep(5)
runtime.unassign()
